# AVCT Aurora -- LEAN pipeline (V26 primary, V28 negative controls, V29 ensemble, V30 hard-case specialization)

This is a dependency-pruned version of the full V2-V29 notebook, built by
static analysis rather than by hand: every retained cell was verified to
actually be read by a downstream cell that matters, and every dropped cell
was verified to have no surviving reader. Two categories were cut:

1. **Pure exploratory dead ends with no surviving readers**: V23 (session-anchored
   ladder), V24 (calibration-row leakage audit), V25 (publication figures).
   Their conclusions are summarized in the project notes; nothing downstream
   of them is still imported here. If you need the actual figures/tables
   again, pull them from the original notebook -- they were not migrated.
2. **Nothing else.** Everything else -- including V19-1, V19-2, V20, and V21,
   which look like superseded diagnostics -- is still here because it is a
   genuine runtime dependency, not a historical narrative choice.

## Why V19-1 -> V19-2 -> V20 -> V21 survived the cut

V26 and V28 both call `V21_CONTEXT_FEATURES` and `v21_fit_and_score`, which
are defined in the V21 cell. That cell, as written, reuses intermediate
variables produced by V20's cell, which reuses V19-2's (~17 minute)
multi-window extraction output, which reuses V19-1's. This is a real,
verified data dependency (confirmed by tracing every name V21 loads back to
where it's defined), not just cells that happened to run earlier. Cutting
V19-2 in particular would save real runtime, but would require rewriting V21
to stop depending on it -- I have not done that rewrite, since I can't verify
a rewrite against your actual Aurora data without execution access. If you
want that further trim, it's a well-scoped follow-up, not a blocker.

## What's actually here

| Section | What it produces | Status |
|---|---|---|
| Setup / feature functions / delta tables / Aurora load / cross-dataset ranking | Shared infrastructure | Required by everything below |
| V19 prerequisites, V19-1, V19-2, V20, V21 | `abl_weights`/`abl_fit_gate`/`abl_apply_gate`, `V21_CONTEXT_FEATURES`, `v21_fit_and_score` | Kept only as dependencies (see above) |
| V22 | `v22_add_skin_features` | Kept as a dependency of the V28 skin-correction arm |
| **V26** | Primary session-anchored model + context ablation + calibration-dose comparison + CI summary + publication decision table | **Current deployable baseline** |
| **V28** | Calibration-dispersion / cross-target arms (both unsupported) and PC vs raw-AVCT skin-correction arms (no clear difference, worse in the V-VI subgroup) | **Confirmed negative controls** |
| **V29** | MAP/PP targets, Extra Trees, Ridge, nested ensemble (CI-supported +0.13 SBP / +0.09 DBP), temporal smoother, uncertainty-based coverage table | **Current best 100%-coverage model + selective-prediction analysis** |
| **V30** | Hard-case specialization (uncertainty-aware features, lability-weighted training), scored at 100% coverage against V29's nested ensemble | **Untested new hypothesis -- run this next** |

Run top to bottom. Runtime is dominated by V21's cached multi-window
extraction (first run only), V26/V28/V29's repeated participant CV, and V30's
extra CV layer (~1.3-1.5x V29's runtime, see V30's own markdown cell).


## Setup, paths, and optical model

In [ ]:
!pip -q install vitaldb numpy pandas==2.2.2 scipy scikit-learn matplotlib seaborn tqdm h5py joblib

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc, math, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import vitaldb
import h5py, joblib
from scipy.signal import butter, sosfiltfilt, find_peaks, welch, wiener
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy.stats import skew, kurtosis
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

BIDMC_DIR = Path('/content/drive/MyDrive/bidmc-ppg-and-respiration-dataset-1.0.0/bidmc-ppg-and-respiration-dataset-1.0.0/bidmc_csv')
OUT_DIR = Path('/content/drive/MyDrive/AVCT_BIDMC_VitalDB_Pigmentation')
VITAL_DIR = OUT_DIR / 'vitaldb_windows'
OUT_DIR.mkdir(parents=True, exist_ok=True)
VITAL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
WINDOW_SECONDS = 10
VITAL_FS = 100
N_VITAL_CASES = 20          # Smoke test. Use 50-100+ for the main experiment.
MAX_VITAL_WINDOWS = 150     # Per subject.
MAX_BIDMC_WINDOWS = 24      # Per subject.

print('BIDMC exists:', BIDMC_DIR.exists())
print('Outputs:', OUT_DIR)


CUFFLESS_DIR=Path('/content/drive/MyDrive/cuff+less+blood+pressure+estimation')
WELLTORY_DIR=Path('/content/drive/MyDrive/welltory-ppg-dataset-master')
MIMIC_DIR=Path('/content/drive/MyDrive/CVD/mimic_ecg_ppg_resp_final (Unzipped Files)')
MAX_MIMIC_SEGMENTS=1500
MAX_WELLTORY_WINDOWS=8
MAX_CUFF_RECORDS_PER_PART=120
MAX_CUFF_WINDOWS_PER_RECORD=12
for name,path in {'BIDMC':BIDMC_DIR,'MIMIC':MIMIC_DIR,'Welltory':WELLTORY_DIR,'UCI cuff-less':CUFFLESS_DIR}.items():
    print(f'{name:14s}', 'FOUND' if path.exists() else 'MISSING', path)

NESTED_PARAMETER_GRID=[
    {'learning_rate':0.03,'max_leaf_nodes':15,'l2_regularization':1.0},
    {'learning_rate':0.05,'max_leaf_nodes':15,'l2_regularization':2.0},
    {'learning_rate':0.03,'max_leaf_nodes':31,'l2_regularization':2.0},
    {'learning_rate':0.05,'max_leaf_nodes':31,'l2_regularization':5.0},
]
BOOTSTRAP_REPEATS=2000


In [ ]:
FITZ_ROMAN = {1:'I', 2:'II', 3:'III', 4:'IV', 5:'V', 6:'VI'}

def synthetic_skin_labels(subject_ids, seed, source):
    ids = np.asarray(sorted(set(map(str, subject_ids))))
    rng = np.random.default_rng(seed)
    ids = rng.permutation(ids)
    z = (np.arange(len(ids)) + 0.5) / max(len(ids), 1)
    d = pd.DataFrame({'subject_id':ids, 'source':source, 'pigmentation_index':z})
    d['monk_synthetic'] = np.clip(np.floor(z*10).astype(int)+1, 1, 10)
    d['fitzpatrick_numeric'] = np.clip(np.floor(z*6).astype(int)+1, 1, 6)
    d['fitzpatrick_synthetic'] = d.fitzpatrick_numeric.map(FITZ_ROMAN)
    d['monk_group'] = pd.cut(d.monk_synthetic, [0,3,6,10], labels=['Monk 1-3','Monk 4-6','Monk 7-10']).astype(str)
    d['fitzpatrick_group'] = pd.cut(d.fitzpatrick_numeric, [0,2,4,6], labels=['Fitzpatrick I-II','Fitzpatrick III-IV','Fitzpatrick V-VI']).astype(str)
    d['darker_synthetic'] = d.monk_synthetic >= 7
    d['labels_are_synthetic'] = True
    return d.sort_values('subject_id').reset_index(drop=True)

def optical_stress(ppg, z, seed, fs, max_attenuation=0.55, max_noise=0.12):
    x=np.asarray(ppg,float);z=float(np.clip(z,0,1));baseline=np.nanmedian(x);ac=x-baseline
    attenuation=max(0.2,1-max_attenuation*z);noise_fraction=max_noise*z
    rng=np.random.default_rng(seed);noise=rng.normal(0,noise_fraction*max(np.nanstd(ac),1e-8),len(x))
    observed=baseline+attenuation*ac+noise
    restored=(observed-baseline)/attenuation
    kernel=max(5,int(round(0.05*fs)));kernel += 1-kernel%2;kernel=min(kernel,21)
    denoised=np.asarray(wiener(restored,mysize=kernel),float)
    denoised=np.where(np.isfinite(denoised),denoised,restored)
    blend=np.clip(0.75*z,0,0.75)
    corrected=baseline+(1-blend)*restored+blend*denoised
    noise_ratio=noise_fraction/max(attenuation,1e-6)
    snr_proxy=1/max(noise_ratio**2,1e-6);reliability=snr_proxy/(1+snr_proxy)
    return observed,corrected,attenuation,reliability,noise_ratio


## AVCT feature functions

In [ ]:
def fill_gaps(x, minimum_valid=0.95):
    x = np.asarray(x, float); good = np.isfinite(x)
    if good.mean() < minimum_valid or good.sum() < 2: return None
    if not good.all():
        i = np.arange(len(x)); x = np.interp(i, i[good], x[good])
    return x

def bandpass(x, fs, lo, hi):
    hi = min(hi, fs/2*0.95)
    return sosfiltfilt(butter(3, [lo,hi], btype='bandpass', fs=fs, output='sos'), x)

def embed(x, m=4, tau=5):
    n = len(x)-(m-1)*tau
    if n < 20: return np.empty((0,m))
    return np.column_stack([x[j*tau:j*tau+n] for j in range(m)])

def sample_entropy(x, m=2, r=0.2):
    sd=np.std(x)
    if len(x)<50 or sd<1e-10: return np.nan
    z=(x-np.mean(x))/sd
    if len(z)>600: z=z[np.linspace(0,len(z)-1,600).astype(int)]
    a=embed(z,m+1,1); b=embed(z,m,1)
    A=len(cKDTree(a).query_pairs(r,p=np.inf,output_type='ndarray'))
    B=len(cKDTree(b).query_pairs(r,p=np.inf,output_type='ndarray'))
    return float(-np.log((A+1)/(B+1)))

def permutation_entropy(x, order=3):
    if len(x)<30: return np.nan
    patterns=np.array([np.argsort(x[i:i+order]) for i in range(len(x)-order+1)])
    _,counts=np.unique(patterns,axis=0,return_counts=True); p=counts/counts.sum()
    return float(-np.sum(p*np.log(p+1e-12))/np.log(math.factorial(order)))

def runs(v):
    q=np.diff(np.r_[0,np.asarray(v,dtype=np.int8),0])
    return np.where(q==-1)[0]-np.where(q==1)[0]

def rqa(x, eps=0.20):
    sd=np.std(x)
    if sd<1e-10: return (np.nan,)*3
    e=embed((x-np.mean(x))/sd)
    if len(e)>300: e=e[np.linspace(0,len(e)-1,300).astype(int)]
    R=cdist(e,e,metric='chebyshev')<=eps; np.fill_diagonal(R,False)
    total=R.sum(); rr=total/(len(R)*(len(R)-1))
    lines=[]
    for k in range(-len(R)+1,len(R)):
        if k:
            q=runs(np.diagonal(R,k)); lines.extend(q[q>=2])
    if total==0 or not lines: return float(rr),0.0,0.0
    lines=np.asarray(lines); det=lines.sum()/total
    _,c=np.unique(lines,return_counts=True); p=c/c.sum()
    return float(rr),float(det),float(-np.sum(p*np.log(p+1e-12)))

def spectral(x,fs):
    f,p=welch(x,fs=fs,nperseg=min(len(x),int(4*fs)))
    full=(f>=0.1)&(f<=min(15,fs/2-0.1)); pulse=(f>=0.5)&(f<=5)
    total=p[full].sum()
    if total<=0:return np.nan,np.nan
    dist=p[full]/total
    return float(p[pulse].sum()/total),float(-np.sum(dist*np.log(dist+1e-12))/np.log(max(len(dist),2)))

def pat_feature(ecg,ppg,fs):
    e=bandpass(ecg,fs,5,min(25,fs/2-1)); p=bandpass(ppg,fs,.5,8)
    if np.abs(np.percentile(e,1))>np.abs(np.percentile(e,99)):e=-e
    peaks,_=find_peaks(e,distance=int(.3*fs),prominence=max(.5*np.std(e),1e-8))
    dp=np.gradient(p); values=[]
    for rpeak in peaks:
        a=rpeak+int(.08*fs); b=min(len(p),rpeak+int(.45*fs))
        if b>a+2: values.append((a+np.argmax(dp[a:b])-rpeak)/fs)
    values=[v for v in values if .08<=v<=.45]
    return float(np.median(values)) if values else np.nan

def avct_features(ppg,ecg,fs,prefix,noise_ratio=0.0):
    ppg=fill_gaps(ppg)
    if ppg is None or np.std(ppg)<1e-8:return None
    if ecg is not None:
        ecg=fill_gaps(ecg)
        if ecg is None or np.std(ecg)<1e-8:ecg=None
    f=bandpass(ppg,fs,.5,8)
    scale=max(abs(np.median(ppg)),np.percentile(ppg,95)-np.percentile(ppg,5),1e-6)
    ac=f/scale;E=embed(ac)
    rho=float(np.clip(noise_ratio,0,1.5));radius_factor=np.sqrt(1+rho**2)
    entropy_radius=0.20*radius_factor;recurrence_radius=0.20*radius_factor
    rr,det,h=rqa(ac,eps=recurrence_radius);quality,spec_h=spectral(f,fs)
    csi=0.4*det+0.3*(1-np.clip(h/5,0,1))+0.3*quality if np.all(np.isfinite([det,h,quality])) else np.nan
    return {prefix+'sigma_m':float(np.std(E)),prefix+'skew_m':float(skew(E.ravel())),
            prefix+'kurtosis_m':float(kurtosis(E.ravel())),prefix+'sample_entropy':sample_entropy(ac,r=entropy_radius),
            prefix+'permutation_entropy':permutation_entropy(ac),prefix+'recurrence_rate':rr,
            prefix+'determinism':det,prefix+'rqa_entropy':h,prefix+'csi':csi,
            prefix+'spectral_entropy':spec_h,prefix+'ppg_quality':quality,
            prefix+'pat_seconds':pat_feature(ecg,ppg,fs) if ecg is not None else np.nan}

def all_channels(ppg,ecg,fs,z,seed):
    observed,corrected,attenuation,reliability,noise_ratio=optical_stress(ppg,z,seed,fs)
    ref=avct_features(ppg,ecg,fs,'ref_',0.0)
    raw=avct_features(observed,ecg,fs,'raw_',noise_ratio)
    pc=avct_features(corrected,ecg,fs,'pc_',0.35*noise_ratio)
    if ref is None or raw is None or pc is None:return None
    return {**ref,**raw,**pc,'optical_attenuation':attenuation,
            'optical_reliability':reliability,'optical_noise_ratio':noise_ratio}


## BIDMC, MIMIC/Welltory, and UCI feature banks

Required: these define `RAW`, `PC`, `make_model`, and the UCI prior.

In [ ]:
bidmc_files=sorted(BIDMC_DIR.glob('bidmc_*_Signals.csv'))
assert bidmc_files, f'No BIDMC signals found in {BIDMC_DIR}'
bidmc_ids=[re.search(r'bidmc_(\d+)',p.name).group(1) for p in bidmc_files]
bidmc_skin=synthetic_skin_labels(bidmc_ids,SEED,'BIDMC')
rows=[]
audit={'files':len(bidmc_files),'candidate_windows':0,'accepted_windows':0,'ppg_rejected':0,'feature_rejected':0,'ecg_missing_windows':0}

for path in tqdm(bidmc_files,desc='BIDMC'):
    sid=re.search(r'bidmc_(\d+)',path.name).group(1)
    label=bidmc_skin[bidmc_skin.subject_id==sid].iloc[0]
    d=pd.read_csv(path)
    d.columns=d.columns.astype(str).str.strip()
    if 'PLETH' not in d or 'Time [s]' not in d:
        continue
    t=pd.to_numeric(d['Time [s]'],errors='coerce').to_numpy(float)
    valid_time=t[np.isfinite(t)]
    if len(valid_time)<2 or valid_time[-1]<=valid_time[0]:
        continue
    fs=(len(valid_time)-1)/(valid_time[-1]-valid_time[0])
    ppg=pd.to_numeric(d['PLETH'],errors='coerce').to_numpy(float)

    # Select the ECG lead with the best finite coverage and variance.
    ecg=None
    lead_scores=[]
    for lead in ['II','V','AVR']:
        if lead in d:
            candidate=pd.to_numeric(d[lead],errors='coerce').to_numpy(float)
            finite=np.isfinite(candidate)
            score=(finite.mean(),np.nanstd(candidate) if finite.any() else 0)
            lead_scores.append((score,lead,candidate))
    if lead_scores:
        _,ecg_lead,ecg=max(lead_scores,key=lambda item:item[0])
    else:
        ecg_lead='none'

    n=int(round(fs*WINDOW_SECONDS))
    for j,start in enumerate(np.arange(0,len(d)-n+1,n)[:MAX_BIDMC_WINDOWS]):
        audit['candidate_windows']+=1
        ppg_window=ppg[start:start+n]
        if np.isfinite(ppg_window).mean()<0.90 or np.nanstd(ppg_window)<1e-8:
            audit['ppg_rejected']+=1
            continue
        ecg_window=None if ecg is None else ecg[start:start+n]
        if ecg_window is not None and (np.isfinite(ecg_window).mean()<0.90 or np.nanstd(ecg_window)<1e-8):
            ecg_window=None
        if ecg_window is None:
            audit['ecg_missing_windows']+=1
        f=all_channels(ppg_window,ecg_window,fs,label.pigmentation_index,SEED+int(sid)*1000+j)
        if f is None:
            audit['feature_rejected']+=1
            continue
        rows.append({'subject_id':sid,'start_seconds':float(t[start]),'ecg_lead':ecg_lead,**label.to_dict(),**f})
        audit['accepted_windows']+=1

bidmc_features=pd.DataFrame(rows)
print('BIDMC audit:',audit)
if bidmc_features.empty:
    sample=pd.read_csv(bidmc_files[0],nrows=5)
    raise RuntimeError(
        'No BIDMC windows passed processing. First-file columns: '
        f'{sample.columns.tolist()}. Run the audit cell and share its output.'
    )
bidmc_features.to_csv(OUT_DIR/'bidmc_features.csv',index=False)
print('BIDMC subjects/windows:',bidmc_features['subject_id'].nunique(),len(bidmc_features))
display(bidmc_features.head())


In [ ]:
PPG_SHARED=['sigma_m','skew_m','kurtosis_m','sample_entropy','permutation_entropy','recurrence_rate','determinism','rqa_entropy','csi','spectral_entropy','ppg_quality']

def ppg_only_channels(ppg,fs,z,seed):
    observed,corrected,attenuation,reliability,noise_ratio=optical_stress(ppg,z,seed,fs)
    dummy=np.sin(2*np.pi*1.2*np.arange(len(ppg))/fs)
    ref=avct_features(ppg,dummy,fs,'ref_',0.0);raw=avct_features(observed,dummy,fs,'raw_',noise_ratio);pc=avct_features(corrected,dummy,fs,'pc_',0.35*noise_ratio)
    if ref is None or raw is None or pc is None:return None
    keep={}
    for prefix,source in [('ref_',ref),('raw_',raw),('pc_',pc)]:
        for name in PPG_SHARED:keep[prefix+name]=source[prefix+name]
    return {**keep,'optical_attenuation':attenuation,'optical_reliability':reliability,'optical_noise_ratio':noise_ratio}

def npy(path):
    try:return np.load(path,mmap_mode='r',allow_pickle=True)
    except ValueError:return np.load(path,allow_pickle=True)

Xppg=npy(MIMIC_DIR/'X_ppg.npy');pids=npy(MIMIC_DIR/'patient_ids.npy') if (MIMIC_DIR/'patient_ids.npy').exists() else np.arange(len(Xppg))
rates=npy(MIMIC_DIR/'sampling_rates.npy') if (MIMIC_DIR/'sampling_rates.npy').exists() else np.full(len(Xppg),125)
n_patients=min(len(Xppg),len(pids))
ids=[str(np.asarray(pids[i]).squeeze().decode() if isinstance(np.asarray(pids[i]).squeeze(),bytes) else np.asarray(pids[i]).squeeze()) for i in range(n_patients)]
skin=synthetic_skin_labels(ids,SEED+10,'MIMIC_numpy');mimic_rows=[];mimic_count=0
for i in tqdm(range(n_patients),desc='MIMIC PPG'):
    sid=ids[i];label=skin[skin.subject_id==sid].iloc[0];patient_ppg=np.asarray(Xppg[i],float).squeeze()
    segments=[patient_ppg] if patient_ppg.ndim==1 else patient_ppg.reshape(-1,patient_ppg.shape[-1])
    try:fs=float(np.asarray(rates[i] if np.ndim(rates) else rates).reshape(-1)[0])
    except Exception:fs=125.
    if not np.isfinite(fs) or fs<10 or fs>1000:fs=125.
    need=int(fs*WINDOW_SECONDS)
    for segment_index,x in enumerate(segments):
        if mimic_count>=MAX_MIMIC_SEGMENTS:break
        if len(x)<need or np.nanstd(x)<1e-8:continue
        start=(len(x)-need)//2;f=ppg_only_channels(x[start:start+need],fs,label.pigmentation_index,SEED+100000+i*1000+segment_index)
        if f:
            mimic_rows.append({'subject_id':sid,'segment_index':segment_index,'source':'MIMIC_numpy',**label.to_dict(),**f})
            mimic_count+=1
    if mimic_count>=MAX_MIMIC_SEGMENTS:break
mimic_features=pd.DataFrame(mimic_rows);mimic_features.to_csv(OUT_DIR/'mimic_ppg_features.csv',index=False)

well_files=sorted(WELLTORY_DIR.rglob('PPG.csv'));well_ids=[p.parent.name for p in well_files]
well_skin=synthetic_skin_labels(well_ids,SEED+20,'Welltory');well_rows=[]
for p in tqdm(well_files,desc='Welltory'):
    sid=p.parent.name;label=well_skin[well_skin.subject_id==sid].iloc[0];d=pd.read_csv(p);cols={c.lower():c for c in d}
    if 'time' not in cols or 'g' not in cols:continue
    t=pd.to_numeric(d[cols['time']],errors='coerce').to_numpy(float);x=pd.to_numeric(d[cols['g']],errors='coerce').to_numpy(float)
    fs=1000/np.nanmedian(np.diff(t));need=int(fs*WINDOW_SECONDS)
    if not 10<=fs<=240:continue
    for j,start in enumerate(np.arange(0,len(x)-need+1,need)[:MAX_WELLTORY_WINDOWS]):
        f=ppg_only_channels(x[start:start+need],fs,label.pigmentation_index,SEED+200000+100*well_ids.index(sid)+j)
        if f:well_rows.append({'subject_id':sid,'source':'Welltory',**label.to_dict(),**f})
welltory_features=pd.DataFrame(well_rows);welltory_features.to_csv(OUT_DIR/'welltory_ppg_features.csv',index=False)
print('MIMIC:',len(mimic_features),'Welltory:',len(welltory_features))


In [ ]:
def bp_labels(art,fs=125):
    art=fill_gaps(art)
    if art is None or np.std(art)<3:return None
    peaks,_=find_peaks(art,distance=int(.35*fs),prominence=10,height=(60,260))
    if len(peaks)<5:return None
    sbp=art[peaks];dbp=np.asarray([np.min(art[a:b]) for a,b in zip(peaks[:-1],peaks[1:])])
    sbp=sbp[(sbp>=70)&(sbp<=250)];dbp=dbp[(dbp>=25)&(dbp<=150)]
    if len(sbp)<4 or len(dbp)<4:return None
    s,d=float(np.median(sbp)),float(np.median(dbp))
    return (s,d,(s+2*d)/3) if 15<=s-d<=150 else None

parts=sorted(CUFFLESS_DIR.rglob('Part_*.mat'));assert parts,'No UCI Part_*.mat files.'
def cell_key(h):
    keys=[k for k in h.keys() if not k.startswith('#') and isinstance(h[k],h5py.Dataset)]
    return max(keys,key=lambda k:h[k].size)

record_ids=[]
for p in parts:
    with h5py.File(p,'r') as h:record_ids += [f'{p.stem}_{i:05d}' for i in range(min(h[cell_key(h)].size,MAX_CUFF_RECORDS_PER_PART))]
uci_skin=synthetic_skin_labels(record_ids,SEED+30,'UCI_cuffless');rows=[]
for p in tqdm(parts,desc='UCI parts'):
    with h5py.File(p,'r') as h:
        refs=np.asarray(h[cell_key(h)]).reshape(-1)[:MAX_CUFF_RECORDS_PER_PART]
        for i,ref in enumerate(refs):
            if not ref:continue
            a=np.asarray(h[ref],float).squeeze()
            if a.ndim!=2:continue
            if a.shape[0]!=3 and a.shape[1]==3:a=a.T
            if a.shape[0]!=3:continue
            ppg,abp,ecg=a;rid=f'{p.stem}_{i:05d}';label=uci_skin[uci_skin.subject_id==rid].iloc[0];need=125*WINDOW_SECONDS
            for j,start in enumerate(np.arange(0,len(ppg)-need+1,need)[:MAX_CUFF_WINDOWS_PER_RECORD]):
                bp=bp_labels(abp[start:start+need]);f=all_channels(ppg[start:start+need],ecg[start:start+need],125,label.pigmentation_index,SEED+300000+i*100+j) if bp else None
                if f:rows.append({'subject_id':rid,'part':p.stem,'start_seconds':start/125,'sbp':bp[0],'dbp':bp[1],'map':bp[2],**label.to_dict(),**f})
uci_features=pd.DataFrame(rows);uci_features.to_csv(OUT_DIR/'uci_cuffless_features.csv',index=False)
print('UCI records/windows:',uci_features.subject_id.nunique(),len(uci_features));display(uci_features.head())


In [ ]:
shared=['pc_'+x for x in PPG_SHARED]
banks=[]
for name,frame in [('BIDMC',bidmc_features),('MIMIC',mimic_features),('Welltory',welltory_features)]:
    if len(frame):banks.append(frame.sample(min(len(frame),1000),random_state=SEED).assign(bank_source=name))
bank=pd.concat(banks,ignore_index=True);imp=SimpleImputer(strategy='median');scale=RobustScaler()
Z=scale.fit_transform(imp.fit_transform(bank[shared].replace([np.inf,-np.inf],np.nan)))
pca=PCA(n_components=min(8,len(shared),Z.shape[0]-1),random_state=SEED).fit(Z)
Uz=pca.transform(scale.transform(imp.transform(uci_features[shared].replace([np.inf,-np.inf],np.nan))))
LAT=[]
for j in range(Uz.shape[1]):
    name=f'external_ppg_pc{j+1}';uci_features[name]=Uz[:,j];LAT.append(name)
joblib.dump({'features':shared,'imputer':imp,'scaler':scale,'pca':pca},OUT_DIR/'external_ppg_representation_v4.joblib')

RAW=[c for c in uci_features if c.startswith('raw_')];PC=[c for c in uci_features if c.startswith('pc_')]
def weights(d):
    counts=d.groupby('monk_group')['subject_id'].transform('count')
    w=len(d)/(d.monk_group.nunique()*counts)*d.optical_reliability.clip(.1,1)
    return np.asarray(w/w.mean())

def make_model(params):
    return make_pipeline(SimpleImputer(strategy='median'),HistGradientBoostingRegressor(
        max_iter=300,random_state=SEED,**params))

def nested_oof(d,features,target,weighted):
    d=d.reset_index(drop=True);X=d[features].replace([np.inf,-np.inf],np.nan)
    groups=d.subject_id.astype(str);pred=np.full(len(d),np.nan);selected=[]
    outer=GroupKFold(n_splits=min(5,groups.nunique()))
    for outer_fold,(tr,te) in enumerate(outer.split(X,d[target],groups),1):
        train=d.iloc[tr].reset_index(drop=True);Xtr=X.iloc[tr].reset_index(drop=True)
        gtr=train.subject_id.astype(str);inner=GroupKFold(n_splits=min(3,gtr.nunique()))
        scores=[]
        for params in NESTED_PARAMETER_GRID:
            fold_scores=[]
            for itr,iva in inner.split(Xtr,train[target],gtr):
                model=make_model(params)
                kw={'histgradientboostingregressor__sample_weight':weights(train.iloc[itr])} if weighted else {}
                model.fit(Xtr.iloc[itr],train[target].iloc[itr],**kw)
                fold_scores.append(mean_absolute_error(train[target].iloc[iva],model.predict(Xtr.iloc[iva])))
            scores.append(np.mean(fold_scores))
        best=NESTED_PARAMETER_GRID[int(np.argmin(scores))]
        model=make_model(best);kw={'histgradientboostingregressor__sample_weight':weights(d.iloc[tr])} if weighted else {}
        model.fit(X.iloc[tr],d[target].iloc[tr],**kw);pred[te]=model.predict(X.iloc[te])
        selected.append({'target':target,'outer_fold':outer_fold,'inner_mae':min(scores),**best})
    return pred,pd.DataFrame(selected)

models={'raw':(RAW,False),'pc':(PC+['optical_reliability','optical_noise_ratio'],True),
        'pc_external':(PC+LAT+['optical_reliability','optical_noise_ratio'],True)}
results=uci_features.reset_index(drop=True).copy();tuning=[]
for target in ['sbp','dbp']:
    for name,(features,weighted) in models.items():
        results[f'{target}_{name}'],selected=nested_oof(results,features,target,weighted)
        selected['model']=name;tuning.append(selected)
tuning_results=pd.concat(tuning,ignore_index=True)
tuning_results.to_csv(OUT_DIR/'nested_model_selection_v4.csv',index=False)
display(tuning_results)


## Delta-table construction

In [ ]:
# Construct a leakage-safe delta table. UCI retains the original session-start
# anchor; Aurora explicitly requests the most recent valid PRIOR measurement.
def build_delta_table(frame, raw_features, pc_features, anchor='session_start'):
    if anchor not in {'session_start', 'most_recent_prior'}:
        raise ValueError("anchor must be 'session_start' or 'most_recent_prior'")
    features = list(dict.fromkeys(list(raw_features) + list(pc_features)))
    rows = []
    for subject_id, g in frame.groupby('subject_id', sort=False):
        order_columns = [c for c in ['start_seconds', '_source_order'] if c in g.columns]
        g = g.sort_values(order_columns, na_position='last') if order_columns else g.copy()
        if len(g) < 2:
            continue
        session = g.iloc[0]
        for position in range(1, len(g)):
            row = g.iloc[position]
            prior = g.iloc[position - 1]
            reference = prior if anchor == 'most_recent_prior' else session
            item = row.to_dict()
            item['original_index'] = g.index[position]
            item['anchor_mode'] = anchor
            item['anchor_start_seconds'] = pd.to_numeric(
                pd.Series([reference.get('start_seconds')]), errors='coerce').iloc[0]
            item['anchor_phase'] = reference.get('phase', np.nan)
            item['anchor_measurement'] = reference.get('measurement', np.nan)

            row_time = pd.to_numeric(pd.Series([row.get('start_seconds')]), errors='coerce').iloc[0]
            ref_time = pd.to_numeric(pd.Series([reference.get('start_seconds')]), errors='coerce').iloc[0]
            session_time = pd.to_numeric(pd.Series([session.get('start_seconds')]), errors='coerce').iloc[0]
            item['elapsed_seconds'] = float(row_time - ref_time) if np.isfinite(row_time) and np.isfinite(ref_time) else np.nan
            item['prior_gap_seconds'] = item['elapsed_seconds']
            item['session_elapsed_seconds'] = float(row_time - session_time) if np.isfinite(row_time) and np.isfinite(session_time) else np.nan

            for target in ['sbp', 'dbp']:
                current = pd.to_numeric(pd.Series([row.get(target)]), errors='coerce').iloc[0]
                ref_value = pd.to_numeric(pd.Series([reference.get(target)]), errors='coerce').iloc[0]
                session_value = pd.to_numeric(pd.Series([session.get(target)]), errors='coerce').iloc[0]
                item['baseline_' + target] = float(ref_value) if np.isfinite(ref_value) else np.nan
                item['delta_' + target] = float(current - ref_value) if np.isfinite(current) and np.isfinite(ref_value) else np.nan
                item['session_baseline_' + target] = float(session_value) if np.isfinite(session_value) else np.nan
                item['session_delta_' + target] = float(current - session_value) if np.isfinite(current) and np.isfinite(session_value) else np.nan

            for feature in features:
                current = pd.to_numeric(pd.Series([row.get(feature)]), errors='coerce').iloc[0]
                initial = pd.to_numeric(pd.Series([reference.get(feature)]), errors='coerce').iloc[0]
                valid = np.isfinite(current) and np.isfinite(initial)
                item['delta_' + feature] = float(current - initial) if valid else np.nan
                denominator = max(abs(initial), 1e-6) if np.isfinite(initial) else np.nan
                relative = (current - initial) / denominator if valid else np.nan
                item['relative_' + feature] = float(np.clip(relative, -10, 10)) if np.isfinite(relative) else np.nan
            rows.append(item)
    return pd.DataFrame(rows)


# Keep the UCI prior on its original definition. Aurora opts into prior anchoring below.
delta_table = build_delta_table(uci_features, RAW, PC, anchor='session_start')
assert len(delta_table), 'No records had at least two valid BP windows.'
print('Delta-evaluation records/windows:', delta_table.subject_id.nunique(), len(delta_table))
display(delta_table[['subject_id', 'start_seconds', 'elapsed_seconds',
                     'baseline_sbp', 'sbp', 'delta_sbp', 'monk_group']].head())


In [ ]:
DELTA_RAW=['delta_'+f for f in RAW]
DELTA_PC=['delta_'+f for f in PC]
RELATIVE_PC=['relative_'+f for f in PC]


## Aurora load, waveform features, and the calibrated delta table

In [ ]:
from zipfile import ZipFile
import os, shutil

AURORA_ZIP=Path('/content/drive/MyDrive/AuroraBP.zip')
AURORA_WORK=Path('/content/AuroraBP_external_v6')
AURORA_RESULTS=OUT_DIR/'aurora_external_v6'
AURORA_WORK.mkdir(parents=True,exist_ok=True);AURORA_RESULTS.mkdir(parents=True,exist_ok=True)
assert AURORA_ZIP.exists(),f'File not found: {AURORA_ZIP}'

def safe_extract(zip_path,destination):
    destination=Path(destination).resolve()
    with ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target=(destination/member.filename).resolve()
            if not (target==destination or str(target).startswith(str(destination)+os.sep)):
                raise RuntimeError(f'Unsafe ZIP path: {member.filename}')
        zf.extractall(destination)

if not list(AURORA_WORK.rglob('participants.tsv')):
    safe_extract(AURORA_ZIP,AURORA_WORK)
nested=list(AURORA_WORK.rglob('measurements_auscultatory.zip'))
raw_waveforms=[p for p in AURORA_WORK.rglob('*.tsv') if 'measurements_auscultatory' in p.parts and p.name!='measurements_auscultatory.tsv']
if nested and not raw_waveforms:
    for archive in nested:safe_extract(archive,archive.parent)

def find_aurora_file(name,required=True):
    matches=sorted(AURORA_WORK.rglob(name),key=lambda p:(len(p.parts),len(str(p))))
    if not matches and required:raise FileNotFoundError(name)
    return matches[0] if matches else None

participants_path=find_aurora_file('participants.tsv')
measurements_path=find_aurora_file('measurements_auscultatory.tsv')
participants=pd.read_csv(participants_path,sep='\t',na_values=['NA','N/A','NaN','nan',''])
measurements=pd.read_csv(measurements_path,sep='\t',na_values=['NA','N/A','NaN','nan',''],low_memory=False)
participants['fitzpatrick_scale']=pd.to_numeric(participants['fitzpatrick_scale'],errors='coerce')
participants.loc[~participants.fitzpatrick_scale.between(1,6),'fitzpatrick_scale']=np.nan
required={'pid','phase','measurement','sbp','dbp','waveform_file_path'}
assert required.issubset(measurements.columns),required-set(measurements.columns)
print('Aurora participants/measurements:', participants.pid.nunique(), len(measurements))
display(participants.fitzpatrick_scale.value_counts(dropna=False).sort_index())

# Measurement-context audit: report the real schema before deriving anything.
print('\nAurora measurement columns:')
print(measurements.columns.tolist())
print('\nObserved phases:')
display(measurements.phase.astype(str).value_counts(dropna=False).rename('rows').to_frame())
print('\nObserved measurement labels (all labels with row counts):')
display(measurements.measurement.astype(str).value_counts(dropna=False).rename('rows').to_frame())
_documented_context_candidates = [
    'phase', 'measurement', 'date_time', 'duration', 'pressure_quality',
    'optical_quality', 'waveforms_generated', 'activity', 'posture'
]
measurement_context_schema = pd.DataFrame({
    'candidate': _documented_context_candidates,
    'present': [c in measurements.columns for c in _documented_context_candidates]
})
print('\nContext-field availability (no missing field will be invented):')
display(measurement_context_schema)


In [ ]:
def resolve_aurora_waveform(value):
    rel=Path(str(value));candidates=[rel,measurements_path.parent/rel,AURORA_WORK/rel]
    for candidate in candidates:
        if candidate.exists():return candidate
    matches=list(AURORA_WORK.rglob(rel.name))
    if matches:return matches[0]
    raise FileNotFoundError(str(rel))

def real_fitzpatrick_channels(ppg,fs,fitzpatrick):
    x=fill_gaps(ppg,minimum_valid=0.90)
    if x is None:return None
    fst=float(np.clip(fitzpatrick,1,6));z=(fst-1)/5
    baseline=np.median(x);ac=x-baseline
    pulse=bandpass(x,fs,.5,8)
    residual=ac-pulse
    signal_sd=max(np.std(pulse),1e-8);noise_sd=np.std(residual)
    noise_ratio=float(np.clip(noise_sd/signal_sd,0,1.5))
    attenuation=max(0.45,1-0.55*z)
    restored=ac/attenuation
    kernel=max(5,int(round(.05*fs)));kernel+=1-kernel%2;kernel=min(kernel,21)
    denoised=np.asarray(wiener(restored,mysize=kernel),float)
    denoised=np.where(np.isfinite(denoised),denoised,restored)
    blend=np.clip(.75*z,0,.75);corrected=baseline+(1-blend)*restored+blend*denoised
    reliability=1/(1+noise_ratio**2)
    raw=avct_features(x,None,fs,'raw_',noise_ratio)
    pc=avct_features(corrected,None,fs,'pc_',0.35*noise_ratio)
    if raw is None or pc is None:return None
    return {**raw,**pc,'optical_attenuation':attenuation,
            'optical_reliability':reliability,'optical_noise_ratio':noise_ratio}

aurora_cache=AURORA_RESULTS/'aurora_avct_features.csv'
if aurora_cache.exists():
    aurora_features=pd.read_csv(aurora_cache)
else:
    metadata=measurements.merge(
        participants[['pid','fitzpatrick_scale']].drop_duplicates('pid'),
        on='pid',how='left',validate='many_to_one'
    )
    metadata['_source_order']=np.arange(len(metadata));rows=[];failures=[]
    for _,row in tqdm(metadata.dropna(subset=['fitzpatrick_scale','waveform_file_path']).iterrows(),
                      total=metadata[['fitzpatrick_scale','waveform_file_path']].dropna().shape[0],desc='Aurora AVCT'):
        try:
            path=resolve_aurora_waveform(row.waveform_file_path)
            w=pd.read_csv(path,sep='\t',usecols=lambda c:c in {'t','optical'},low_memory=False)
            t=pd.to_numeric(w.t,errors='coerce').to_numpy(float);x=pd.to_numeric(w.optical,errors='coerce').to_numpy(float)
            valid=np.isfinite(t)&np.isfinite(x);t,x=t[valid],x[valid];order=np.argsort(t);t,x=t[order],x[order]
            if len(x)<1000 or t[-1]<=t[0]:raise ValueError('waveform too short')
            fs=(len(t)-1)/(t[-1]-t[0]);need=int(round(WINDOW_SECONDS*fs))
            if len(x)<need:raise ValueError('less than 10 seconds')
            start=(len(x)-need)//2;f=real_fitzpatrick_channels(x[start:start+need],fs,row.fitzpatrick_scale)
            if f is None:raise ValueError('feature rejection')
            rows.append({'pid':row.pid,'phase':row.phase,'measurement':row.measurement,
                         'sbp':float(row.sbp),'dbp':float(row.dbp),'fitzpatrick_scale':float(row.fitzpatrick_scale),
                         'date_time':row.get('date_time',np.nan),'_source_order':int(row._source_order),**f})
        except Exception as exc:
            failures.append({'pid':row.pid,'phase':row.phase,'measurement':row.measurement,'error':str(exc)})
    aurora_features=pd.DataFrame(rows);aurora_features.to_csv(aurora_cache,index=False)
    pd.DataFrame(failures).to_csv(AURORA_RESULTS/'aurora_feature_failures.csv',index=False)


# Reattach measurement-level metadata even when the waveform feature cache was
# created by an older notebook version. Keys are defined by the Aurora schema.
_context_source_candidates = [
    'date_time', 'duration', 'pressure_quality', 'optical_quality',
    'waveforms_generated', 'activity', 'posture'
]
_context_source_columns = [c for c in _context_source_candidates if c in measurements.columns]
_context_keys = ['pid', 'phase', 'measurement']
_measurement_context = measurements[_context_keys + _context_source_columns].copy()
if _measurement_context.duplicated(_context_keys).any():
    raise ValueError('Measurement context keys are not unique; refusing an ambiguous merge.')
for column in _context_source_columns:
    if column in aurora_features.columns:
        continue
    aurora_features = aurora_features.merge(
        _measurement_context[_context_keys + [column]], on=_context_keys,
        how='left', validate='many_to_one')

aurora_features['fst_group']=pd.cut(
    aurora_features.fitzpatrick_scale,[0,2,4,6],
    labels=['Fitzpatrick I-II','Fitzpatrick III-IV','Fitzpatrick V-VI']
).astype(str)
print('Aurora usable participants/measurements:',aurora_features.pid.nunique(),len(aurora_features))
display(aurora_features.head())


In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import Ridge

V9_RESULTS=OUT_DIR/'aurora_calibration_corrected_v9'
V9_RESULTS.mkdir(parents=True,exist_ok=True)
V9_OUTER_FOLDS=5
V9_TOP_K=[12,24]
V9_ALPHA_GRID=np.round(np.linspace(0,1,11),2)
V9_FAIRNESS_LAMBDA=.15
V9_DEGRADATION_PENALTY=2.0
V9_PARAMETER_GRID=[
    {'learning_rate':.03,'max_leaf_nodes':15,'l2_regularization':1.0},
    {'learning_rate':.03,'max_leaf_nodes':31,'l2_regularization':2.0},
    {'learning_rate':.05,'max_leaf_nodes':15,'l2_regularization':2.0},
]

def build_replicate_calibrated_aurora(frame):
    source=frame.copy()
    source['sbp']=pd.to_numeric(source.sbp,errors='coerce')
    source['dbp']=pd.to_numeric(source.dbp,errors='coerce')
    source['_parsed_time']=pd.to_datetime(source.date_time,errors='coerce')
    source['_is_calibration_start']=(
        source.phase.astype(str).str.lower().eq('initial') &
        source.measurement.astype(str).str.contains(
            r'^\s*calibration\s+start',case=False,regex=True,na=False
        )
    )
    rows=[];audit=[]
    numeric_columns=source.select_dtypes(include=[np.number]).columns.tolist()
    feature_columns=[c for c in numeric_columns if c.startswith(('raw_','pc_','h1_'))]
    for pid,g in source.dropna(subset=['sbp','dbp']).groupby('pid'):
        g=g.sort_values(['_parsed_time','_source_order'],na_position='last')
        calibrations=g[g._is_calibration_start].copy()
        if len(calibrations)<2:
            audit.append({'pid':pid,'status':'excluded','calibration_replicates':len(calibrations)})
            continue
        # Only calibration-start replicates are averaged; challenge readings remain separate.
        last_cal_order=calibrations._source_order.max()
        last_cal_time=calibrations._parsed_time.max()
        baseline=calibrations.iloc[0].copy()
        baseline['sbp']=calibrations.sbp.mean()
        baseline['dbp']=calibrations.dbp.mean()
        for column in feature_columns:
            baseline[column]=pd.to_numeric(calibrations[column],errors='coerce').median()
        baseline['_source_order']=last_cal_order
        baseline['_parsed_time']=last_cal_time
        baseline['date_time']=last_cal_time
        baseline['measurement']='Calibration start replicate mean'
        baseline['subject_id']=str(pid)
        baseline['start_seconds']=0.0

        if pd.notna(last_cal_time):
            future=g[(~g._is_calibration_start)&(
                (g._parsed_time>last_cal_time)|
                (g._parsed_time.eq(last_cal_time)&(g._source_order>last_cal_order))
            )]
        else:
            future=g[(~g._is_calibration_start)&(g._source_order>last_cal_order)]
        if future.empty:
            audit.append({'pid':pid,'status':'excluded_no_future',
                          'calibration_replicates':len(calibrations)})
            continue
        rows.append(baseline)
        for _,row in future.iterrows():
            item=row.copy();item['subject_id']=str(pid)
            if pd.notna(row._parsed_time) and pd.notna(last_cal_time):
                item['start_seconds']=(row._parsed_time-last_cal_time).total_seconds()
            else:
                item['start_seconds']=float((row._source_order-last_cal_order)*WINDOW_SECONDS)
            rows.append(item)
        audit.append({'pid':pid,'status':'included','calibration_replicates':len(calibrations),
                      'future_measurements':len(future)})
    return pd.DataFrame(rows),pd.DataFrame(audit)


def add_aurora_measurement_context(frame):
    """Create only context features supported by columns/labels in `frame`."""
    out = frame.copy()
    created, audit = [], []

    if 'phase' in out.columns:
        phase = out.phase.fillna('missing').astype(str).str.strip().str.lower()
        for value in sorted(v for v in phase.unique() if v and v != 'missing'):
            safe = re.sub(r'[^a-z0-9]+', '_', value).strip('_')
            column = 'context_phase_' + safe
            out[column] = (phase == value).astype(float)
            created.append(column)
        if phase.eq('ambulatory').any():
            out['context_is_ambulatory'] = phase.eq('ambulatory').astype(float)
            out['context_is_in_lab'] = phase.isin(['initial', 'return']).astype(float)
            created += ['context_is_ambulatory', 'context_is_in_lab']
            audit.append({'source': 'phase', 'derived': 'in-lab/ambulatory', 'status': 'added'})
        else:
            audit.append({'source': 'phase', 'derived': 'in-lab/ambulatory',
                          'status': 'not added: no ambulatory rows in this subset'})

    label = (out['measurement'].fillna('').astype(str).str.lower()
             if 'measurement' in out.columns else pd.Series('', index=out.index))
    semantic_patterns = {
        'context_posture_supine': r'\bsupine\b',
        'context_posture_seated': r'\b(seated|sitting)\b',
        'context_posture_standing': r'\bstanding\b',
        'context_activity_post_exercise': r'\b(post[ -]?exercise|exercise)\b',
        'context_measurement_calibration': r'\bcalibration\b',
    }
    for column, pattern in semantic_patterns.items():
        matched = label.str.contains(pattern, regex=True, na=False)
        if matched.any():
            out[column] = matched.astype(float)
            created.append(column)
            audit.append({'source': 'measurement label', 'derived': column,
                          'status': f'added ({int(matched.sum())} rows)'})
        else:
            audit.append({'source': 'measurement label', 'derived': column,
                          'status': 'not added: no matching observed labels'})

    numeric_context = {
        'duration': 'context_duration_seconds',
        'pressure_quality': 'context_pressure_quality',
        'optical_quality': 'context_optical_quality',
    }
    for source, column in numeric_context.items():
        if source in out.columns:
            values = pd.to_numeric(out[source], errors='coerce')
            if values.notna().any():
                out[column] = values
                created.append(column)
                audit.append({'source': source, 'derived': column, 'status': 'added'})
            else:
                audit.append({'source': source, 'derived': column, 'status': 'not added: all missing'})

    created = list(dict.fromkeys(created))
    return out, created, pd.DataFrame(audit)


aurora_v9_ordered, aurora_v9_audit = build_replicate_calibrated_aurora(aurora_features)
aurora_v9 = build_delta_table(
    aurora_v9_ordered, RAW, PC, anchor='most_recent_prior'
).replace([np.inf, -np.inf], np.nan)
aurora_v9, AURORA_CONTEXT_FEATURES, aurora_context_audit = add_aurora_measurement_context(aurora_v9)
aurora_v9['subject_id'] = aurora_v9.subject_id.astype(str)
aurora_v9['fst_band'] = pd.cut(
    aurora_v9.fitzpatrick_scale, [0, 2, 4, 6],
    labels=['Fitzpatrick I-II', 'Fitzpatrick III-IV', 'Fitzpatrick V-VI']
).astype(str)
aurora_v9['fst_z'] = (aurora_v9.fitzpatrick_scale - 1) / 5
aurora_v9['dark_skin_indicator'] = (aurora_v9.fitzpatrick_scale >= 5).astype(float)
# elapsed_seconds is now the strictly prior-measurement gap. The original
# day-scale clock remains available separately for context and diagnostics.
aurora_v9['log_elapsed_days'] = np.log1p(aurora_v9.elapsed_seconds.clip(lower=0) / 86400)
aurora_v9['log_session_elapsed_days'] = np.log1p(
    aurora_v9.session_elapsed_seconds.clip(lower=0) / 86400)
aurora_v9['fst_time_interaction'] = aurora_v9.fst_z * aurora_v9.log_session_elapsed_days
aurora_v9['fst_noise_interaction'] = aurora_v9.fst_z * aurora_v9.optical_noise_ratio
aurora_v9['fst_reliability_interaction'] = aurora_v9.fst_z * aurora_v9.optical_reliability
aurora_v9['synthetic_monk'] = np.rint(1 + 9 * aurora_v9.fst_z).clip(1, 10).astype(int)
aurora_v9['synthetic_monk_band'] = pd.cut(
    aurora_v9.synthetic_monk, [0, 3, 6, 10],
    labels=['Monk 1-3', 'Monk 4-6', 'Monk 7-10']).astype(str)

included_audit=aurora_v9_audit[aurora_v9_audit.status=='included']
print('Participants with >=2 calibration replicates:',len(included_audit))
print('Future measurements scored:',len(aurora_v9))
display(aurora_v9_audit.status.value_counts())
display(aurora_v9[['subject_id', 'anchor_measurement', 'baseline_sbp', 'sbp',
                   'delta_sbp', 'elapsed_seconds', 'session_elapsed_seconds',
                   'fst_band']].head())
print('Context features created:', AURORA_CONTEXT_FEATURES)
display(aurora_context_audit)


## Cross-dataset representation, information ranking, and gate grids

In [ ]:
COMMON_PC=['pc_'+name for name in PPG_SHARED]
external_representation_parts=[]
for source_name,frame in [('BIDMC',bidmc_features),('MIMIC',mimic_features),('Welltory',welltory_features)]:
    available=[c for c in COMMON_PC if c in frame.columns]
    if len(frame) and len(available)==len(COMMON_PC):
        external_representation_parts.append(
            frame[COMMON_PC].sample(min(len(frame),1200),random_state=SEED).assign(_source=source_name)
        )
external_representation_bank=pd.concat(external_representation_parts,ignore_index=True)
print(external_representation_bank._source.value_counts())

def fit_cross_dataset_representation(fit_frame,n_components=8):
    current=fit_frame[COMMON_PC].copy()
    baseline=pd.DataFrame({c:fit_frame[c]-fit_frame['delta_'+c] for c in COMMON_PC})
    aurora_fit=pd.concat([current,baseline],ignore_index=True)
    bank=pd.concat([external_representation_bank[COMMON_PC],aurora_fit],ignore_index=True)
    imputer=SimpleImputer(strategy='median').fit(bank)
    scaler=RobustScaler().fit(imputer.transform(bank))
    Z=scaler.transform(imputer.transform(bank))
    pca=PCA(n_components=min(n_components,len(COMMON_PC)),random_state=SEED).fit(Z)
    return {'imputer':imputer,'scaler':scaler,'pca':pca}

def add_representation(frame,bundle):
    out=frame.copy()
    current=out[COMMON_PC]
    baseline=pd.DataFrame({c:out[c]-out['delta_'+c] for c in COMMON_PC},index=out.index)
    def transform(values):
        return bundle['pca'].transform(
            bundle['scaler'].transform(bundle['imputer'].transform(values))
        )
    current_z=transform(current);baseline_z=transform(baseline)
    names=[]
    for j in range(current_z.shape[1]):
        name=f'cross_dataset_delta_pc{j+1}'
        out[name]=current_z[:,j]-baseline_z[:,j];names.append(name)
    return out,names

RELATIVE_RAW=['relative_'+f for f in RAW]
uci_prior_models={}
uci_prior_features={}

def v9_uci_prior_parameters(target):
    # Select from the completed UCI-only nested tuning table. This local helper
    # avoids depending on selected_parameters(), which belongs to the earlier
    # optional Aurora external-validation section.
    default={'learning_rate':.03,'max_leaf_nodes':15,'l2_regularization':1.0}
    if 'delta_tuning' not in globals() or not isinstance(delta_tuning,pd.DataFrame):
        return default
    candidates=delta_tuning[
        (delta_tuning.target=='delta_'+target)&
        (delta_tuning.model=='pc_delta_relative')
    ].copy()
    if candidates.empty:return default
    summary=(candidates.groupby(
        ['learning_rate','max_leaf_nodes','l2_regularization'],as_index=False
    ).inner_mae.mean().sort_values('inner_mae'))
    best=summary.iloc[0]
    return {'learning_rate':float(best.learning_rate),
            'max_leaf_nodes':int(best.max_leaf_nodes),
            'l2_regularization':float(best.l2_regularization)}

for target in ['sbp','dbp']:
    features=DELTA_PC+RELATIVE_PC+[
        'elapsed_seconds','optical_reliability','optical_noise_ratio','baseline_'+target
    ]
    features=[f for f in features if f in delta_table.columns]
    params=v9_uci_prior_parameters(target)
    model=make_model(params)
    model.fit(delta_table[features],delta_table['delta_'+target],
              histgradientboostingregressor__sample_weight=weights(delta_table))
    uci_prior_models[target]=model;uci_prior_features[target]=features
    aurora_v9['uci_prior_delta_'+target]=model.predict(aurora_v9[features])


In [ ]:
ENGINEERED_SKIN=['fst_z','dark_skin_indicator','fst_time_interaction',
                 'fst_noise_interaction','fst_reliability_interaction']

def v9_participant_skin_weights(frame):
    visits=frame.groupby('subject_id').subject_id.transform('size').astype(float)
    people=frame[['subject_id','fst_band']].drop_duplicates()
    group_counts=people.fst_band.value_counts()
    group_factor=frame.fst_band.map((len(people)/(len(group_counts)*group_counts)).to_dict()).astype(float)
    w=(1/visits)*group_factor
    w=np.clip(w,np.quantile(w,.02),np.quantile(w,.98))
    return np.asarray(w/w.mean(),float)

def normalized_conditional_mi(frame,features,target,conditioning):
    conditioning=[c for c in conditioning if c in frame.columns]
    C=SimpleImputer(strategy='median').fit_transform(frame[conditioning])
    C=RobustScaler().fit_transform(C)
    y=np.asarray(frame[target],float)
    y_res=y-Ridge(alpha=1).fit(C,y).predict(C)
    X=SimpleImputer(strategy='median').fit_transform(frame[features])
    X_res=np.empty_like(X,float)
    for j in range(X.shape[1]):
        X_res[:,j]=X[:,j]-Ridge(alpha=1).fit(C,X[:,j]).predict(C)
    values=mutual_info_regression(X_res,y_res,random_state=SEED)
    return values/max(float(np.max(values)),1e-12)

def v9_information_rank(frame,features,target):
    features=[f for f in dict.fromkeys(features) if f in frame and frame[f].notna().any()]
    pressure='sbp' if target.endswith('sbp') else 'dbp'
    conditioning=['baseline_'+pressure,'log_elapsed_days']
    global_mi=normalized_conditional_mi(frame,features,target,conditioning)
    group_mi=[]
    for _,group in frame.groupby('fst_band'):
        if len(group)>=30 and group[target].nunique()>3:
            group_mi.append(normalized_conditional_mi(group,features,target,conditioning))
    worst=np.min(np.vstack(group_mi),axis=0) if group_mi else global_mi
    ranking=pd.DataFrame({'feature':features,'global_mi':global_mi,'worst_group_mi':worst})
    ranking['score']=.5*ranking.global_mi+.5*ranking.worst_group_mi
    return ranking.sort_values('score',ascending=False).reset_index(drop=True)

def v9_participant_metrics(frame,target,prediction):
    evaluated=frame.assign(_ae=np.abs(frame[prediction]-frame[target]))
    per_person=(evaluated.groupby(['subject_id','fst_band'],as_index=False)['_ae'].mean())
    group_mae=per_person.groupby('fst_band')['_ae'].mean()
    return float(per_person['_ae'].mean()),float(group_mae.max()),group_mae

def v9_group_policy(frame,target,base,delta_prediction):
    policy={};rows=[]
    working=frame.assign(_delta_prediction=np.asarray(delta_prediction,float))
    for group,g in working.groupby('fst_band'):
        baseline=float(g.assign(_ae=np.abs(g[base]-g[target])).groupby('subject_id')['_ae'].mean().mean())
        candidates=[]
        for alpha in V9_ALPHA_GRID:
            pred=g[base]+alpha*g._delta_prediction
            mae=float(g.assign(_ae=np.abs(pred-g[target])).groupby('subject_id')['_ae'].mean().mean())
            candidates.append((float(alpha),mae))
        feasible=[x for x in candidates if x[1]<=baseline+1e-12]
        best=min(feasible,key=lambda x:(x[1],x[0]))
        policy[group]=best[0]
        rows.append({'fst_band':group,'alpha':best[0],'baseline_mae':baseline,
                     'selected_mae':best[1],'degradation':best[1]-baseline})
    return policy,pd.DataFrame(rows)

def v9_apply_policy(frame,base,delta_prediction,policy):
    alpha=frame.fst_band.map(policy).fillna(0).to_numpy(float)
    return frame[base].to_numpy(float)+alpha*np.asarray(delta_prediction,float)


In [ ]:
V10_RESULTS=OUT_DIR/'aurora_change_gated_v10'
V10_RESULTS.mkdir(parents=True,exist_ok=True)
V10_OUTER_FOLDS=5
V10_TOP_K=[12,24]
V10_ALPHA_GRID=np.round(np.linspace(0,1,11),2)
V10_THRESHOLD_GRID=np.array([0,1,2,3,4,5,7.5,10.0],float)
V10_PARAMETER_GRID=[
    {'learning_rate':.03,'max_leaf_nodes':15,'l2_regularization':1.0},
    {'learning_rate':.03,'max_leaf_nodes':31,'l2_regularization':2.0},
    {'learning_rate':.05,'max_leaf_nodes':15,'l2_regularization':2.0},
]

def v10_group_gate_policy(frame,target,base,delta_prediction):
    working=frame.assign(_delta_prediction=np.asarray(delta_prediction,float))
    policy={};rows=[]
    for group,g in working.groupby('fst_band'):
        baseline_per_person=(g.assign(_ae=np.abs(g[base]-g[target]))
                             .groupby('subject_id')['_ae'].mean())
        baseline_mae=float(baseline_per_person.mean())
        candidates=[]
        for threshold in V10_THRESHOLD_GRID:
            active=np.abs(g['_delta_prediction'].to_numpy(float))>=threshold
            for alpha in V10_ALPHA_GRID:
                pred=g[base].to_numpy(float)+np.where(
                    active,alpha*g['_delta_prediction'].to_numpy(float),0
                )
                per_person=(g.assign(_ae=np.abs(pred-g[target]))
                            .groupby('subject_id')['_ae'].mean())
                effective_activation=float(active.mean()) if alpha>0 else 0.0
                candidates.append({'threshold':float(threshold),'alpha':float(alpha),
                                   'mae':float(per_person.mean()),
                                   'activation_rate':effective_activation})
        feasible=[x for x in candidates if x['mae']<=baseline_mae+1e-12]
        best=min(feasible,key=lambda x:(x['mae'],x['activation_rate'],x['threshold']))
        policy[group]={'threshold':best['threshold'],'alpha':best['alpha']}
        rows.append({'fst_band':group,'baseline_mae':baseline_mae,**best,
                     'improvement':baseline_mae-best['mae']})
    return policy,pd.DataFrame(rows)

def v10_apply_gate(frame,base,delta_prediction,policy,return_active=False):
    delta_prediction=np.asarray(delta_prediction,float)
    thresholds=frame.fst_band.map({g:p['threshold'] for g,p in policy.items()}).fillna(np.inf).to_numpy(float)
    alphas=frame.fst_band.map({g:p['alpha'] for g,p in policy.items()}).fillna(0).to_numpy(float)
    active=(np.abs(delta_prediction)>=thresholds)&(alphas>0)
    prediction=frame[base].to_numpy(float)+np.where(active,alphas*delta_prediction,0)
    return (prediction,active) if return_active else prediction


## V19 prerequisites

In [ ]:
# ============================================================================
# V19 prerequisites -- helper definitions lifted out of the superseded cells
# ----------------------------------------------------------------------------
# These were previously defined inside the V12/V17/V18 analysis cells. Only the
# definitions are needed, not the experiments that used them, so they are
# collected here and the experiments are gone.
# ============================================================================
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, train_test_split

V12_PARAMS = {'learning_rate': .03, 'max_leaf_nodes': 15, 'l2_regularization': 1.0}
V12_BOOT = 4000
V12_TOP_K = 24
DARK_BAND = 'Fitzpatrick V-VI'


def make_model_v17(params, loss='squared_error'):
    """make_model, but with the loss exposed. The default is squared_error."""
    return make_pipeline(SimpleImputer(strategy='median'),
                         HistGradientBoostingRegressor(max_iter=300, loss=loss,
                                                       random_state=SEED, **params))


def abl_weights(frame, band_col, use_skin):
    visits = frame.groupby('subject_id').subject_id.transform('size').astype(float)
    w = 1.0 / visits
    if use_skin:
        people = frame[['subject_id', band_col]].drop_duplicates()
        counts = people[band_col].value_counts()
        w = w * frame[band_col].map(
            (len(people) / (len(counts) * counts)).to_dict()).astype(float)
    w = np.clip(w, np.quantile(w, .02), np.quantile(w, .98))
    return np.asarray(w / w.mean(), float)


def abl_rank(frame, features, target, band_col, use_group):
    features = [f for f in dict.fromkeys(features)
                if f in frame and frame[f].notna().any()]
    pressure = 'sbp' if target.endswith('sbp') else 'dbp'
    conditioning = ['baseline_' + pressure, 'log_elapsed_days']
    global_mi = normalized_conditional_mi(frame, features, target, conditioning)
    if not use_group:
        return (pd.DataFrame({'feature': features, 'score': global_mi})
                  .sort_values('score', ascending=False).reset_index(drop=True))
    group_mi = []
    for _, g in frame.groupby(band_col):
        if len(g) >= 30 and g[target].nunique() > 3:
            group_mi.append(normalized_conditional_mi(g, features, target, conditioning))
    worst = np.min(np.vstack(group_mi), axis=0) if group_mi else global_mi
    return (pd.DataFrame({'feature': features, 'score': .5 * global_mi + .5 * worst})
              .sort_values('score', ascending=False).reset_index(drop=True))


def abl_gate_search(g, target, base, dp):
    baseline_mae = float(g.assign(_ae=np.abs(g[base] - g[target]))
                          .groupby('subject_id')['_ae'].mean().mean())
    best = None
    for threshold in V10_THRESHOLD_GRID:
        active = np.abs(dp) >= threshold
        for alpha in V10_ALPHA_GRID:
            pred = g[base].to_numpy(float) + np.where(active, alpha * dp, 0.0)
            mae = float(g.assign(_ae=np.abs(pred - g[target]))
                         .groupby('subject_id')['_ae'].mean().mean())
            if mae <= baseline_mae + 1e-12:
                if best is None or (mae, float(threshold)) < (best['mae'], best['threshold']):
                    best = {'threshold': float(threshold), 'alpha': float(alpha), 'mae': mae}
    return best or {'threshold': float('inf'), 'alpha': 0.0, 'mae': baseline_mae}


def abl_fit_gate(frame, target, base, dp, band_col, per_group):
    working = frame.assign(_dp=np.asarray(dp, float))
    if not per_group:
        return {'_global': abl_gate_search(working, target, base,
                                           working._dp.to_numpy(float))}
    return {group: abl_gate_search(g, target, base, g._dp.to_numpy(float))
            for group, g in working.groupby(band_col)}


def abl_apply_gate(frame, base, dp, gate, band_col):
    dp = np.asarray(dp, float)
    if '_global' in gate:
        threshold = np.full(len(frame), gate['_global']['threshold'])
        alpha = np.full(len(frame), gate['_global']['alpha'])
    else:
        threshold = frame[band_col].map(
            {k: v['threshold'] for k, v in gate.items()}).fillna(np.inf).to_numpy(float)
        alpha = frame[band_col].map(
            {k: v['alpha'] for k, v in gate.items()}).fillna(0.0).to_numpy(float)
    active = np.abs(dp) >= threshold
    return frame[base].to_numpy(float) + np.where(active, alpha * dp, 0.0), active


def avct_features_hybrid(ppg, ecg, fs, prefix, noise_ratio=0.0, radius_mode='fixed',
                         radius_gain=1.0):
    """H1 configuration selected in V18-1: scale from the BANDPASSED signal
    (not DC), legacy fixed radius. Best of the four variants tested --
    median |SMD| 0.500, morphology AUC 0.541, all-features AUC 0.783."""
    ppg = fill_gaps(ppg)
    if ppg is None or np.std(ppg) < 1e-8:
        return None
    if ecg is not None:
        ecg = fill_gaps(ecg)
        if ecg is None or np.std(ecg) < 1e-8:
            ecg = None
    f = bandpass(ppg, fs, .5, 8)
    scale = max(float(np.percentile(f, 95) - np.percentile(f, 5)), 1e-12)
    ac = f / scale
    E = embed(ac)
    rho = float(np.clip(noise_ratio, 0, 1.5))
    radius_factor = np.sqrt(1 + rho ** 2)
    r = (0.20 * radius_factor if radius_mode == 'fixed'
         else 0.20 * radius_factor * max(float(np.std(ac)), 1e-12) * radius_gain)
    rr, det, h = rqa(ac, eps=r)
    quality, spec_h = spectral(f, fs)
    csi = (0.4 * det + 0.3 * (1 - np.clip(h / 5, 0, 1)) + 0.3 * quality
           if np.all(np.isfinite([det, h, quality])) else np.nan)
    return {prefix + 'sigma_m': float(np.std(E)),
            prefix + 'skew_m': float(skew(E.ravel())),
            prefix + 'kurtosis_m': float(kurtosis(E.ravel())),
            prefix + 'sample_entropy': sample_entropy(ac, r=r),
            prefix + 'permutation_entropy': permutation_entropy(ac),
            prefix + 'recurrence_rate': rr, prefix + 'determinism': det,
            prefix + 'rqa_entropy': h, prefix + 'csi': csi,
            prefix + 'spectral_entropy': spec_h, prefix + 'ppg_quality': quality,
            prefix + 'pat_seconds': pat_feature(ecg, ppg, fs) if ecg is not None else np.nan}


# waveform metadata used by the V19-2 extraction pass
_v18_meta = measurements.merge(
    participants[['pid', 'fitzpatrick_scale']].drop_duplicates('pid'),
    on='pid', how='left', validate='many_to_one').dropna(
    subset=['fitzpatrick_scale', 'waveform_file_path'])

# the SAME holdout participants reserved in V17-2, reproduced from the same seed
_people = aurora_v9.groupby('subject_id', as_index=False).agg(fst_band=('fst_band', 'first'))
dev_people, hold_people = train_test_split(
    _people, test_size=0.25, random_state=SEED + 11100, stratify=_people.fst_band)
dev = aurora_v9[aurora_v9.subject_id.isin(dev_people.subject_id)].copy()
hold = aurora_v9[aurora_v9.subject_id.isin(hold_people.subject_id)].copy()
print(f'development {dev.subject_id.nunique()} participants | '
      f'holdout {hold.subject_id.nunique()} participants')
print(f'waveform metadata rows: {len(_v18_meta)}')


## V19-1: cuff noise floor and recalibration cadence

Cheap. Run this first.

In [ ]:
# ============================================================================
# V19-1: How much of the remaining error is even predictable?
# ----------------------------------------------------------------------------
# Two diagnostics, both cheap, both using data already loaded.
#
# PART A -- THE CUFF NOISE FLOOR.
# The model predicts delta_sbp, whose within-subject SD is 11-12 mmHg. But that
# is the spread of the OBSERVED change: real physiological drift PLUS cuff
# measurement error on both the anchor and the target reading. AuroraBP records
# TWO cuff readings per measurement (primary_* and secondary_*), so the cuff's
# own reproducibility is directly measurable:
#
#     sd(primary - secondary) = sqrt(2) * sigma_cuff
#
# Whatever share of the target variance is cuff noise is unpredictable from PPG
# by any model. It sets a floor on MAE that no amount of modelling can cross.
#
# PART B -- RECALIBRATION CADENCE.
# Proposition 1 in your supplementary decomposes uncalibrated error as
# |delta| + E|eps_within| + covariance, and reports 91.5% error reduction from
# single-point calibration -- i.e. label shift dominates. But the current
# baseline is anchored ONCE at session start, so delta regrows as elapsed time
# increases. If it does, recalibrating more often is a deployment change that
# attacks the dominant term directly, with no modelling required.
#
# This is measured without a model: for each measurement, re-anchor the
# prediction to the most recent PRIOR measurement within a cadence window and
# score plain calibration retention. That is exactly what a cuff recalibration
# would provide, and it uses only past information.
# ============================================================================
V19_RESULTS = OUT_DIR / 'avct_v19_headroom'
V19_RESULTS.mkdir(parents=True, exist_ok=True)

# --- PART A: cuff reproducibility ------------------------------------------
print('=== PART A: cuff measurement noise ===')
_c = measurements.copy()
for col in ['sbp', 'dbp', 'primary_systolic', 'secondary_systolic',
            'primary_diastolic', 'secondary_diastolic']:
    if col in _c.columns:
        _c[col] = pd.to_numeric(_c[col], errors='coerce')

pairs = [('sbp', 'primary_systolic', 'secondary_systolic'),
         ('dbp', 'primary_diastolic', 'secondary_diastolic')]
cuff_rows = []
for target, p, s in pairs:
    if not {p, s} <= set(_c.columns):
        print(f'{target}: primary/secondary columns absent -- skipped')
        continue
    d = _c[[target, p, s]].dropna()
    if len(d) < 100:
        print(f'{target}: too few paired readings ({len(d)})')
        continue
    diff = (d[p] - d[s]).to_numpy(float)
    sd_diff = float(np.std(diff, ddof=1))
    sigma_cuff = sd_diff / np.sqrt(2)
    # is the stored target one reading, or an average of the two?
    to_mean = float(np.mean(np.abs(d[target] - (d[p] + d[s]) / 2)))
    to_primary = float(np.mean(np.abs(d[target] - d[p])))
    kind = 'mean of both readings' if to_mean < to_primary else 'a single reading'
    sigma_target = sigma_cuff / np.sqrt(2) if to_mean < to_primary else sigma_cuff
    cuff_rows.append({'target': target, 'paired_readings': len(d),
                      'sd_primary_minus_secondary': sd_diff,
                      'sigma_cuff': sigma_cuff,
                      'target_appears_to_be': kind,
                      'sigma_on_stored_target': sigma_target,
                      # delta = target(t2) - anchor(t1): two independent readings
                      'sigma_on_delta': sigma_target * np.sqrt(2),
                      'mae_floor_mmhg': sigma_target * np.sqrt(2) * np.sqrt(2 / np.pi)})
v19_cuff = pd.DataFrame(cuff_rows)
display(v19_cuff.round(3))

for r in v19_cuff.itertuples():
    obs_sd = float(aurora_v9['delta_' + r.target].std())
    noise_var = r.sigma_on_delta ** 2
    phys_var = max(obs_sd ** 2 - noise_var, 0.0)
    current = {'sbp': 8.109, 'dbp': 6.149}[r.target]   # B0_tone_blind, full-cohort CV
    print(f'\n{r.target.upper()}')
    print(f'  observed sd of delta        : {obs_sd:.2f} mmHg')
    print(f'  cuff-noise share of that     : {100*noise_var/obs_sd**2:.1f}%')
    print(f'  physiological drift sd       : {np.sqrt(phys_var):.2f} mmHg')
    print(f'  MAE floor from cuff noise    : {r.mae_floor_mmhg:.2f} mmHg')
    print(f'  current model MAE            : {current:.2f} mmHg')
    print(f'  remaining headroom           : {current - r.mae_floor_mmhg:.2f} mmHg')
print("""
The floor assumes a model that predicts physiological drift PERFECTLY and is
still scored against a noisy cuff reading. It is a hard lower bound, not a
target. If the headroom is small, the honest conclusion is that the problem is
close to solved given this reference standard.
""")
v19_cuff.to_csv(V19_RESULTS / 'v19_cuff_noise.csv', index=False)

# --- PART B: does error grow with time since calibration? ------------------
print('\n=== PART B: recalibration cadence ===')
_t = aurora_v9.copy()
_t['elapsed_hours'] = _t.session_elapsed_seconds / 3600.0
_t['elapsed_bin'] = pd.cut(_t.elapsed_hours, [-.01, 1, 3, 6, 12, 24, 1e9],
                           labels=['<1h', '1-3h', '3-6h', '6-12h', '12-24h', '>24h'])
drift_rows = []
for target in ['sbp', 'dbp']:
    for b, g in _t.groupby('elapsed_bin', observed=True):
        if len(g) < 30:
            continue
        cal_mae = float(g.assign(_e=np.abs(g['session_baseline_' + target] - g[target]))
                         .groupby('subject_id')._e.mean().mean())
        drift_rows.append({'target': target, 'elapsed_bin': str(b),
                           'measurements': len(g), 'participants': g.subject_id.nunique(),
                           'mean_abs_delta': float(np.abs(g['session_delta_' + target]).mean()),
                           'calibration_retention_mae': cal_mae})
v19_drift = pd.DataFrame(drift_rows)
print('\nError as a function of time since the calibration anchor:')
display(v19_drift.pivot(index='elapsed_bin', columns='target',
                        values=['mean_abs_delta', 'calibration_retention_mae']).round(3))
print('If both columns climb steadily with elapsed time, label shift is')
print('accumulating and a shorter recalibration interval attacks it directly.')

# --- simulate recalibration cadences, model-free ---------------------------
CADENCES = [1, 3, 6, 12, 24, 1e9]        # hours; 1e9 = current session-start anchor
_s = aurora_v9.sort_values(['subject_id', 'session_elapsed_seconds']).copy()
_s['elapsed_hours'] = _s.session_elapsed_seconds / 3600.0
cad_rows = []
for target in ['sbp', 'dbp']:
    truth = _s[target].to_numpy(float)
    base_all = _s['session_baseline_' + target].to_numpy(float)
    times_all = _s.elapsed_hours.to_numpy(float)
    for cadence in CADENCES:
        anchors = np.full(len(_s), np.nan)
        for _, idx in _s.groupby('subject_id').indices.items():
            # score with the current anchor FIRST, then recalibrate -- so a
            # measurement is never used to predict itself
            anchor_value = base_all[idx[0]]
            anchor_time = 0.0
            for k in idx:
                anchors[k] = anchor_value
                if times_all[k] - anchor_time >= cadence:
                    anchor_value = truth[k]      # a fresh, and therefore NOISY, cuff reading
                    anchor_time = times_all[k]
        per = (pd.DataFrame({'subject_id': _s.subject_id.to_numpy(),
                             '_e': np.abs(anchors - truth)})
                 .groupby('subject_id')._e.mean())
        cad_rows.append({'target': target,
                         'recalibration_every_h': 'session start' if cadence > 1e8 else f'{cadence:g}h',
                         'calibration_retention_mae': float(per.mean())})
v19_cadence = pd.DataFrame(cad_rows).pivot(index='recalibration_every_h',
                                           columns='target',
                                           values='calibration_retention_mae')
print('\n=== Calibration-retention MAE under different recalibration cadences ===')
print('(no model at all -- just "predict the last cuff reading")')
display(v19_cadence.round(3))
print(f"""
Read the gap between the session-start row and the shorter cadences, and compare
it against the ~1.3 mmHg the current model buys over session-start calibration
retention. If recalibrating every few hours beats that, cadence is the bigger
lever and needs no PPG modelling at all.

THE TRADE-OFF THIS MEASURES, which is not obvious a priori:
recalibrating replaces an OLD anchor with a FRESH one, so it removes accumulated
drift but injects the new cuff reading's own noise into every prediction that
follows. From Part A, a fresh anchor carries sd = sigma_on_delta. Recalibration
therefore only pays when the drift accumulated over the interval exceeds that.
This is why shorter is not automatically better, and why the table may well be
non-monotonic or even rise as the cadence shortens.

Three regimes are possible, and all three are informative:
  MAE falls as cadence shortens -> drift dominates. Recalibration is the lever,
      and the table says how often to ask the user for a cuff reading.
  MAE roughly flat             -> anchor noise and drift are comparable at this
      timescale. Recalibration buys nothing.
  MAE RISES as cadence shortens -> the anchor is being made WORSE. This happens
      when the session-start baseline averages several cuff readings while a
      recalibration is a single one, so the swap trades sigma/sqrt(n) for sigma.
      It would mean the reference standard, not the physiology, is binding --
      and that AVERAGING several readings at each recalibration is the fix.

Each recalibration also costs the user a cuff measurement, so the practical
comparison is accuracy against burden, not accuracy alone.
""")
v19_drift.to_csv(V19_RESULTS / 'v19_drift_by_time.csv', index=False)
v19_cadence.to_csv(V19_RESULTS / 'v19_cadence.csv')


## V19-2: multi-window averaging

~17 minutes of extraction, cached. Only worth running if V19-1 shows headroom.

In [ ]:
# ============================================================================
# V19-2: MULTI-WINDOW AVERAGING -- reduce feature noise by averaging windows
# ----------------------------------------------------------------------------
# Extraction currently takes ONE 10-second window from the centre of each
# recording, while `duration` metadata shows roughly 20 seconds available. The
# rest is discarded.
#
# Averaging a feature over K windows reduces its measurement-noise variance by
# up to K. This is the same averaging-gain argument as Corollary 1 Step 2
# (sigma^2 = O(1/N_e)), applied to the feature ESTIMATE rather than to the
# embedding points -- so it is motivated by your own theory, not bolted on.
#
# HONEST EXPECTATION: the windows overlap (three 10 s windows at 5 s stride
# inside ~20 s), so successive windows share half their samples and the noise
# reduction will be well short of sqrt(3). Physiological variation within 20
# seconds is also real signal, not noise, and averaging removes some of it.
# Expect tenths of a mmHg at best.
#
# Uses the H1 configuration (new scale, legacy fixed radius), which V18-1
# selected: median |SMD| 0.500 and all-features AUC 0.783, the best of the four
# variants tested.
#
# RUNTIME: one extraction pass over every waveform (~17 min), cached.
# ============================================================================
V19_MW_CACHE = V19_RESULTS / 'v19_multiwindow_features.csv'
V19_STRIDE_SECONDS = 5.0
V19_MAX_WINDOWS = 3

if V19_MW_CACHE.exists():
    v19_mw = pd.read_csv(V19_MW_CACHE)
    print(f'loaded cached multi-window features: {len(v19_mw)} rows')
else:
    rows, failures, window_counts = [], 0, []
    for _, row in tqdm(_v18_meta.iterrows(), total=len(_v18_meta), desc='multi-window'):
        try:
            w = pd.read_csv(resolve_aurora_waveform(row.waveform_file_path), sep='\t',
                            usecols=lambda c: c in {'t', 'optical'}, low_memory=False)
            t = pd.to_numeric(w.t, errors='coerce').to_numpy(float)
            x = pd.to_numeric(w.optical, errors='coerce').to_numpy(float)
            m = np.isfinite(t) & np.isfinite(x)
            t, x = t[m], x[m]
            o = np.argsort(t); t, x = t[o], x[o]
            if len(x) < 1000 or t[-1] <= t[0]:
                raise ValueError('short')
            fs = (len(t) - 1) / (t[-1] - t[0])
            need = int(round(WINDOW_SECONDS * fs))
            stride = int(round(V19_STRIDE_SECONDS * fs))
            if len(x) < need:
                raise ValueError('short')
            starts = list(range(0, len(x) - need + 1, max(stride, 1)))[:V19_MAX_WINDOWS]
            if not starts:
                starts = [(len(x) - need) // 2]
            q = pd.to_numeric(pd.Series([row.get('optical_quality')]), errors='coerce').iloc[0]
            nr = float(np.clip(1 - q, 0, 1.5)) if np.isfinite(q) else 0.0
            per_window = []
            for st in starts:
                f = avct_features_hybrid(x[st:st + need], None, fs, '', nr, 'fixed', 1.0)
                if f is not None:
                    per_window.append(f)
            if not per_window:
                raise ValueError('no usable window')
            window_counts.append(len(per_window))
            wdf = pd.DataFrame(per_window)
            entry = {'pid': str(row.pid), 'phase': str(row.phase),
                     'measurement': str(row.measurement),
                     'fitzpatrick_scale': float(row.fitzpatrick_scale),
                     'n_windows': len(per_window)}
            for c in wdf.columns:
                entry['mw_' + c] = float(wdf[c].mean())         # averaged
                entry['sw_' + c] = float(wdf[c].iloc[len(wdf) // 2])   # single centre
                entry['wsd_' + c] = float(wdf[c].std()) if len(wdf) > 1 else np.nan
            rows.append(entry)
        except Exception:
            failures += 1
    v19_mw = pd.DataFrame(rows)
    v19_mw.to_csv(V19_MW_CACHE, index=False)
    print(f'extracted {len(rows)} rows ({failures} failures); '
          f'median windows per recording: {np.median(window_counts):.0f}')

print(f"\nwindows per recording: {v19_mw.n_windows.value_counts().to_dict()}")

# --- how much noise does averaging actually remove? ------------------------
STEM_LIST = ['sigma_m', 'skew_m', 'kurtosis_m', 'sample_entropy',
             'permutation_entropy', 'recurrence_rate', 'determinism',
             'rqa_entropy', 'csi', 'spectral_entropy']
noise_rows = []
for stem in STEM_LIST:
    wsd, sw = 'wsd_' + stem, 'sw_' + stem
    if wsd not in v19_mw.columns:
        continue
    within = pd.to_numeric(v19_mw[wsd], errors='coerce')
    across = pd.to_numeric(v19_mw[sw], errors='coerce')
    if within.notna().sum() < 100 or across.std() == 0:
        continue
    noise_rows.append({
        'feature': stem,
        'within_recording_sd': float(within.mean()),
        'between_recording_sd': float(across.std()),
        'noise_fraction': float(within.mean() / max(across.std(), 1e-12)),
        'expected_noise_cut': float(1 - 1 / np.sqrt(v19_mw.n_windows.median()))})
v19_noise = pd.DataFrame(noise_rows).sort_values('noise_fraction', ascending=False)
print('\n=== Feature measurement noise, within a recording vs between recordings ===')
display(v19_noise.round(4))
print('noise_fraction near or above 1 means the feature varies as much between')
print('two windows of the SAME recording as it does between different')
print('recordings -- those features are mostly measurement noise, and averaging')
print('should help them most. Near 0 means the feature is already stable and')
print('averaging will change little.')
v19_noise.to_csv(V19_RESULTS / 'v19_feature_noise.csv', index=False)

# --- does averaging lower MAE on the untouched holdout? --------------------
_keys = ['pid', 'phase', 'measurement']
if not set(_keys) <= set(aurora_v9.columns):
    print('\naurora_v9 lacks pid/phase/measurement -- cannot merge; skipping the MAE test.')
else:
    frame = aurora_v9.copy()
    for k in _keys:
        frame[k] = frame[k].astype(str)
        v19_mw[k] = v19_mw[k].astype(str)
    keep = _keys + [c for c in v19_mw.columns if c.startswith(('mw_', 'sw_'))]
    frame = frame.merge(v19_mw[keep], on=_keys, how='left')
    MW_DELTA, SW_DELTA = [], []
    for stem in STEM_LIST:
        for prefix, store in [('mw_', MW_DELTA), ('sw_', SW_DELTA)]:
            c = prefix + stem
            if c in frame.columns:
                ref = frame.groupby('subject_id')[c].transform('first')
                frame['delta_' + c] = frame[c] - ref
                store.append('delta_' + c)
    print(f'\nmerged: {len(MW_DELTA)} averaged and {len(SW_DELTA)} single-window '
          f'delta features')

    hold_ids = set(hold.subject_id)
    dev19 = frame[~frame.subject_id.isin(hold_ids)]
    hold19 = frame[frame.subject_id.isin(hold_ids)]
    print(f'development {dev19.subject_id.nunique()} | holdout {hold19.subject_id.nunique()} '
          '(same split reserved in V17-2)')

    mw_rows = []
    for target in ['sbp', 'dbp']:
        base_col = 'baseline_' + target
        cal = float(hold19.assign(_e=np.abs(hold19[base_col] - hold19[target]))
                           .groupby('subject_id')._e.mean().mean())
        for label, extra in [('legacy_features_only', []),
                             ('plus_single_window', SW_DELTA),
                             ('plus_multiwindow_avg', MW_DELTA)]:
            pool = [f for f in dict.fromkeys(
                DELTA_RAW + RELATIVE_RAW + ['elapsed_seconds', 'log_elapsed_days',
                                            'uci_prior_delta_' + target] + extra + [base_col])
                if f in dev19.columns]
            fit = dev19.dropna(subset=['delta_' + target])
            ranking = abl_rank(fit, pool, 'delta_' + target, 'fst_band', False)
            feats = list(dict.fromkeys([base_col, 'log_elapsed_days']
                                       + ranking.feature.head(24).tolist()))
            feats = [f for f in feats if f in fit.columns and f in hold19.columns]
            model = make_model_v17(V12_PARAMS, loss='squared_error')
            model.fit(fit[feats], fit['delta_' + target],
                      histgradientboostingregressor__sample_weight=
                      abl_weights(fit, 'fst_band', False))
            gate = abl_fit_gate(fit, target, base_col, model.predict(fit[feats]),
                                'fst_band', False)
            pred, _ = abl_apply_gate(hold19, base_col, model.predict(hold19[feats]),
                                     gate, 'fst_band')
            per = (hold19.assign(_e=np.abs(pred - hold19[target]))
                         .groupby('subject_id')._e.mean())
            mw_rows.append({'target': target, 'features': label,
                            'holdout_mae': float(per.mean()),
                            'calibration_mae': cal})
    v19_mw_result = pd.DataFrame(mw_rows).pivot(index='features', columns='target',
                                                values='holdout_mae')
    print('\n=== Holdout MAE: single window vs averaged windows ===')
    display(v19_mw_result.round(3))
    print('\nThe comparison that matters is plus_single_window vs')
    print('plus_multiwindow_avg: identical feature definitions, identical')
    print('pipeline, differing only in whether windows were averaged. Any gap')
    print('between them is the averaging effect and nothing else.')
    v19_mw_result.to_csv(V19_RESULTS / 'v19_multiwindow_mae.csv')


## V20: prior-anchored context stack on the untouched holdout

This final ablation uses the same participant holdout as V17-2. It reconstructs
H1 features at the calibration replicate mean, re-anchors every target and
feature delta to the most recent valid prior measurement, and reports
participant-level bootstrap confidence intervals. Feature ranking, gate fitting,
and all preprocessing are learned on development participants only.


In [ ]:
# ============================================================================
# V20: ABSOLUTE-ERROR + H1 + CONTEXT + MOST-RECENT-PRIOR ANCHOR
# ============================================================================
V20_RESULTS = OUT_DIR / 'avct_v20_prior_context_stack'
V20_RESULTS.mkdir(parents=True, exist_ok=True)
V20_BOOTSTRAP_REPEATS = 4000

# Use the H1 single-centre-window features extracted in V19-2. Reconstructing
# the ordered table first is essential: it lets the calibration-start replicate
# mean supply the H1 anchor for the first scored measurement.
_h1_source_columns = [c for c in v19_mw.columns if c.startswith('sw_')]
if not _h1_source_columns:
    raise RuntimeError('No H1 single-window features found in v19_mw.')
_h1_lookup = v19_mw[['pid', 'phase', 'measurement'] + _h1_source_columns].copy()
_h1_rename = {c: 'h1_' + c[len('sw_'):] for c in _h1_source_columns}
_h1_lookup = _h1_lookup.rename(columns=_h1_rename)
_h1_current = list(_h1_rename.values())

_aurora_h1 = aurora_features.copy()
for key in ['pid', 'phase', 'measurement']:
    _aurora_h1[key] = _aurora_h1[key].astype(str)
    _h1_lookup[key] = _h1_lookup[key].astype(str)
_aurora_h1 = _aurora_h1.merge(
    _h1_lookup, on=['pid', 'phase', 'measurement'], how='left', validate='one_to_one')

aurora_v20_ordered, aurora_v20_audit = build_replicate_calibrated_aurora(_aurora_h1)
aurora_v20 = build_delta_table(
    aurora_v20_ordered, RAW + PC + _h1_current, [], anchor='most_recent_prior'
).replace([np.inf, -np.inf], np.nan)
aurora_v20, V20_CONTEXT_FEATURES, v20_context_audit = add_aurora_measurement_context(aurora_v20)
aurora_v20['subject_id'] = aurora_v20.subject_id.astype(str)
aurora_v20['fst_band'] = pd.cut(
    aurora_v20.fitzpatrick_scale, [0, 2, 4, 6],
    labels=['Fitzpatrick I-II', 'Fitzpatrick III-IV', 'Fitzpatrick V-VI']).astype(str)
aurora_v20['log_elapsed_days'] = np.log1p(
    aurora_v20.elapsed_seconds.clip(lower=0) / 86400)
aurora_v20['log_session_elapsed_days'] = np.log1p(
    aurora_v20.session_elapsed_seconds.clip(lower=0) / 86400)

# The cross-dataset prior is trained on generic within-subject changes. Apply it
# to the newly prior-anchored feature changes without touching the holdout fit.
for target in ['sbp', 'dbp']:
    prior_features = [f for f in uci_prior_features[target] if f in aurora_v20.columns]
    if prior_features == uci_prior_features[target]:
        aurora_v20['uci_prior_delta_' + target] = uci_prior_models[target].predict(
            aurora_v20[prior_features])

_hold_ids = set(hold_people.subject_id.astype(str))
v20_dev = aurora_v20[~aurora_v20.subject_id.isin(_hold_ids)].copy()
v20_hold = aurora_v20[aurora_v20.subject_id.isin(_hold_ids)].copy()
assert not set(v20_dev.subject_id) & set(v20_hold.subject_id)
print(f'V20 development {v20_dev.subject_id.nunique()} participants | '
      f'holdout {v20_hold.subject_id.nunique()} participants')
print('Anchor mode:', aurora_v20.anchor_mode.value_counts().to_dict())
print('H1 features:', _h1_current)
print('Context features:', V20_CONTEXT_FEATURES)
display(v20_context_audit)


def v20_subject_bootstrap(values, repeats=V20_BOOTSTRAP_REPEATS, seed=SEED + 20000):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(repeats, len(values)))].mean(axis=1)
    return tuple(np.quantile(means, [0.025, 0.975]))


V20_H1_DELTA = ['delta_' + c for c in _h1_current]
v20_rows, v20_per_subject, v20_selected = [], {}, []
for target in ['sbp', 'dbp']:
    base = 'baseline_' + target
    legacy_pool = (DELTA_RAW + RELATIVE_RAW + [
        'elapsed_seconds', 'log_elapsed_days', 'session_elapsed_seconds',
        'log_session_elapsed_days', 'uci_prior_delta_' + target])
    configurations = [
        ('reanchored_legacy_squared_error', 'squared_error', legacy_pool, []),
        ('reanchored_absolute_error', 'absolute_error', legacy_pool, []),
        ('reanchored_absolute_error_plus_H1', 'absolute_error',
         legacy_pool + V20_H1_DELTA, V20_H1_DELTA),
        ('combined_absolute_error_H1_context', 'absolute_error',
         legacy_pool + V20_H1_DELTA + V20_CONTEXT_FEATURES,
         V20_H1_DELTA + V20_CONTEXT_FEATURES),
    ]

    # Prior-measurement retention is the no-model deployment reference.
    retention = (v20_hold.assign(_e=np.abs(v20_hold[base] - v20_hold[target]))
                 .groupby('subject_id')._e.mean())
    v20_per_subject[(target, 'prior_measurement_retention')] = retention
    lo, hi = v20_subject_bootstrap(retention, seed=SEED + 20001 + (target == 'dbp'))
    v20_rows.append({'target': target, 'configuration': 'prior_measurement_retention',
                     'holdout_mae': float(retention.mean()), 'mae_ci_low': lo,
                     'mae_ci_high': hi, 'feature_count': 0})

    for config_index, (label, loss, pool, forced) in enumerate(configurations):
        pool = [f for f in dict.fromkeys(pool)
                if f in v20_dev.columns and f in v20_hold.columns
                and v20_dev[f].notna().any() and v20_dev[f].nunique(dropna=True) > 1]
        forced = [f for f in dict.fromkeys(forced) if f in pool]
        fit = v20_dev.dropna(subset=['delta_' + target]).copy()
        ranking = abl_rank(fit, pool, 'delta_' + target, 'fst_band', False)
        # Forced terms make this a literal stack test: H1 and available context
        # cannot silently disappear merely because a top-K rank cut is applied.
        features = list(dict.fromkeys(
            [base, 'log_elapsed_days'] + forced
            + ranking.feature.head(V12_TOP_K).tolist()))
        features = [f for f in features if f in fit.columns and f in v20_hold.columns]
        model = make_model_v17(V12_PARAMS, loss=loss)
        model.fit(
            fit[features], fit['delta_' + target],
            histgradientboostingregressor__sample_weight=abl_weights(fit, 'fst_band', False))
        dev_delta = model.predict(fit[features])
        gate = abl_fit_gate(fit, target, base, dev_delta, 'fst_band', False)
        hold_delta = model.predict(v20_hold[features])
        prediction, active = abl_apply_gate(
            v20_hold, base, hold_delta, gate, 'fst_band')
        per_subject = (v20_hold.assign(_e=np.abs(prediction - v20_hold[target]))
                       .groupby('subject_id')._e.mean())
        v20_per_subject[(target, label)] = per_subject
        lo, hi = v20_subject_bootstrap(
            per_subject, seed=SEED + 20100 + 10 * (target == 'dbp') + config_index)
        v20_rows.append({'target': target, 'configuration': label,
                         'holdout_mae': float(per_subject.mean()),
                         'mae_ci_low': lo, 'mae_ci_high': hi,
                         'feature_count': len(features),
                         'gate_activation_rate': float(active.mean())})
        v20_selected.append({'target': target, 'configuration': label,
                             'features': '|'.join(features), 'gate': repr(gate)})

v20_results = pd.DataFrame(v20_rows)

# Paired subject bootstrap: positive values mean the requested stack improves
# on the reanchored legacy model for that same holdout participant.
comparison = 'reanchored_legacy_squared_error'
for row_index, row in v20_results.iterrows():
    target, label = row.target, row.configuration
    if label == comparison:
        v20_results.loc[row_index, ['improvement_vs_legacy',
                                    'improvement_ci_low', 'improvement_ci_high']] = [0.0, 0.0, 0.0]
        continue
    legacy = v20_per_subject[(target, comparison)]
    candidate = v20_per_subject[(target, label)]
    aligned = pd.concat([legacy.rename('legacy'), candidate.rename('candidate')], axis=1).dropna()
    improvement = aligned.legacy - aligned.candidate
    lo, hi = v20_subject_bootstrap(
        improvement, seed=SEED + 20200 + row_index)
    v20_results.loc[row_index, 'improvement_vs_legacy'] = float(improvement.mean())
    v20_results.loc[row_index, 'improvement_ci_low'] = lo
    v20_results.loc[row_index, 'improvement_ci_high'] = hi

print('\n=== V20 untouched-holdout MAE and participant-bootstrap 95% CI ===')
display(v20_results.round(3))
print('Positive improvement_vs_legacy favors the candidate. A paired CI that')
print('crosses zero is not an established improvement on this holdout.')

v20_results.to_csv(V20_RESULTS / 'v20_holdout_stack_ci.csv', index=False)
pd.DataFrame(v20_selected).to_csv(V20_RESULTS / 'v20_selected_features.csv', index=False)
aurora_v20[['subject_id', 'phase', 'measurement', 'anchor_phase',
            'anchor_measurement', 'elapsed_seconds', 'session_elapsed_seconds']
           ].to_csv(V20_RESULTS / 'v20_anchor_audit.csv', index=False)


## V21: controlled isolation ladder

This diagnostic holds one development-only legacy feature set fixed across all
rows. The sequence is:

1. session anchor + squared-error legacy model;
2. most-recent-prior anchor + the same model and feature names;
3. switch only the loss to `absolute_error`;
4. append available context, without H1;
5. append H1 to obtain the full stack.

Participant-level paired bootstrap intervals are reported for every adjacent
transition and for the total session-anchor-to-full-stack change. Because V21
holds a common feature set fixed to isolate components, its final point estimate
may differ slightly from V20's current best of 5.880 mmHg SBP / 4.461 mmHg DBP.


In [ ]:
# ============================================================================
# V21: SAME-SPLIT CONTROLLED ISOLATION LADDER
# ============================================================================
V21_RESULTS = OUT_DIR / 'avct_v21_isolation_ladder'
V21_RESULTS.mkdir(parents=True, exist_ok=True)
V21_BOOTSTRAP_REPEATS = 4000


def v21_prepare_anchor(anchor):
    frame = build_delta_table(
        aurora_v20_ordered, RAW + PC + _h1_current, [], anchor=anchor
    ).replace([np.inf, -np.inf], np.nan)
    frame, context_features, context_audit = add_aurora_measurement_context(frame)
    frame['subject_id'] = frame.subject_id.astype(str)
    frame['fst_band'] = pd.cut(
        frame.fitzpatrick_scale, [0, 2, 4, 6],
        labels=['Fitzpatrick I-II', 'Fitzpatrick III-IV',
                'Fitzpatrick V-VI']).astype(str)
    frame['log_elapsed_days'] = np.log1p(
        frame.elapsed_seconds.clip(lower=0) / 86400)
    frame['log_session_elapsed_days'] = np.log1p(
        frame.session_elapsed_seconds.clip(lower=0) / 86400)
    for target in ['sbp', 'dbp']:
        features = uci_prior_features[target]
        if all(f in frame.columns for f in features):
            frame['uci_prior_delta_' + target] = uci_prior_models[target].predict(
                frame[features])
    return frame, context_features, context_audit


v21_session, V21_SESSION_CONTEXT, _ = v21_prepare_anchor('session_start')
v21_prior, V21_PRIOR_CONTEXT, v21_context_audit = v21_prepare_anchor(
    'most_recent_prior')
V21_CONTEXT_FEATURES = [
    f for f in V21_PRIOR_CONTEXT if f in set(V21_SESSION_CONTEXT)]
V21_H1_DELTA = ['delta_' + c for c in _h1_current]

_v21_hold_ids = set(hold_people.subject_id.astype(str))
v21_frames = {}
for anchor, frame in [('session_start', v21_session),
                      ('most_recent_prior', v21_prior)]:
    development = frame[~frame.subject_id.isin(_v21_hold_ids)].copy()
    evaluation = frame[frame.subject_id.isin(_v21_hold_ids)].copy()
    assert not set(development.subject_id) & set(evaluation.subject_id)
    v21_frames[anchor] = {'dev': development, 'hold': evaluation}

_session_keys = set(zip(v21_frames['session_start']['hold'].subject_id,
                        v21_frames['session_start']['hold'].original_index))
_prior_keys = set(zip(v21_frames['most_recent_prior']['hold'].subject_id,
                      v21_frames['most_recent_prior']['hold'].original_index))
assert _session_keys == _prior_keys, 'Anchor tables do not contain identical holdout rows.'
print(f"V21 development {v21_frames['most_recent_prior']['dev'].subject_id.nunique()} "
      f"participants | holdout "
      f"{v21_frames['most_recent_prior']['hold'].subject_id.nunique()} participants")
print('Rows aligned across anchors:', len(_session_keys))
print('Context features:', V21_CONTEXT_FEATURES)


def v21_bootstrap(values, seed, repeats=V21_BOOTSTRAP_REPEATS):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = values[
        rng.integers(0, len(values), size=(repeats, len(values)))
    ].mean(axis=1)
    return tuple(np.quantile(means, [0.025, 0.975]))


def v21_fit_and_score(dev_frame, hold_frame, target, features, loss):
    features = [
        f for f in dict.fromkeys(features)
        if f in dev_frame.columns and f in hold_frame.columns
        and dev_frame[f].notna().any()
        and dev_frame[f].nunique(dropna=True) > 1
    ]
    fit = dev_frame.dropna(subset=['delta_' + target]).copy()
    model = make_model_v17(V12_PARAMS, loss=loss)
    model.fit(
        fit[features], fit['delta_' + target],
        histgradientboostingregressor__sample_weight=abl_weights(
            fit, 'fst_band', False))
    development_delta = model.predict(fit[features])
    gate = abl_fit_gate(
        fit, target, 'baseline_' + target, development_delta,
        'fst_band', False)
    hold_delta = model.predict(hold_frame[features])
    prediction, active = abl_apply_gate(
        hold_frame, 'baseline_' + target, hold_delta, gate, 'fst_band')
    per_subject = (hold_frame.assign(
        _e=np.abs(prediction - hold_frame[target]))
        .groupby('subject_id')._e.mean())
    return per_subject, features, gate, float(active.mean())


v21_rows, v21_per_subject, v21_feature_rows = [], {}, []
for target_index, target in enumerate(['sbp', 'dbp']):
    base = 'baseline_' + target
    candidate_pool = DELTA_RAW + RELATIVE_RAW + [
        'elapsed_seconds', 'log_elapsed_days',
        'session_elapsed_seconds', 'log_session_elapsed_days',
        'uci_prior_delta_' + target]
    session_dev = v21_frames['session_start']['dev']
    prior_dev = v21_frames['most_recent_prior']['dev']
    shared_pool = [
        f for f in dict.fromkeys(candidate_pool)
        if f in session_dev.columns and f in prior_dev.columns
        and session_dev[f].notna().any() and prior_dev[f].notna().any()
        and session_dev[f].nunique(dropna=True) > 1
        and prior_dev[f].nunique(dropna=True) > 1
    ]

    # Symmetric development-only ranking: neither anchor gets to select the
    # legacy feature names alone, and the holdout is never consulted.
    session_rank = abl_rank(
        session_dev, shared_pool, 'delta_' + target,
        'fst_band', False)[['feature', 'score']].rename(
            columns={'score': 'session_score'})
    prior_rank = abl_rank(
        prior_dev, shared_pool, 'delta_' + target,
        'fst_band', False)[['feature', 'score']].rename(
            columns={'score': 'prior_score'})
    shared_rank = session_rank.merge(prior_rank, on='feature', how='outer').fillna(0)
    shared_rank['score'] = (
        shared_rank.session_score + shared_rank.prior_score) / 2
    core = list(dict.fromkeys(
        [base] + shared_rank.sort_values(
            ['score', 'feature'], ascending=[False, True]
        ).feature.head(V12_TOP_K).tolist()))

    ladder = [
        ('session_anchor_legacy', 'session_start', 'squared_error', core),
        ('prior_anchor_legacy', 'most_recent_prior', 'squared_error', core),
        ('prior_anchor_absolute_error', 'most_recent_prior',
         'absolute_error', core),
        ('prior_absolute_error_plus_context', 'most_recent_prior',
         'absolute_error', core + V21_CONTEXT_FEATURES),
        ('full_absolute_error_context_H1', 'most_recent_prior',
         'absolute_error', core + V21_CONTEXT_FEATURES + V21_H1_DELTA),
    ]

    for step_index, (label, anchor, loss, features) in enumerate(ladder):
        per_subject, used, gate, activation = v21_fit_and_score(
            v21_frames[anchor]['dev'], v21_frames[anchor]['hold'],
            target, features, loss)
        v21_per_subject[(target, label)] = per_subject
        lo, hi = v21_bootstrap(
            per_subject, SEED + 21000 + 100 * target_index + step_index)
        v21_rows.append({
            'target': target, 'step': step_index + 1,
            'configuration': label, 'anchor': anchor, 'loss': loss,
            'holdout_mae': float(per_subject.mean()),
            'mae_ci_low': lo, 'mae_ci_high': hi,
            'feature_count': len(used), 'gate_activation_rate': activation,
        })
        v21_feature_rows.append({
            'target': target, 'configuration': label,
            'features': '|'.join(used), 'gate': repr(gate),
        })

v21_results = pd.DataFrame(v21_rows)

# Every adjacent row changes exactly one requested component.
v21_comparisons = []
for target_index, target in enumerate(['sbp', 'dbp']):
    ordered = (v21_results[v21_results.target == target]
               .sort_values('step').configuration.tolist())
    pairs = list(zip(ordered[:-1], ordered[1:]))
    pairs.append((ordered[0], ordered[-1]))
    for pair_index, (before, after) in enumerate(pairs):
        aligned = pd.concat([
            v21_per_subject[(target, before)].rename('before'),
            v21_per_subject[(target, after)].rename('after')
        ], axis=1).dropna()
        improvement = aligned.before - aligned.after
        lo, hi = v21_bootstrap(
            improvement,
            SEED + 21500 + 100 * target_index + pair_index)
        v21_comparisons.append({
            'target': target, 'from': before, 'to': after,
            'improvement_mmhg': float(improvement.mean()),
            'improvement_percent': float(
                100 * improvement.mean() / aligned.before.mean()),
            'ci_low': lo, 'ci_high': hi,
            'supported_95pct': bool(lo > 0),
        })
v21_comparisons = pd.DataFrame(v21_comparisons)

print('\n=== V21 controlled holdout ladder ===')
display(v21_results.round(3))
print('\n=== Paired participant-bootstrap changes (positive favors later row) ===')
display(v21_comparisons.round(3))

v21_results.to_csv(V21_RESULTS / 'v21_holdout_ladder.csv', index=False)
v21_comparisons.to_csv(
    V21_RESULTS / 'v21_adjacent_paired_ci.csv', index=False)
pd.DataFrame(v21_feature_rows).to_csv(
    V21_RESULTS / 'v21_fixed_features_and_gates.csv', index=False)


## V22: updated skin-tone model versus tone-blind model

This comparison starts from V21's recommended prior-anchored
`absolute_error + context` model (without H1). The tone-blind and skin-aware
models share the same participants, targets, holdout, base features, estimator,
and hyperparameters.

The updated skin-aware model adds continuous Fitzpatrick terms and interactions,
balances development participants across Fitzpatrick bands, and learns
group-specific shrinkage gates on development data only. An ablation separates
skin features, skin-balanced weighting, and group-specific gating. Results are
reported overall and within each Fitzpatrick band with paired participant-level
bootstrap confidence intervals. Positive error differences favor the skin-aware
model.


In [ ]:
# ============================================================================
# V22: UPDATED SKIN-TONE MODEL VS TONE-BLIND MODEL
# ============================================================================
V22_RESULTS = OUT_DIR / 'avct_v22_skintone_comparison'
V22_RESULTS.mkdir(parents=True, exist_ok=True)
V22_BOOTSTRAP_REPEATS = 4000


def v22_add_skin_features(frame):
    """Skin terms available at prediction time; no outcome-derived features."""
    out = frame.copy()
    fst = pd.to_numeric(out.fitzpatrick_scale, errors='coerce')
    out['skin_fitzpatrick_z'] = (fst - 1) / 5
    out['skin_dark_indicator'] = (fst >= 5).astype(float)
    created = ['skin_fitzpatrick_z', 'skin_dark_indicator']
    interactions = {
        'log_session_elapsed_days': 'skin_x_session_time',
        'log_elapsed_days': 'skin_x_prior_gap',
        'optical_noise_ratio': 'skin_x_optical_noise',
        'optical_reliability': 'skin_x_optical_reliability',
        'context_phase_return': 'skin_x_return_phase',
        'context_posture_seated': 'skin_x_seated',
        'context_activity_post_exercise': 'skin_x_post_exercise',
        'context_optical_quality': 'skin_x_optical_quality',
    }
    for existing, new in interactions.items():
        if existing in out.columns:
            values = pd.to_numeric(out[existing], errors='coerce')
            if values.notna().any():
                out[new] = out.skin_fitzpatrick_z * values
                created.append(new)
    return out, created


v22_dev, V22_SKIN_FEATURES = v22_add_skin_features(
    v21_frames['most_recent_prior']['dev'])
v22_hold, _ = v22_add_skin_features(
    v21_frames['most_recent_prior']['hold'])
assert not set(v22_dev.subject_id) & set(v22_hold.subject_id)

# Recover the exact V21 context-only feature set so the tone-blind row
# reproduces the recommended 5.795 SBP / 4.486 DBP comparison point.
_v21_feature_table = pd.DataFrame(v21_feature_rows)
V22_BASE_FEATURES = {}
for target in ['sbp', 'dbp']:
    match = _v21_feature_table[
        (_v21_feature_table.target == target) &
        (_v21_feature_table.configuration ==
         'prior_absolute_error_plus_context')]
    if len(match) != 1:
        raise RuntimeError(
            f'Expected one V21 context-only feature row for {target}; found {len(match)}')
    V22_BASE_FEATURES[target] = match.iloc[0].features.split('|')

print(f'V22 development {v22_dev.subject_id.nunique()} participants | '
      f'holdout {v22_hold.subject_id.nunique()} participants')
print('Updated skin-tone features:', V22_SKIN_FEATURES)
display(v22_hold[['subject_id', 'fst_band']].drop_duplicates()
        .fst_band.value_counts().rename('holdout_participants').to_frame())


def v22_bootstrap(values, seed, repeats=V22_BOOTSTRAP_REPEATS):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    sampled = values[
        rng.integers(0, len(values), size=(repeats, len(values)))]
    return tuple(np.quantile(sampled.mean(axis=1), [0.025, 0.975]))


def v22_fit_and_score(target, features, use_skin_weights, per_group_gate):
    features = [
        f for f in dict.fromkeys(features)
        if f in v22_dev.columns and f in v22_hold.columns
        and v22_dev[f].notna().any()
        and v22_dev[f].nunique(dropna=True) > 1
    ]
    fit = v22_dev.dropna(subset=['delta_' + target]).copy()
    model = make_model_v17(V12_PARAMS, loss='absolute_error')
    model.fit(
        fit[features], fit['delta_' + target],
        histgradientboostingregressor__sample_weight=abl_weights(
            fit, 'fst_band', use_skin_weights))
    development_delta = model.predict(fit[features])
    gate = abl_fit_gate(
        fit, target, 'baseline_' + target, development_delta,
        'fst_band', per_group_gate)
    hold_delta = model.predict(v22_hold[features])
    prediction, active = abl_apply_gate(
        v22_hold, 'baseline_' + target, hold_delta, gate, 'fst_band')
    per_subject = (v22_hold.assign(
        _e=np.abs(prediction - v22_hold[target]))
        .groupby(['subject_id', 'fst_band'], as_index=False)._e.mean())
    return per_subject, features, gate, float(active.mean())


V22_CONFIGURATIONS = [
    # label, add skin features, balance skin bands, group-specific gate
    ('tone_blind', False, False, False),
    ('skin_features_only', True, False, False),
    ('skin_features_plus_balance', True, True, False),
    ('updated_skin_aware', True, True, True),
]

v22_overall_rows, v22_group_rows = [], []
v22_per_subject, v22_feature_rows = {}, []
for target_index, target in enumerate(['sbp', 'dbp']):
    for config_index, (label, add_skin, balanced, group_gate) in enumerate(
            V22_CONFIGURATIONS):
        features = V22_BASE_FEATURES[target] + (
            V22_SKIN_FEATURES if add_skin else [])
        per_subject, used, gate, activation = v22_fit_and_score(
            target, features, balanced, group_gate)
        v22_per_subject[(target, label)] = per_subject
        lo, hi = v22_bootstrap(
            per_subject._e,
            SEED + 22000 + 100 * target_index + config_index)
        group_mae = per_subject.groupby('fst_band')._e.mean()
        v22_overall_rows.append({
            'target': target, 'configuration': label,
            'holdout_mae': float(per_subject._e.mean()),
            'mae_ci_low': lo, 'mae_ci_high': hi,
            'worst_group_mae': float(group_mae.max()),
            'between_group_gap': float(group_mae.max() - group_mae.min()),
            'feature_count': len(used),
            'gate_activation_rate': activation,
        })
        for band_index, (band, group) in enumerate(
                per_subject.groupby('fst_band')):
            group_lo, group_hi = v22_bootstrap(
                group._e,
                SEED + 22500 + 1000 * target_index
                + 100 * config_index + band_index)
            v22_group_rows.append({
                'target': target, 'configuration': label,
                'fst_band': band, 'participants': len(group),
                'mae': float(group._e.mean()),
                'mae_ci_low': group_lo, 'mae_ci_high': group_hi,
            })
        v22_feature_rows.append({
            'target': target, 'configuration': label,
            'features': '|'.join(used), 'gate': repr(gate),
        })

v22_overall = pd.DataFrame(v22_overall_rows)
v22_by_group = pd.DataFrame(v22_group_rows)

# Primary paired comparison: complete updated skin-aware model vs tone-blind.
v22_comparisons = []
for target_index, target in enumerate(['sbp', 'dbp']):
    blind = v22_per_subject[(target, 'tone_blind')]
    aware = v22_per_subject[(target, 'updated_skin_aware')]
    for scope_index, scope in enumerate(
            ['overall'] + sorted(blind.fst_band.unique().tolist())):
        left = blind if scope == 'overall' else blind[blind.fst_band == scope]
        right = aware if scope == 'overall' else aware[aware.fst_band == scope]
        aligned = left[['subject_id', '_e']].merge(
            right[['subject_id', '_e']], on='subject_id',
            suffixes=('_blind', '_aware'), validate='one_to_one')
        improvement = aligned._e_blind - aligned._e_aware
        lo, hi = v22_bootstrap(
            improvement,
            SEED + 23000 + 100 * target_index + scope_index)
        v22_comparisons.append({
            'target': target, 'scope': scope,
            'participants': len(aligned),
            'tone_blind_mae': float(aligned._e_blind.mean()),
            'skin_aware_mae': float(aligned._e_aware.mean()),
            'improvement_mmhg': float(improvement.mean()),
            'improvement_percent': float(
                100 * improvement.mean() / aligned._e_blind.mean()),
            'ci_low': lo, 'ci_high': hi,
            'supported_95pct': bool(lo > 0),
        })
v22_comparisons = pd.DataFrame(v22_comparisons)

print('\n=== V22 overall holdout comparison ===')
display(v22_overall.round(3))
print('\n=== V22 MAE within Fitzpatrick bands ===')
display(v22_by_group.round(3))
print('\n=== Updated skin-aware minus tone-blind paired comparison ===')
display(v22_comparisons.round(3))
print('Positive improvement favors the updated skin-aware model. Interpret the')
print('Fitzpatrick V-VI interval cautiously because that holdout group is small.')

v22_overall.to_csv(
    V22_RESULTS / 'v22_overall_skintone_comparison.csv', index=False)
v22_by_group.to_csv(
    V22_RESULTS / 'v22_fitzpatrick_group_mae.csv', index=False)
v22_comparisons.to_csv(
    V22_RESULTS / 'v22_paired_skin_aware_vs_blind_ci.csv', index=False)
pd.DataFrame(v22_feature_rows).to_csv(
    V22_RESULTS / 'v22_features_and_gates.csv', index=False)


---

# V26: strengthened validation for publication

This section addresses four review-critical questions without changing the historical V25
results above:

1. **Calibration dose:** compare one cuff reading with the mean of two and all available
   initial calibration readings on identical future rows.
2. **Deployment endpoint:** exclude every row whose protocol label contains
   `calibration` from both fitting and scoring.
3. **Single-split stability:** evaluate the prespecified final comparison with repeated,
   participant-stratified outer cross-validation. Feature ranking, preprocessing, fitting,
   and gating are repeated inside each outer training fold.
4. **Context transportability:** separate sensor-available context from protocol-label and
   acquisition-duration context.

V26 is an internal robustness analysis, not a substitute for a new external test cohort.
Because the final model specification was developed using AuroraBP, do not call these
folds a new untouched validation. Call them repeated participant-level internal validation.


In [ ]:
# V26 configuration and calibration-dose reconstruction.
V26_RESULTS = OUT_DIR / "avct_v26_strengthened_validation"
V26_RESULTS.mkdir(parents=True, exist_ok=True)

V26_OUTER_SPLITS = 5
V26_REPEATS = 3
V26_BOOTSTRAPS = 5000
V26_TOP_K = 24
V26_CAL_PATTERN = r"^\s*calibration\s+start"
V26_ANY_CAL_PATTERN = r"\bcalibration\b"

def v26_build_calibration_dose(frame, anchor_readings):
    # Build a common post-calibration cohort using exactly k anchor readings.
    # Scoring begins only after the complete initial calibration block, so all dose arms
    # see identical future rows. Later calibration-labelled rows remain auditable.
    source = frame.copy()
    source["sbp"] = pd.to_numeric(source["sbp"], errors="coerce")
    source["dbp"] = pd.to_numeric(source["dbp"], errors="coerce")
    source["_parsed_time"] = pd.to_datetime(source["date_time"], errors="coerce")
    source["_is_initial_calibration"] = (
        source["phase"].astype(str).str.lower().eq("initial")
        & source["measurement"].astype(str).str.contains(
            V26_CAL_PATTERN, case=False, regex=True, na=False
        )
    )
    numeric = source.select_dtypes(include=[np.number]).columns.tolist()
    waveform_features = [
        column for column in numeric if column.startswith(("raw_", "pc_"))
    ]
    rows, audit = [], []
    for pid, group in source.dropna(subset=["sbp", "dbp"]).groupby("pid"):
        group = group.sort_values(
            ["_parsed_time", "_source_order"], na_position="last"
        )
        calibrations = group[group["_is_initial_calibration"]].copy()
        if len(calibrations) < anchor_readings:
            audit.append({
                "pid": pid, "status": "excluded_insufficient_calibration",
                "available": len(calibrations), "anchor_readings": anchor_readings,
            })
            continue
        selected = calibrations.head(anchor_readings)
        calibration_end_order = calibrations["_source_order"].max()
        calibration_end_time = calibrations["_parsed_time"].max()

        anchor = selected.iloc[0].copy()
        anchor["sbp"] = selected["sbp"].mean()
        anchor["dbp"] = selected["dbp"].mean()
        for column in waveform_features:
            anchor[column] = pd.to_numeric(
                selected[column], errors="coerce"
            ).median()
        # Time zero is the end of the full calibration block for every dose arm.
        anchor["_source_order"] = calibration_end_order
        anchor["_parsed_time"] = calibration_end_time
        anchor["date_time"] = calibration_end_time
        anchor["measurement"] = f"V26 anchor from {anchor_readings} reading(s)"
        anchor["subject_id"] = str(pid)
        anchor["start_seconds"] = 0.0

        if pd.notna(calibration_end_time):
            future = group[
                (~group["_is_initial_calibration"])
                & (
                    (group["_parsed_time"] > calibration_end_time)
                    | (
                        group["_parsed_time"].eq(calibration_end_time)
                        & group["_source_order"].gt(calibration_end_order)
                    )
                )
            ]
        else:
            future = group[
                (~group["_is_initial_calibration"])
                & group["_source_order"].gt(calibration_end_order)
            ]
        if future.empty:
            audit.append({
                "pid": pid, "status": "excluded_no_future",
                "available": len(calibrations), "anchor_readings": anchor_readings,
            })
            continue

        rows.append(anchor)
        for _, current in future.iterrows():
            current = current.copy()
            current["subject_id"] = str(pid)
            if pd.notna(current["_parsed_time"]) and pd.notna(calibration_end_time):
                current["start_seconds"] = (
                    current["_parsed_time"] - calibration_end_time
                ).total_seconds()
            else:
                current["start_seconds"] = float(
                    (current["_source_order"] - calibration_end_order) * WINDOW_SECONDS
                )
            rows.append(current)
        audit.append({
            "pid": pid, "status": "included", "available": len(calibrations),
            "anchor_readings": anchor_readings, "future_rows": len(future),
        })
    return pd.DataFrame(rows), pd.DataFrame(audit)

def v26_prepare_dose(anchor_readings):
    ordered, audit = v26_build_calibration_dose(
        aurora_features, anchor_readings
    )
    frame = build_delta_table(
        ordered, RAW, PC, anchor="session_start"
    ).replace([np.inf, -np.inf], np.nan)
    frame, context, context_audit = add_aurora_measurement_context(frame)
    frame["subject_id"] = frame["subject_id"].astype(str)
    frame["fst_band"] = pd.cut(
        frame["fitzpatrick_scale"], [0, 2, 4, 6],
        labels=["Fitzpatrick I-II", "Fitzpatrick III-IV", "Fitzpatrick V-VI"],
    ).astype(str)
    frame["log_elapsed_days"] = np.log1p(
        frame["elapsed_seconds"].clip(lower=0) / 86400
    )
    frame["log_session_elapsed_days"] = np.log1p(
        frame["session_elapsed_seconds"].clip(lower=0) / 86400
    )
    frame["v26_is_calibration_labeled"] = frame["measurement"].astype(str).str.contains(
        V26_ANY_CAL_PATTERN, case=False, regex=True, na=False
    )
    for target in ["sbp", "dbp"]:
        prior_features = uci_prior_features[target]
        if all(feature in frame.columns for feature in prior_features):
            frame["uci_prior_delta_" + target] = uci_prior_models[target].predict(
                frame[prior_features]
            )
    return frame, context, audit, context_audit

v26_dose_frames = {}
v26_dose_audits = {}
for dose in [1, 2, 3]:
    dose_frame, dose_context, dose_audit, dose_context_audit = v26_prepare_dose(dose)
    v26_dose_frames[dose] = dose_frame
    v26_dose_audits[dose] = dose_audit
    print(
        f"dose={dose}: {dose_frame.subject_id.nunique()} participants, "
        f"{len(dose_frame)} post-calibration rows, "
        f"{dose_frame.v26_is_calibration_labeled.mean():.1%} calibration-labelled"
    )

# Use the exact intersection of participant-row keys for every calibration dose.
def v26_row_key(frame):
    return pd.MultiIndex.from_frame(
        # _source_order comes from the original Aurora table and is stable even when a
        # stricter calibration dose excludes additional participants before reconstruction.
        frame[["subject_id", "_source_order"]].astype(str)
    )

common_keys = None
for dose, frame in v26_dose_frames.items():
    keys = set(v26_row_key(frame).tolist())
    common_keys = keys if common_keys is None else common_keys.intersection(keys)
common_keys = set(common_keys)
for dose, frame in list(v26_dose_frames.items()):
    keys = v26_row_key(frame)
    keep = np.array([key in common_keys for key in keys], dtype=bool)
    v26_dose_frames[dose] = frame.loc[keep].reset_index(drop=True)

print("Common dose-comparison rows:", len(common_keys))
print(
    "Common participants:",
    v26_dose_frames[1]["subject_id"].nunique(),
)


In [ ]:
# Calibration-dose audit before fitting any model.
dose_rows = []
dose_per_subject = {}
for dose, frame in v26_dose_frames.items():
    for endpoint_name, endpoint in [
        ("all_post_initial_calibration", frame),
        ("strict_noncalibration_rows", frame[~frame["v26_is_calibration_labeled"]]),
    ]:
        for target in ["sbp", "dbp"]:
            per_subject = endpoint.assign(
                _ae=np.abs(endpoint["baseline_" + target] - endpoint[target])
            ).groupby("subject_id")["_ae"].mean()
            dose_per_subject[(dose, endpoint_name, target)] = per_subject
            lo, hi = v21_bootstrap(
                per_subject.to_numpy(), SEED + 26000 + 100 * dose
            )
            dose_rows.append({
                "anchor_readings": dose,
                "endpoint": endpoint_name,
                "target": target,
                "participants": len(per_subject),
                "rows": len(endpoint),
                "calibration_retention_MAE": per_subject.mean(),
                "CI_low": lo,
                "CI_high": hi,
            })

v26_calibration_dose = pd.DataFrame(dose_rows)
display(v26_calibration_dose.round(3))

dose_contrasts = []
for endpoint_name in ["all_post_initial_calibration", "strict_noncalibration_rows"]:
    for target in ["sbp", "dbp"]:
        one = dose_per_subject[(1, endpoint_name, target)]
        for dose in [2, 3]:
            repeated = dose_per_subject[(dose, endpoint_name, target)]
            aligned = pd.concat(
                [one.rename("one"), repeated.rename("repeated")], axis=1
            ).dropna()
            # Positive means averaging repeated calibration readings reduces error.
            improvement = aligned["one"] - aligned["repeated"]
            lo, hi = v21_bootstrap(
                improvement, SEED + 26100 + 100 * dose
            )
            dose_contrasts.append({
                "endpoint": endpoint_name,
                "target": target,
                "contrast": f"one_reading_minus_{dose}_reading_mean",
                "participants": len(aligned),
                "improvement_from_repeated_calibration": improvement.mean(),
                "CI_low": lo,
                "CI_high": hi,
                "supported": bool(lo > 0),
            })
v26_calibration_contrasts = pd.DataFrame(dose_contrasts)
display(v26_calibration_contrasts.round(3))

v26_calibration_dose.to_csv(
    V26_RESULTS / "v26_calibration_dose_mae.csv", index=False
)
v26_calibration_contrasts.to_csv(
    V26_RESULTS / "v26_calibration_dose_contrasts.csv", index=False
)
print(
    "Paper rule: call the method one-reading calibration only if the dose=1 model "
    "is the reported primary analysis. Otherwise use one-session repeated calibration."
)


## V26 repeated participant-level internal validation

The final V25 comparison is frozen before this run: squared-error legacy features versus
the same model plus context. Every outer test participant is excluded from feature ranking,
preprocessing, model fitting, and gate selection. The test is repeated with three shuffled
five-fold participant partitions. Results are averaged within participant across repeats,
then bootstrapped by participant.

This strengthens stability evidence but does not erase the fact that AuroraBP informed model
development. A new dataset or prospective cohort remains necessary for external validation.


In [ ]:
# Use the three-reading mean to reproduce the current one-session definition, but enforce
# the strict deployment endpoint: calibration-labelled rows are absent from fit and score.
v26_cv_frame = v26_dose_frames[3].copy()
v26_cv_frame = v26_cv_frame[~v26_cv_frame["v26_is_calibration_labeled"]].copy()
v26_cv_frame = v26_cv_frame.dropna(subset=["fst_band"]).reset_index(drop=True)

V26_CONTEXT_SETS = {
    "no_context": [],
    # Directly available from the optical acquisition at prediction time.
    "sensor_only": [
        feature for feature in ["context_optical_quality"]
        if feature in v26_cv_frame.columns
    ],
    # Requires a wearable posture/activity classifier; no protocol phase or duration.
    "wearable_state": [
        feature for feature in [
            "context_optical_quality", "context_posture_seated",
            "context_activity_post_exercise",
        ] if feature in v26_cv_frame.columns
    ],
    # Removes the calibration flag, protocol phase, cuff-pressure quality, and duration.
    "deployment_conservative": [
        feature for feature in V21_CONTEXT_FEATURES
        if feature in {
            "context_optical_quality", "context_posture_seated",
            "context_activity_post_exercise",
        }
    ],
    # Historical V25 context, except calibration-labelled rows are absent by construction.
    "full_historical_context": [
        feature for feature in V21_CONTEXT_FEATURES
        if feature != "context_measurement_calibration"
        and feature in v26_cv_frame.columns
    ],
}
# Remove duplicate context arms while preserving interpretable names in the audit.
print("Context arms:")
for name, features in V26_CONTEXT_SETS.items():
    print(name, features)

def v26_participant_folds(frame, n_splits, repeat):
    people = frame[["subject_id", "fst_band"]].drop_duplicates("subject_id")
    people = people.sort_values("subject_id").reset_index(drop=True)
    splitter = StratifiedKFold(
        n_splits=n_splits, shuffle=True,
        random_state=SEED + 26200 + repeat,
    )
    for fold, (train_people, test_people) in enumerate(
        splitter.split(people["subject_id"], people["fst_band"]), 1
    ):
        train_ids = set(people.iloc[train_people]["subject_id"])
        test_ids = set(people.iloc[test_people]["subject_id"])
        train = frame[frame["subject_id"].isin(train_ids)].copy()
        test = frame[frame["subject_id"].isin(test_ids)].copy()
        assert not set(train["subject_id"]) & set(test["subject_id"])
        yield fold, train, test

def v26_core_features(train, target):
    base = "baseline_" + target
    pool = [
        feature for feature in dict.fromkeys(
            DELTA_RAW + RELATIVE_RAW + [
                "elapsed_seconds", "log_elapsed_days",
                "session_elapsed_seconds", "log_session_elapsed_days",
                "uci_prior_delta_" + target,
            ]
        )
        if feature in train.columns and train[feature].notna().any()
        and train[feature].nunique(dropna=True) > 1
    ]
    ranking = abl_rank(
        train.dropna(subset=["delta_" + target]), pool,
        "delta_" + target, "fst_band", False,
    )
    return list(dict.fromkeys(
        [base] + ranking["feature"].head(V26_TOP_K).tolist()
    ))

cv_error_rows = []
cv_fold_rows = []
for repeat in range(V26_REPEATS):
    for fold, train, test in v26_participant_folds(
        v26_cv_frame, V26_OUTER_SPLITS, repeat
    ):
        for target_index, target in enumerate(["sbp", "dbp"]):
            core = v26_core_features(train, target)
            baseline_per = test.assign(
                _ae=np.abs(test["baseline_" + target] - test[target])
            ).groupby("subject_id")["_ae"].mean()
            for arm, context_features in V26_CONTEXT_SETS.items():
                features = core + context_features
                per_subject, used, gate, activation = v21_fit_and_score(
                    train, test, target, features, "squared_error"
                )
                for subject_id, mae in per_subject.items():
                    cv_error_rows.append({
                        "repeat": repeat + 1,
                        "outer_fold": fold,
                        "target": target,
                        "arm": arm,
                        "subject_id": subject_id,
                        "mae": mae,
                        "calibration_retention_mae": baseline_per.get(subject_id, np.nan),
                    })
                cv_fold_rows.append({
                    "repeat": repeat + 1,
                    "outer_fold": fold,
                    "target": target,
                    "arm": arm,
                    "test_participants": test["subject_id"].nunique(),
                    "test_rows": len(test),
                    "MAE": per_subject.mean(),
                    "calibration_retention_MAE": baseline_per.mean(),
                    "feature_count": len(used),
                    "gate_activation": activation,
                })
        print(f"repeat {repeat + 1}/{V26_REPEATS}, fold {fold}/{V26_OUTER_SPLITS} complete")

v26_cv_errors = pd.DataFrame(cv_error_rows)
v26_cv_folds = pd.DataFrame(cv_fold_rows)
display(v26_cv_folds.groupby(["target", "arm"]).agg(
    mean_MAE=("MAE", "mean"),
    fold_SD=("MAE", "std"),
    mean_calibration_retention=("calibration_retention_MAE", "mean"),
).round(3))

# Trained-model calibration-dose comparison. Each dose uses the same participant folds,
# future-row keys, strict endpoint, model family, and full historical context feature names.
# Feature ranking is repeated within the corresponding dose/fold training partition.
dose_model_error_rows = []
dose_model_fold_rows = []
for dose, dose_source in v26_dose_frames.items():
    dose_frame = dose_source[~dose_source["v26_is_calibration_labeled"]].copy()
    dose_frame = dose_frame.dropna(subset=["fst_band"]).reset_index(drop=True)
    dose_context = [
        feature for feature in V26_CONTEXT_SETS["full_historical_context"]
        if feature in dose_frame.columns
    ]
    for repeat in range(V26_REPEATS):
        for fold, train, test in v26_participant_folds(
            dose_frame, V26_OUTER_SPLITS, repeat
        ):
            for target in ["sbp", "dbp"]:
                core = v26_core_features(train, target)
                per_subject, used, gate, activation = v21_fit_and_score(
                    train, test, target, core + dose_context, "squared_error"
                )
                baseline_per = test.assign(
                    _ae=np.abs(test["baseline_" + target] - test[target])
                ).groupby("subject_id")["_ae"].mean()
                for subject_id, mae in per_subject.items():
                    dose_model_error_rows.append({
                        "anchor_readings": dose,
                        "repeat": repeat + 1,
                        "outer_fold": fold,
                        "target": target,
                        "subject_id": subject_id,
                        "mae": mae,
                        "calibration_retention_mae": baseline_per.get(subject_id, np.nan),
                    })
                dose_model_fold_rows.append({
                    "anchor_readings": dose,
                    "repeat": repeat + 1,
                    "outer_fold": fold,
                    "target": target,
                    "participants": len(per_subject),
                    "rows": len(test),
                    "MAE": per_subject.mean(),
                    "calibration_retention_MAE": baseline_per.mean(),
                    "feature_count": len(used),
                    "gate_activation": activation,
                })
        print(f"calibration dose {dose}, repeat {repeat + 1}/{V26_REPEATS} complete")

v26_dose_model_errors = pd.DataFrame(dose_model_error_rows)
v26_dose_model_folds = pd.DataFrame(dose_model_fold_rows)


In [ ]:
# Aggregate repeated predictions/errors within participant and perform paired cluster bootstrap.
v26_cv_participant = (
    v26_cv_errors.groupby(["target", "arm", "subject_id"], as_index=False)
    .agg(
        mae=("mae", "mean"),
        calibration_retention_mae=("calibration_retention_mae", "mean"),
        repeats=("repeat", "nunique"),
    )
)

def v26_bootstrap_mean(values, seed):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    boot = np.empty(V26_BOOTSTRAPS)
    for iteration in range(V26_BOOTSTRAPS):
        boot[iteration] = rng.choice(values, len(values), replace=True).mean()
    return tuple(np.quantile(boot, [0.025, 0.975]))

summary_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    target_errors = v26_cv_participant[v26_cv_participant["target"] == target]
    for arm_index, (arm, group) in enumerate(target_errors.groupby("arm")):
        mae_low, mae_high = v26_bootstrap_mean(
            group["mae"], SEED + 26300 + 100 * target_index + arm_index
        )
        improvement = group["calibration_retention_mae"] - group["mae"]
        imp_low, imp_high = v26_bootstrap_mean(
            improvement, SEED + 26400 + 100 * target_index + arm_index
        )
        summary_rows.append({
            "target": target,
            "arm": arm,
            "participants": group["subject_id"].nunique(),
            "participant_MAE": group["mae"].mean(),
            "MAE_CI_low": mae_low,
            "MAE_CI_high": mae_high,
            "calibration_retention_MAE": group["calibration_retention_mae"].mean(),
            "improvement_vs_calibration": improvement.mean(),
            "improvement_CI_low": imp_low,
            "improvement_CI_high": imp_high,
            "fraction_helped_vs_calibration": (improvement > 0).mean(),
        })
v26_cv_summary = pd.DataFrame(summary_rows)

contrast_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    pivot = v26_cv_participant[
        v26_cv_participant["target"] == target
    ].pivot(index="subject_id", columns="arm", values="mae")
    for arm_index, arm in enumerate(
        [name for name in V26_CONTEXT_SETS if name != "no_context"]
    ):
        aligned = pivot[["no_context", arm]].dropna()
        improvement = aligned["no_context"] - aligned[arm]
        low, high = v26_bootstrap_mean(
            improvement, SEED + 26500 + 100 * target_index + arm_index
        )
        contrast_rows.append({
            "target": target,
            "from": "no_context",
            "to": arm,
            "participants": len(aligned),
            "context_improvement_MAE": improvement.mean(),
            "CI_low": low,
            "CI_high": high,
            "supported": bool(low > 0),
            "fraction_helped": (improvement > 0).mean(),
        })
v26_context_contrasts = pd.DataFrame(contrast_rows)

# Aggregate the one/two/three-reading trained-model comparison.
v26_dose_model_participant = (
    v26_dose_model_errors.groupby(
        ["anchor_readings", "target", "subject_id"], as_index=False
    ).agg(
        mae=("mae", "mean"),
        calibration_retention_mae=("calibration_retention_mae", "mean"),
        repeats=("repeat", "nunique"),
    )
)
dose_model_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    for dose in [1, 2, 3]:
        group = v26_dose_model_participant[
            (v26_dose_model_participant["target"] == target)
            & (v26_dose_model_participant["anchor_readings"] == dose)
        ]
        low, high = v26_bootstrap_mean(
            group["mae"], SEED + 26600 + 100 * target_index + dose
        )
        improvement = group["calibration_retention_mae"] - group["mae"]
        imp_low, imp_high = v26_bootstrap_mean(
            improvement, SEED + 26700 + 100 * target_index + dose
        )
        dose_model_rows.append({
            "target": target,
            "anchor_readings": dose,
            "participants": group["subject_id"].nunique(),
            "model_MAE": group["mae"].mean(),
            "MAE_CI_low": low,
            "MAE_CI_high": high,
            "calibration_retention_MAE": group["calibration_retention_mae"].mean(),
            "improvement_vs_retention": improvement.mean(),
            "improvement_CI_low": imp_low,
            "improvement_CI_high": imp_high,
        })
v26_dose_model_summary = pd.DataFrame(dose_model_rows)

dose_model_contrast_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    pivot = v26_dose_model_participant[
        v26_dose_model_participant["target"] == target
    ].pivot(index="subject_id", columns="anchor_readings", values="mae")
    for dose in [2, 3]:
        aligned = pivot[[1, dose]].dropna()
        improvement = aligned[1] - aligned[dose]
        low, high = v26_bootstrap_mean(
            improvement, SEED + 26800 + 100 * target_index + dose
        )
        dose_model_contrast_rows.append({
            "target": target,
            "contrast": f"one_reading_minus_{dose}_reading_model",
            "participants": len(aligned),
            "model_improvement_from_repeated_calibration": improvement.mean(),
            "CI_low": low,
            "CI_high": high,
            "supported": bool(low > 0),
        })
v26_dose_model_contrasts = pd.DataFrame(dose_model_contrast_rows)

print("=== Repeated participant-CV summary ===")
display(v26_cv_summary.round(3))
print("=== Paired context contrasts; positive favors context ===")
display(v26_context_contrasts.round(3))
print("=== Trained model by calibration dose ===")
display(v26_dose_model_summary.round(3))
print("=== Paired trained-model calibration-dose contrasts ===")
display(v26_dose_model_contrasts.round(3))

v26_cv_errors.to_csv(V26_RESULTS / "v26_repeated_cv_errors.csv", index=False)
v26_cv_folds.to_csv(V26_RESULTS / "v26_repeated_cv_folds.csv", index=False)
v26_cv_participant.to_csv(
    V26_RESULTS / "v26_repeated_cv_participant.csv", index=False
)
v26_cv_summary.to_csv(V26_RESULTS / "v26_repeated_cv_summary.csv", index=False)
v26_context_contrasts.to_csv(
    V26_RESULTS / "v26_context_transportability.csv", index=False
)
v26_dose_model_errors.to_csv(
    V26_RESULTS / "v26_calibration_dose_model_errors.csv", index=False
)
v26_dose_model_folds.to_csv(
    V26_RESULTS / "v26_calibration_dose_model_folds.csv", index=False
)
v26_dose_model_participant.to_csv(
    V26_RESULTS / "v26_calibration_dose_model_participant.csv", index=False
)
v26_dose_model_summary.to_csv(
    V26_RESULTS / "v26_calibration_dose_model_summary.csv", index=False
)
v26_dose_model_contrasts.to_csv(
    V26_RESULTS / "v26_calibration_dose_model_contrasts.csv", index=False
)


In [ ]:
# Publication decision table: selects wording from the actual V26 outcomes.
dose_one = v26_calibration_dose[
    (v26_calibration_dose["anchor_readings"] == 1)
    & (v26_calibration_dose["endpoint"] == "strict_noncalibration_rows")
]
dose_three = v26_calibration_dose[
    (v26_calibration_dose["anchor_readings"] == 3)
    & (v26_calibration_dose["endpoint"] == "strict_noncalibration_rows")
]

decision_rows = []
for target in ["sbp", "dbp"]:
    one = dose_one[dose_one["target"] == target].iloc[0]
    three = dose_three[dose_three["target"] == target].iloc[0]
    one_model = v26_dose_model_summary[
        (v26_dose_model_summary["target"] == target)
        & (v26_dose_model_summary["anchor_readings"] == 1)
    ].iloc[0]
    three_model = v26_dose_model_summary[
        (v26_dose_model_summary["target"] == target)
        & (v26_dose_model_summary["anchor_readings"] == 3)
    ].iloc[0]
    conservative = v26_cv_summary[
        (v26_cv_summary["target"] == target)
        & (v26_cv_summary["arm"] == "deployment_conservative")
    ].iloc[0]
    full = v26_cv_summary[
        (v26_cv_summary["target"] == target)
        & (v26_cv_summary["arm"] == "full_historical_context")
    ].iloc[0]
    decision_rows.append({
        "target": target.upper(),
        "one_reading_retention_MAE": one["calibration_retention_MAE"],
        "three_reading_mean_retention_MAE": three["calibration_retention_MAE"],
        "one_reading_model_MAE": one_model["model_MAE"],
        "three_reading_mean_model_MAE": three_model["model_MAE"],
        "repeated_CV_deployment_context_MAE": conservative["participant_MAE"],
        "repeated_CV_full_context_MAE": full["participant_MAE"],
        "deployment_context_supported": bool(
            v26_context_contrasts.loc[
                (v26_context_contrasts["target"] == target)
                & (v26_context_contrasts["to"] == "deployment_conservative"),
                "supported",
            ].iloc[0]
        ),
    })
v26_publication_decisions = pd.DataFrame(decision_rows)
display(v26_publication_decisions.round(3))

v26_publication_decisions.to_csv(
    V26_RESULTS / "v26_publication_decision_table.csv", index=False
)

# Correct the sign label used in the manuscript's skin-tone comparison. The stored V22
# quantity is tone-blind MAE minus skin-aware MAE: positive favors skin-aware; negative
# favors tone-blind. It is not full-minus-tone-blind.
v26_skin_sign_corrected = v22_comparisons.copy().rename(columns={
    "improvement_mmhg": "tone_blind_minus_skin_aware_MAE",
    "improvement_percent": "tone_blind_minus_skin_aware_percent",
})
v26_skin_sign_corrected["favored_model"] = np.select(
    [
        v26_skin_sign_corrected["ci_low"] > 0,
        v26_skin_sign_corrected["ci_high"] < 0,
    ],
    ["skin_aware", "tone_blind"],
    default="no_established_difference",
)
display(v26_skin_sign_corrected.round(3))
v26_skin_sign_corrected.to_csv(
    V26_RESULTS / "v26_skin_comparison_sign_corrected.csv", index=False
)

print("\nRequired manuscript wording after V26")
print("-------------------------------------")
print("1. Use 'one-reading calibration' only for the dose=1 arm.")
print("2. Otherwise use 'one-session calibration using repeated readings'.")
print("3. Lead with the strict non-calibration-row endpoint.")
print("4. Call V26 repeated participant-CV internal validation, not untouched external validation.")
print("5. Call posture/activity context deployment-available only if a wearable classifier supplies it.")
print("6. Retain the original 168-person split as historical development evidence, not a pristine final test.")
print("7. A new cohort remains the only complete fix for prior reuse of the Aurora holdout.")
print("8. Label the skin contrast as tone-blind minus skin-aware; the manuscript currently reverses it.")
print("9. Replace 'pre-registered' with 'prespecified' unless a timestamped registration exists.")
print("Saved V26 outputs to:", V26_RESULTS)


---

# V28: calibration dispersion and cross-target features

## Where the idea comes from

`v26_build_calibration_dose` builds the anchor as
`anchor["sbp"] = selected["sbp"].mean()` — it averages the k initial readings
and **throws their spread away**.

That spread is a free per-participant estimate of BP lability, and lability is
what predicts |Δ| — the dominant term in your own Proposition 1 decomposition.
Table III puts physiological drift SD at 13.52 mmHg against a cuff-noise floor
of 2.198, so almost all of the remaining error is real drift, and the anchor
block already contains a measurement of how much each person drifts.

Four features per target, all available at prediction time:

| feature | what it captures |
|---|---|
| `cal_sd` | dispersion across the k readings — BP lability |
| `cal_range` | max − min |
| `cal_trend` | last − first: **was BP still settling when we calibrated?** |
| `cal_n` | how many readings the anchor averaged |

`cal_trend` is the interesting one. If BP was still falling across the
calibration block, the participant had not settled, and the *direction* of
subsequent drift is partly predictable rather than just its magnitude.

These are computed from only the readings that dose actually used, so no
information crosses between dose arms. At k = 1 they are undefined and left NaN
— which is itself informative, and gives a mechanism for Table II.

## The second block: cross-target features

`v26_core_features` builds the SBP and DBP models in complete isolation, but the
two deltas are correlated. This adds the other target's anchor level and prior,
plus **pulse pressure** and **mean arterial pressure** — standard haemodynamic
quantities that neither model currently sees.

## Arms

A reproduces the V26 primary; B adds dispersion, C adds cross-target, D adds
both. Same 3×5-fold participant CV, same strict non-calibration-row endpoint,
same context set, paired participant bootstrap against A.

## Verified before shipping

The leakage test matters most here: corrupting **every** non-calibration row
leaves the dispersion features bit-identical, confirming they read only the k
calibration readings. On synthetic data where lability genuinely drives drift,
`cal_sd` recovers the underlying lability at r = 0.66 and predicts subsequent
drift at r = 0.64. `cal_trend` returns −20.0 for a calibration block falling
140 → 130 → 120.

## Runtime

Four arms × two targets × 15 folds — the same order as the V26 primary cell.

## V28: dispersion and cross-target arms

In [ ]:
# ============================================================================
# V28: CALIBRATION DISPERSION AND CROSS-TARGET FEATURES
# ----------------------------------------------------------------------------
# Two additions, both free at prediction time, both evaluated under the same
# 3x5-fold participant CV and strict non-calibration-row endpoint as V26.
#
# 1. CALIBRATION DISPERSION.
#    v26_build_calibration_dose collapses the k initial readings to
#    selected["sbp"].mean() and discards their spread. That spread is a
#    per-participant estimate of BP lability, and lability is what predicts
#    |delta| -- the dominant term in the Proposition 1 decomposition, since
#    Table III put physiological drift SD at 13.52 mmHg against a cuff-noise
#    floor of 2.198.
#      cal_sd     dispersion across the k readings
#      cal_range  max - min
#      cal_trend  last - first: was BP still settling when we calibrated?
#      cal_n      how many readings the anchor averaged
#
#    Computed from ONLY the k readings that dose actually used, so no
#    information crosses between dose arms. At k=1 they are undefined and left
#    NaN -- itself informative, since it gives a mechanism for why Table II
#    shows three readings beating one beyond simple averaging.
#
# 2. CROSS-TARGET FEATURES.
#    v26_core_features builds the SBP and DBP models in complete isolation,
#    but the two deltas are correlated. This adds the other target's anchor
#    level and prior, plus pulse pressure and mean arterial pressure.
#
# Arms: A reproduces the V26 primary; B, C, D add the two blocks and both.
# ============================================================================
import inspect

V28_RESULTS = OUT_DIR / 'avct_v28_dispersion_crosstarget'
V28_RESULTS.mkdir(parents=True, exist_ok=True)


def v28_ci(values, seed):
    """Participant bootstrap CI, tolerant of v26_bootstrap_mean's signature.

    The earlier version of this cell assumed (values, n_boot, seed) and failed
    with a TypeError. Rather than guess, adapt to whatever signature exists and
    fall back to a self-contained bootstrap if the call still does not fit.
    """
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    try:
        params = inspect.signature(v26_bootstrap_mean).parameters
        second = list(params)[1] if len(params) > 1 else ''
        if len(params) >= 3:
            return v26_bootstrap_mean(values, V26_BOOTSTRAPS, seed)
        if len(params) == 2 and any(k in second.lower()
                                    for k in ('boot', 'repeat', 'resample', 'n_')):
            return v26_bootstrap_mean(values, V26_BOOTSTRAPS)
        if len(params) == 2:
            return v26_bootstrap_mean(values, seed)
        return v26_bootstrap_mean(values)
    except Exception:
        rng = np.random.default_rng(seed)
        draws = rng.choice(values, size=(V26_BOOTSTRAPS, len(values)),
                           replace=True).mean(axis=1)
        return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


print('v26_bootstrap_mean signature:', inspect.signature(v26_bootstrap_mean))
_probe = v28_ci(np.random.default_rng(0).normal(7.5, 1.0, 200), SEED)
print(f'v28_ci probe on synthetic data (expect ~[7.36, 7.64]): '
      f'[{_probe[0]:.3f}, {_probe[1]:.3f}]')
assert np.isfinite(_probe).all() and _probe[0] < _probe[1], 'v28_ci is not returning a CI'


def v28_calibration_dispersion(anchor_readings):
    """Dispersion of the k initial calibration readings, per participant.

    Mirrors the selection logic in v26_build_calibration_dose exactly so the
    features describe the readings that dose actually averaged.
    """
    source = aurora_features.copy()
    for c in ['sbp', 'dbp']:
        source[c] = pd.to_numeric(source[c], errors='coerce')
    source['_parsed_time'] = pd.to_datetime(source['date_time'], errors='coerce')
    source['_is_initial_calibration'] = (
        source['phase'].astype(str).str.lower().eq('initial')
        & source['measurement'].astype(str).str.contains(
            V26_CAL_PATTERN, case=False, regex=True, na=False))
    rows = []
    for pid, group in source.dropna(subset=['sbp', 'dbp']).groupby('pid'):
        group = group.sort_values(['_parsed_time', '_source_order'], na_position='last')
        cal = group[group._is_initial_calibration]
        if len(cal) < anchor_readings:
            continue
        sel = cal.head(anchor_readings)
        entry = {'subject_id': str(pid), 'cal_n': float(anchor_readings)}
        for t in ['sbp', 'dbp']:
            v = sel[t].to_numpy(float)
            entry['cal_sd_' + t] = float(np.std(v, ddof=1)) if len(v) > 1 else np.nan
            entry['cal_range_' + t] = float(v.max() - v.min()) if len(v) > 1 else np.nan
            entry['cal_trend_' + t] = float(v[-1] - v[0]) if len(v) > 1 else np.nan
        if anchor_readings > 1:
            entry['cal_sd_mean'] = float(np.nanmean([entry['cal_sd_sbp'],
                                                     entry['cal_sd_dbp']]))
        else:
            entry['cal_sd_mean'] = np.nan
        rows.append(entry)
    return pd.DataFrame(rows)


CAL_DISPERSION = ['cal_n', 'cal_sd_mean',
                  'cal_sd_sbp', 'cal_range_sbp', 'cal_trend_sbp',
                  'cal_sd_dbp', 'cal_range_dbp', 'cal_trend_dbp']

# attach to every dose frame, using that dose's own readings
v28_dose_frames = {}
for dose, frame in v26_dose_frames.items():
    disp = v28_calibration_dispersion(dose)
    merged = frame.merge(disp, on='subject_id', how='left')
    merged['anchor_pulse_pressure'] = merged.baseline_sbp - merged.baseline_dbp
    merged['anchor_map'] = merged.baseline_dbp + (merged.baseline_sbp
                                                  - merged.baseline_dbp) / 3
    v28_dose_frames[dose] = merged
    if dose == 3:
        print(f'\ndose 3 dispersion coverage: '
              f'{merged.cal_sd_sbp.notna().mean():.1%} of rows')
        print('dispersion summary (participant level):')
        display(merged.drop_duplicates('subject_id')[
            [c for c in CAL_DISPERSION if merged[c].notna().any()]].describe().T.round(3))

# does lability actually predict drift? a direct check before any modelling
_chk = v28_dose_frames[3]
_chk = _chk[~_chk.v26_is_calibration_labeled]
print('\n=== Does calibration dispersion predict subsequent drift? ===')
for target in ['sbp', 'dbp']:
    per = (_chk.assign(_abs=np.abs(_chk['delta_' + target]))
               .groupby('subject_id')
               .agg(mean_abs_delta=('_abs', 'mean'),
                    cal_sd=('cal_sd_' + target, 'first'))).dropna()
    if len(per) > 10:
        r = float(np.corrcoef(per.cal_sd, per.mean_abs_delta)[0, 1])
        q = pd.qcut(per.cal_sd, 4, labels=['Q1 stable', 'Q2', 'Q3', 'Q4 labile'],
                    duplicates='drop')
        print(f'{target.upper()}: corr(cal_sd, mean |delta|) = {r:+.3f} '
              f'over {len(per)} participants')
        display(per.groupby(q, observed=True).mean_abs_delta.agg(['size', 'mean']).round(3))

# ---------------------------------------------------------------- arms -----
CROSS_TARGET = {'sbp': ['baseline_dbp', 'uci_prior_delta_dbp',
                        'anchor_pulse_pressure', 'anchor_map'],
                'dbp': ['baseline_sbp', 'uci_prior_delta_sbp',
                        'anchor_pulse_pressure', 'anchor_map']}
V28_ARMS = ['A_v26_primary', 'B_plus_dispersion', 'C_plus_crosstarget', 'D_both']

v28_frame = v28_dose_frames[3]
v28_frame = v28_frame[~v28_frame.v26_is_calibration_labeled]
v28_frame = v28_frame.dropna(subset=['fst_band']).reset_index(drop=True)
BASE_CONTEXT = V26_CONTEXT_SETS['full_historical_context']
print(f'\nCV frame: {v28_frame.subject_id.nunique()} participants, {len(v28_frame)} rows')


def v28_extra(arm, target):
    extra = []
    if arm in ('B_plus_dispersion', 'D_both'):
        extra += CAL_DISPERSION
    if arm in ('C_plus_crosstarget', 'D_both'):
        extra += CROSS_TARGET[target]
    return [f for f in dict.fromkeys(extra)
            if f in v28_frame.columns and v28_frame[f].notna().any()
            and v28_frame[f].nunique(dropna=True) > 1]


for arm in V28_ARMS:
    print(f'{arm}: +{len(v28_extra(arm, "sbp"))} SBP extras, '
          f'+{len(v28_extra(arm, "dbp"))} DBP extras')

v28_rows = []
for repeat in range(V26_REPEATS):
    for fold, train, test in v26_participant_folds(v28_frame, V26_OUTER_SPLITS, repeat):
        for target in ['sbp', 'dbp']:
            core = v26_core_features(train, target)
            retention = (test.assign(_ae=np.abs(test['baseline_' + target] - test[target]))
                             .groupby('subject_id')._ae.mean())
            for arm in V28_ARMS:
                feats = core + BASE_CONTEXT + v28_extra(arm, target)
                per, used, _, _ = v21_fit_and_score(train, test, target, feats,
                                                    'squared_error')
                for sid, mae in per.items():
                    v28_rows.append({'repeat': repeat + 1, 'fold': fold,
                                     'target': target, 'arm': arm,
                                     'subject_id': sid, 'mae': float(mae),
                                     'retention': float(retention.get(sid, np.nan))})
        print(f'repeat {repeat+1}/{V26_REPEATS} fold {fold}/{V26_OUTER_SPLITS} done')

v28_errors = pd.DataFrame(v28_rows)
v28_subject = (v28_errors.groupby(['target', 'arm', 'subject_id'])
               .agg(mae=('mae', 'mean'), retention=('retention', 'mean')).reset_index())

summary = []
for (target, arm), g in v28_subject.groupby(['target', 'arm']):
    lo, hi = v28_ci(g.mae.to_numpy(), SEED + 28100)
    summary.append({'target': target, 'arm': arm,
                    'participants': len(g), 'MAE': float(g.mae.mean()),
                    'ci_low': lo, 'ci_high': hi,
                    'retention': float(g.retention.mean()),
                    'gain_vs_retention': float((g.retention - g.mae).mean())})
v28_summary = pd.DataFrame(summary)
for target in ['sbp', 'dbp']:
    print(f'\n=== {target.upper()} repeated participant CV ===')
    display(v28_summary[v28_summary.target == target]
            .drop(columns='target').set_index('arm').round(3))

# paired bootstrap against arm A
print('\n=== Paired gain over the V26 primary arm ===')
paired = []
for target in ['sbp', 'dbp']:
    base = v28_subject[(v28_subject.target == target) &
                       (v28_subject.arm == 'A_v26_primary')].set_index('subject_id').mae
    for arm in V28_ARMS[1:]:
        other = v28_subject[(v28_subject.target == target) &
                            (v28_subject.arm == arm)].set_index('subject_id').mae
        common = base.index.intersection(other.index)
        d = (base.loc[common] - other.loc[common]).to_numpy(float)
        lo, hi = v28_ci(d, SEED + 28200)
        paired.append({'target': target, 'arm': arm,
                       'gain_mmhg': float(d.mean()), 'ci_low': lo, 'ci_high': hi,
                       'supported': bool(np.isfinite(lo) and lo > 0),
                       'fraction_helped': float((d > 0).mean()),
                       'participants': len(common)})
v28_paired = pd.DataFrame(paired)
display(v28_paired.round(3))

v28_summary.to_csv(V28_RESULTS / 'v28_summary.csv', index=False)
v28_paired.to_csv(V28_RESULTS / 'v28_paired.csv', index=False)
v28_subject.to_csv(V28_RESULTS / 'v28_subject_level.csv', index=False)

print("""
Reading this
------------
The correlation check above is the honest early indicator. If corr(cal_sd,
mean |delta|) is near zero, calibration dispersion does not measure lability in
this cohort and arm B will not help however it is modelled -- read that before
the arms.

supported = the paired CI excludes zero. fraction_helped near 0.50 is a coin
flip whatever the mean says; that is how the absolute-error loss failed three
times.

If B is supported, it also gives a mechanism for Table II: three readings beat
one not only by averaging out cuff noise but by MEASURING the participant's
lability, which one reading cannot do at all. Averaging alone cuts anchor noise
by at most sqrt(3), and Table III puts cuff noise at 2.6% of the delta SD, so
averaging cannot account for the full 0.733 mmHg gain on its own.

If D is no better than the best of B and C, the two blocks are redundant --
report the simpler winning arm rather than the stack.
""")

## Reading V28

**Read the correlation check first.** Before any arm, the cell reports
corr(`cal_sd`, mean |Δ|) across participants and a quartile table. If that is
near zero, calibration dispersion does not measure lability in this cohort and
arm B will not help however it is modelled. That check is cheap and it tells you
whether the rest of the table is worth reading.

**If B is supported**, it also gives Section V-B a better story. Three readings
currently beat one by 0.733 mmHg SBP, explained only as averaging. But averaging
reduces anchor noise by √3 at most, and cuff noise is just 2.6% of the delta SD
— so averaging alone cannot account for the whole gain. Dispersion offers the
missing mechanism: three readings **measure the participant's lability**, which
one reading cannot do at all. That is worth a sentence in the paper regardless
of arm B's size.

**If D is no better than the best of B and C**, the blocks are redundant —
report the simpler winning arm, not the stack.

**If nothing clears zero**, that is still a clean negative for Table X, and it
tightens the argument that the remaining 6.36 mmHg of headroom is not reachable
from anchor-side information.

### On expectations

I would budget tenths of a mmHg, as with every other lever tested here. The one
reason to think dispersion could do better is that it is the first feature added
that targets *between-participant variation in drift magnitude* rather than
waveform morphology — a different axis from everything in the ablation log.

### Still worth doing separately, from the paper review

- Quantify the AVCT 2.05 vs PC-AVCT 7.526 gap by reporting within-subject
  reference BP SD for both protocols side by side. This is the paper's biggest
  vulnerability and one sentence with two numbers fixes it.
- Add a ridge baseline and a learning curve over participant count.
- Drop the gate: activation is 1.0 in every configuration, so it never fires.
- Update Table III, which still quotes the 8.11 development figure against a
  7.526 primary result.

## V28 (continued): raw AVCT vs. pigmentation-corrected (PC) channel

This arm was not given its own markdown header in the original notebook --
it follows directly from "Reading V28" above but tests a different question
than the dispersion/cross-target arms just discussed: does routing the PPG
features through the pigmentation-corrected (`PC`) channel, with or without
explicit skin-tone interaction terms, beat the raw (tone-blind) channel?

Uses V26's strict endpoint (three-reading session anchor, real Fitzpatrick
labels, no calibration-labelled rows). Three arms: raw AVCT (no skin
correction), PC-corrected channel, and PC-corrected channel plus explicit
skin-aware interaction terms (`v22_add_skin_features`).

This is the sixth independent ablation arm in this project testing whether
tone-aware processing beats tone-blind -- all six have come back "no clear
difference" or worse. Treat the "PC-AVCT" framing question as settled by this
point rather than something still open to re-test.


In [ ]:
# ============================================================================
# V28: PAIRED RAW AVCT VS PC-AVCT SKIN-CORRECTION ABLATION
# ============================================================================
V28_RESULTS = OUT_DIR / "avct_v28_pc_skin_correction_ablation"
V28_RESULTS.mkdir(parents=True, exist_ok=True)

V28_OUTER_SPLITS = V26_OUTER_SPLITS
V28_REPEATS = V26_REPEATS
V28_TOP_K = V26_TOP_K
V28_BOOTSTRAPS = 5000

# Use V26's strict, deployment-oriented endpoint: three calibration readings,
# session-start anchoring, real Fitzpatrick labels, and no calibration-labelled
# rows in fitting or scoring.
v28_frame = v26_cv_frame.copy().replace([np.inf, -np.inf], np.nan)
v28_frame, V28_SKIN_TERMS = v22_add_skin_features(v28_frame)

# Keep context identical and forced in every arm. This is the primary V26 model
# context, minus the calibration flag (already absent by construction).
V28_CONTEXT = list(V26_CONTEXT_SETS["full_historical_context"])
V28_SHARED_NON_PPG = [
    feature for feature in [
        "elapsed_seconds", "log_elapsed_days",
        "session_elapsed_seconds", "log_session_elapsed_days",
    ]
    if feature in v28_frame.columns
]

V28_ARMS = {
    "avct_raw_no_skin_correction": {
        "ppg_pool": DELTA_RAW + RELATIVE_RAW,
        "forced": [],
    },
    "pc_avct_corrected_channel": {
        "ppg_pool": DELTA_PC + RELATIVE_PC,
        "forced": [],
    },
    "pc_avct_full_skin_aware": {
        "ppg_pool": DELTA_PC + RELATIVE_PC,
        "forced": V28_SKIN_TERMS,
    },
}

required_columns = {
    "subject_id", "fst_band", "fitzpatrick_scale", "sbp", "dbp",
    "baseline_sbp", "baseline_dbp", "delta_sbp", "delta_dbp",
}
missing_required = sorted(required_columns - set(v28_frame.columns))
if missing_required:
    raise RuntimeError(f"V28 missing required columns: {missing_required}")
if not any(feature in v28_frame.columns for feature in DELTA_RAW):
    raise RuntimeError("No raw AVCT delta features are available.")
if not any(feature in v28_frame.columns for feature in DELTA_PC):
    raise RuntimeError("No pigmentation-corrected AVCT delta features are available.")

print(
    f"V28 strict cohort: {v28_frame.subject_id.nunique()} participants, "
    f"{len(v28_frame)} scored rows"
)
print(f"Repeated participant CV: {V28_REPEATS} x {V28_OUTER_SPLITS} folds")
print("Forced context:", V28_CONTEXT)
print("Explicit skin terms:", V28_SKIN_TERMS)
display(
    v28_frame[["subject_id", "fst_band"]]
    .drop_duplicates("subject_id")["fst_band"]
    .value_counts()
    .rename("participants")
    .to_frame()
)


def v28_select_features(train, target, arm_spec):
    """Rank only within the outer-training participants."""
    base = "baseline_" + target
    pool = [
        feature for feature in dict.fromkeys(
            arm_spec["ppg_pool"] + V28_SHARED_NON_PPG
        )
        if feature in train.columns
        and train[feature].notna().any()
        and train[feature].nunique(dropna=True) > 1
    ]
    if not pool:
        raise RuntimeError(f"No eligible V28 features for {target}.")
    fit = train.dropna(subset=["delta_" + target])
    ranking = abl_rank(
        fit, pool, "delta_" + target, "fst_band", False
    )
    ranked = ranking["feature"].head(V28_TOP_K).tolist()
    forced = [
        feature for feature in arm_spec["forced"] + V28_CONTEXT
        if feature in train.columns and train[feature].notna().any()
        and train[feature].nunique(dropna=True) > 1
    ]
    return list(dict.fromkeys([base] + ranked + forced))


v28_error_rows = []
v28_fold_rows = []
v28_feature_rows = []

for repeat in range(V28_REPEATS):
    # This generator gives every arm the exact same train/test participants.
    for fold, train, test in v26_participant_folds(
        v28_frame, V28_OUTER_SPLITS, repeat
    ):
        for target in ["sbp", "dbp"]:
            for arm, arm_spec in V28_ARMS.items():
                features = v28_select_features(train, target, arm_spec)
                per_subject, used, gate, activation = v21_fit_and_score(
                    train, test, target, features, "squared_error"
                )
                band_lookup = (
                    test[["subject_id", "fst_band"]]
                    .drop_duplicates("subject_id")
                    .set_index("subject_id")["fst_band"]
                )
                for subject_id, mae in per_subject.items():
                    v28_error_rows.append({
                        "repeat": repeat + 1,
                        "outer_fold": fold,
                        "target": target,
                        "arm": arm,
                        "subject_id": subject_id,
                        "fst_band": band_lookup.get(subject_id, np.nan),
                        "mae": float(mae),
                    })
                v28_fold_rows.append({
                    "repeat": repeat + 1,
                    "outer_fold": fold,
                    "target": target,
                    "arm": arm,
                    "participants": len(per_subject),
                    "rows": len(test),
                    "participant_MAE": float(per_subject.mean()),
                    "feature_count": len(used),
                    "gate_activation": activation,
                })
                v28_feature_rows.append({
                    "repeat": repeat + 1,
                    "outer_fold": fold,
                    "target": target,
                    "arm": arm,
                    "features": "|".join(used),
                    "gate": repr(gate),
                })
        print(
            f"V28 repeat {repeat + 1}/{V28_REPEATS}, "
            f"fold {fold}/{V28_OUTER_SPLITS} complete"
        )

v28_errors = pd.DataFrame(v28_error_rows)
v28_folds = pd.DataFrame(v28_fold_rows)
v28_features = pd.DataFrame(v28_feature_rows)

# Average the repeated out-of-fold errors once per participant before inference.
v28_participant = (
    v28_errors.groupby(
        ["target", "arm", "subject_id", "fst_band"], as_index=False
    )
    .agg(mae=("mae", "mean"), repeats=("repeat", "nunique"))
)


def v28_bootstrap_mean(values, seed):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.empty(V28_BOOTSTRAPS, float)
    for index in range(V28_BOOTSTRAPS):
        boot[index] = rng.choice(values, len(values), replace=True).mean()
    return tuple(np.quantile(boot, [0.025, 0.975]))


# Absolute MAE for each arm.
summary_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    for arm_index, arm in enumerate(V28_ARMS):
        group = v28_participant[
            (v28_participant.target == target) & (v28_participant.arm == arm)
        ]
        low, high = v28_bootstrap_mean(
            group.mae, SEED + 28000 + 100 * target_index + arm_index
        )
        summary_rows.append({
            "target": target,
            "arm": arm,
            "participants": group.subject_id.nunique(),
            "participant_MAE": group.mae.mean(),
            "MAE_CI_low": low,
            "MAE_CI_high": high,
        })
v28_summary = pd.DataFrame(summary_rows)


def v28_paired_comparison(target, comparator, scope, seed):
    subset = v28_participant[v28_participant.target == target]
    if scope != "overall":
        subset = subset[subset.fst_band == scope]
    pivot = subset.pivot(index="subject_id", columns="arm", values="mae")
    aligned = pivot[["avct_raw_no_skin_correction", comparator]].dropna()
    # Positive means the PC-AVCT arm reduced error.
    difference = (
        aligned["avct_raw_no_skin_correction"] - aligned[comparator]
    )
    low, high = v28_bootstrap_mean(difference, seed)
    if low > 0:
        conclusion = "supported improvement"
    elif high < 0:
        conclusion = "supported harm"
    else:
        conclusion = "no clear difference"
    return {
        "target": target,
        "comparison": comparator + " vs raw AVCT",
        "scope": scope,
        "participants": len(aligned),
        "raw_AVCT_MAE": aligned["avct_raw_no_skin_correction"].mean(),
        "PC_AVCT_MAE": aligned[comparator].mean(),
        "error_reduction_mmhg": difference.mean(),
        "error_reduction_percent": (
            100 * difference.mean()
            / aligned["avct_raw_no_skin_correction"].mean()
        ),
        "CI_low": low,
        "CI_high": high,
        "fraction_participants_helped": (difference > 0).mean(),
        "conclusion": conclusion,
    }


contrast_rows = []
comparators = [
    "pc_avct_corrected_channel",
    "pc_avct_full_skin_aware",
]
scopes = ["overall"] + sorted(v28_participant.fst_band.dropna().unique())
for target_index, target in enumerate(["sbp", "dbp"]):
    for comparator_index, comparator in enumerate(comparators):
        for scope_index, scope in enumerate(scopes):
            contrast_rows.append(v28_paired_comparison(
                target,
                comparator,
                scope,
                SEED + 28500 + 1000 * target_index
                + 100 * comparator_index + scope_index,
            ))
v28_paired_contrasts = pd.DataFrame(contrast_rows)

print("\n=== V28 participant-level MAE ===")
display(v28_summary.round(3))
print("\n=== V28 paired error differences ===")
print("Positive error reduction favors PC-AVCT; negative favors raw AVCT.")
display(v28_paired_contrasts.round(3))

print("\n=== Simple decision ===")
display(
    v28_paired_contrasts[v28_paired_contrasts.scope == "overall"][[
        "target", "comparison", "raw_AVCT_MAE", "PC_AVCT_MAE",
        "error_reduction_mmhg", "CI_low", "CI_high", "conclusion",
    ]].round(3)
)
print(
    "Keep a skin-correction claim only if the overall paired interval is "
    "entirely above zero. Treat Fitzpatrick V-VI results as exploratory if "
    "that subgroup remains small."
)

# Confidence-interval figure for the two overall comparisons.
overall_plot = v28_paired_contrasts[
    v28_paired_contrasts.scope == "overall"
].copy()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
for axis, target in zip(axes, ["sbp", "dbp"]):
    shown = overall_plot[overall_plot.target == target].reset_index(drop=True)
    y = np.arange(len(shown))
    x = shown.error_reduction_mmhg.to_numpy(float)
    xerr = np.vstack([
        x - shown.CI_low.to_numpy(float),
        shown.CI_high.to_numpy(float) - x,
    ])
    axis.errorbar(x, y, xerr=xerr, fmt="o", capsize=4, color="#0173B2")
    axis.axvline(0, color="#333333", linestyle="--", linewidth=1)
    axis.set_yticks(y)
    axis.set_yticklabels([
        "Corrected channel",
        "Corrected + skin terms",
    ])
    axis.set_title(target.upper())
    axis.set_xlabel("MAE reduction vs raw AVCT (mmHg)")
    axis.grid(axis="x", alpha=0.25)
fig.suptitle("Does PC-AVCT skin correction reduce error?\n95% participant-bootstrap CI")
fig.tight_layout()
fig.savefig(V28_RESULTS / "v28_skin_correction_error_difference.png", dpi=300)
plt.show()

v28_errors.to_csv(V28_RESULTS / "v28_repeated_cv_errors.csv", index=False)
v28_folds.to_csv(V28_RESULTS / "v28_repeated_cv_folds.csv", index=False)
v28_features.to_csv(V28_RESULTS / "v28_selected_features_and_gates.csv", index=False)
v28_participant.to_csv(V28_RESULTS / "v28_participant_mae.csv", index=False)
v28_summary.to_csv(V28_RESULTS / "v28_arm_summary.csv", index=False)
v28_paired_contrasts.to_csv(
    V28_RESULTS / "v28_paired_skin_correction_contrasts.csv", index=False
)
print("Saved V28 outputs to:", V28_RESULTS)


---

# V29: physiologic targets, nested ensemble, temporal state, and uncertainty

This experiment preserves the V26 three-reading session anchor, strict removal
of calibration-labelled rows, participant-separated outer folds, and primary
context set.

Within each outer training partition, three additional participant-level folds
are used to choose ensemble weights, temporal smoothing strength, and uncertainty
thresholds. The outer test participants never influence those choices.

The primary table is `v29_contrasts`. Positive paired improvement means the
candidate reduced participant-level MAE relative to the current raw-HGB model.
The accuracy–coverage table is secondary: lower MAE at reduced coverage means
the system can identify readings that should be repeated, not that it improved
the universal 100% coverage endpoint.


In [ ]:
# ============================================================================
# V29: MAP/PP TARGETS + NESTED ENSEMBLE + CAUSAL TEMPORAL MODEL + UNCERTAINTY
# ============================================================================
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge

V29_RESULTS = OUT_DIR / "avct_v29_target_temporal_ensemble"
V29_RESULTS.mkdir(parents=True, exist_ok=True)

V29_OUTER_SPLITS = V26_OUTER_SPLITS
V29_REPEATS = V26_REPEATS
V29_INNER_SPLITS = 3
V29_TOP_K = V26_TOP_K
V29_BOOTSTRAPS = 5000
V29_WEIGHT_RESOLUTION = 10       # nonnegative ensemble weights in steps of 0.10
V29_TEMPORAL_ALPHA_GRID = [0.35, 0.50, 0.65, 0.80, 0.90, 1.00]
V29_COVERAGES = [1.00, 0.90, 0.80, 0.70]


def v29_add_targets(frame):
    """Add physiologically coupled MAP/pulse-pressure targets and baselines."""
    out = frame.copy().replace([np.inf, -np.inf], np.nan)
    out["map"] = (out["sbp"] + 2.0 * out["dbp"]) / 3.0
    out["pulse_pressure"] = out["sbp"] - out["dbp"]
    out["baseline_map"] = (
        out["baseline_sbp"] + 2.0 * out["baseline_dbp"]
    ) / 3.0
    out["baseline_pulse_pressure"] = (
        out["baseline_sbp"] - out["baseline_dbp"]
    )
    out["delta_map"] = out["map"] - out["baseline_map"]
    out["delta_pulse_pressure"] = (
        out["pulse_pressure"] - out["baseline_pulse_pressure"]
    )
    if {"uci_prior_delta_sbp", "uci_prior_delta_dbp"}.issubset(out.columns):
        out["uci_prior_delta_map"] = (
            out["uci_prior_delta_sbp"] + 2.0 * out["uci_prior_delta_dbp"]
        ) / 3.0
        out["uci_prior_delta_pulse_pressure"] = (
            out["uci_prior_delta_sbp"] - out["uci_prior_delta_dbp"]
        )
    return out


v29_frame = v29_add_targets(v26_cv_frame)
V29_CONTEXT = list(V26_CONTEXT_SETS["full_historical_context"])
V29_TIME_FEATURES = [
    feature for feature in [
        "elapsed_seconds", "log_elapsed_days",
        "session_elapsed_seconds", "log_session_elapsed_days",
    ]
    if feature in v29_frame.columns
]

# Calibration dispersion failed as a direct prediction block in V28, but it can
# still be tested honestly as an uncertainty feature. It never enters the BP model.
_v29_calibration = aurora_features[
    aurora_features["phase"].astype(str).str.lower().eq("initial")
    & aurora_features["measurement"].astype(str).str.contains(
        r"^\s*calibration\s+start", case=False, regex=True, na=False
    )
].copy()
_v29_calibration["subject_id"] = _v29_calibration["pid"].astype(str)
_v29_dispersion = _v29_calibration.groupby("subject_id").agg(
    cal_sd_sbp=("sbp", "std"),
    cal_sd_dbp=("dbp", "std"),
    cal_range_sbp=("sbp", lambda values: values.max() - values.min()),
    cal_range_dbp=("dbp", lambda values: values.max() - values.min()),
).reset_index()
# Make rerunning the cell idempotent if an upstream experimental cell already
# attached columns with these names.
v29_frame = v29_frame.drop(
    columns=[column for column in _v29_dispersion.columns if column != "subject_id"
             and column in v29_frame.columns],
    errors="ignore",
)
v29_frame = v29_frame.merge(
    _v29_dispersion, on="subject_id", how="left", validate="many_to_one"
)

required = {
    "subject_id", "fst_band", "_source_order", "sbp", "dbp",
    "baseline_sbp", "baseline_dbp", "delta_sbp", "delta_dbp",
    "map", "pulse_pressure", "delta_map", "delta_pulse_pressure",
}
missing = sorted(required - set(v29_frame.columns))
if missing:
    raise RuntimeError(f"V29 missing required columns: {missing}")

print(
    f"V29 strict cohort: {v29_frame.subject_id.nunique()} participants, "
    f"{len(v29_frame)} rows"
)
print(
    f"Design: {V29_REPEATS} x {V29_OUTER_SPLITS} outer participant CV, "
    f"{V29_INNER_SPLITS}-fold participant CV for weights and smoothing"
)


def v29_participant_splits(frame, n_splits, seed):
    people = (
        frame[["subject_id", "fst_band"]]
        .drop_duplicates("subject_id")
        .sort_values("subject_id")
        .reset_index(drop=True)
    )
    splitter = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=seed
    )
    for fold, (train_index, test_index) in enumerate(
        splitter.split(people.subject_id, people.fst_band), 1
    ):
        train_ids = set(people.iloc[train_index].subject_id)
        test_ids = set(people.iloc[test_index].subject_id)
        train = frame[frame.subject_id.isin(train_ids)].copy()
        test = frame[frame.subject_id.isin(test_ids)].copy()
        assert not set(train.subject_id) & set(test.subject_id)
        yield fold, train, test


def v29_rank_features(train, target_column, baseline_column, prior_column=None):
    pool = DELTA_RAW + RELATIVE_RAW + V29_TIME_FEATURES
    if prior_column and prior_column in train.columns:
        pool = pool + [prior_column]
    pool = [
        feature for feature in dict.fromkeys(pool)
        if feature in train.columns
        and train[feature].notna().any()
        and train[feature].nunique(dropna=True) > 1
    ]
    fit = train.dropna(subset=[target_column, baseline_column]).copy()
    if not pool:
        raise RuntimeError(f"No candidate features for {target_column}")
    values = normalized_conditional_mi(
        fit, pool, target_column,
        [baseline_column, "log_elapsed_days"],
    )
    ranking = (
        pd.DataFrame({"feature": pool, "score": values})
        .sort_values(["score", "feature"], ascending=[False, True])
    )
    selected = ranking.feature.head(V29_TOP_K).tolist()
    forced = [
        feature for feature in V29_CONTEXT
        if feature in fit.columns
        and fit[feature].notna().any()
        and fit[feature].nunique(dropna=True) > 1
    ]
    return list(dict.fromkeys([baseline_column] + selected + forced))


def v29_model(family, seed):
    if family == "hgb":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingRegressor(
                # Match the frozen V26 baseline exactly. HistGradientBoosting
                # uses this seed for its early-stopping validation partition.
                max_iter=300, loss="squared_error", random_state=SEED,
                **V12_PARAMS,
            ),
        )
    if family == "extra_trees":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            ExtraTreesRegressor(
                n_estimators=250,
                min_samples_leaf=5,
                max_features=0.70,
                n_jobs=-1,
                random_state=seed,
            ),
        )
    if family == "ridge":
        return make_pipeline(
            SimpleImputer(strategy="median"),
            RobustScaler(),
            Ridge(alpha=10.0),
        )
    raise ValueError(f"Unknown model family: {family}")


def v29_fit_predict_delta(
    train, test, target_column, baseline_column, features, family, seed
):
    fit = train.dropna(subset=[target_column, baseline_column]).copy()
    model = v29_model(family, seed)
    weights = abl_weights(fit, "fst_band", False)
    final_step = model.steps[-1][0]
    model.fit(
        fit[features], fit[target_column],
        **{final_step + "__sample_weight": weights},
    )
    # Retain V26's training-only shrinkage/gate procedure so raw HGB is a true
    # reproduction of the current model rather than a subtly changed baseline.
    absolute_target = target_column[len("delta_"):]
    fit_delta = model.predict(fit[features])
    gate = abl_fit_gate(
        fit, absolute_target, baseline_column, fit_delta, "fst_band", False
    )
    test_delta = model.predict(test[features])
    prediction, _ = abl_apply_gate(
        test, baseline_column, test_delta, gate, "fst_band"
    )
    return np.asarray(prediction, float)


def v29_predict_components(train, test, seed):
    """Fit all component models without consulting the supplied test labels."""
    predicted = test.copy().reset_index(drop=True)
    audit = []

    # Three model families using the current raw-AVCT targets.
    for target_index, target in enumerate(["sbp", "dbp"]):
        features = v29_rank_features(
            train,
            "delta_" + target,
            "baseline_" + target,
            "uci_prior_delta_" + target,
        )
        for family_index, family in enumerate(["hgb", "extra_trees", "ridge"]):
            column = f"raw_{family}_{target}"
            predicted[column] = v29_fit_predict_delta(
                train, predicted,
                "delta_" + target, "baseline_" + target,
                features, family,
                seed + 100 * target_index + 10 * family_index,
            )
            audit.append({
                "target": target, "component": column,
                "feature_count": len(features),
                "features": "|".join(features),
            })

    # HGB models on MAP and pulse-pressure changes, reconstructed to SBP/DBP.
    latent_predictions = {}
    latent_specs = [
        ("map", "delta_map", "baseline_map", "uci_prior_delta_map"),
        (
            "pulse_pressure", "delta_pulse_pressure",
            "baseline_pulse_pressure", "uci_prior_delta_pulse_pressure",
        ),
    ]
    for latent_index, (name, target_column, baseline_column, prior) in enumerate(
        latent_specs
    ):
        features = v29_rank_features(
            train, target_column, baseline_column, prior
        )
        latent_predictions[name] = v29_fit_predict_delta(
            train, predicted, target_column, baseline_column,
            features, "hgb", seed + 500 + latent_index,
        )
        audit.append({
            "target": name, "component": "mappp_hgb_" + name,
            "feature_count": len(features),
            "features": "|".join(features),
        })

    predicted["mappp_hgb_sbp"] = (
        latent_predictions["map"]
        + (2.0 / 3.0) * latent_predictions["pulse_pressure"]
    )
    predicted["mappp_hgb_dbp"] = (
        latent_predictions["map"]
        - (1.0 / 3.0) * latent_predictions["pulse_pressure"]
    )
    return predicted, pd.DataFrame(audit)


def v29_component_columns(target):
    return [
        f"raw_hgb_{target}",
        f"raw_extra_trees_{target}",
        f"raw_ridge_{target}",
        f"mappp_hgb_{target}",
    ]


def v29_participant_mae(frame, truth, prediction):
    return (
        frame.assign(_error=np.abs(frame[prediction] - frame[truth]))
        .groupby("subject_id")._error.mean().mean()
    )


def v29_weight_candidates(parts, total):
    if parts == 1:
        yield (total,)
        return
    for first in range(total + 1):
        for remainder in v29_weight_candidates(parts - 1, total - first):
            yield (first,) + remainder


def v29_choose_weights(oof, target):
    columns = v29_component_columns(target)
    matrix = oof[columns].to_numpy(float)
    best = None
    for integers in v29_weight_candidates(len(columns), V29_WEIGHT_RESOLUTION):
        weights = np.asarray(integers, float) / V29_WEIGHT_RESOLUTION
        candidate = matrix @ weights
        score = (
            oof.assign(_error=np.abs(candidate - oof[target]))
            .groupby("subject_id")._error.mean().mean()
        )
        # Prefer the simpler/raw-HGB-heavy solution when scores tie.
        key = (float(score), int(np.count_nonzero(weights)), -weights[0])
        if best is None or key < best[0]:
            best = (key, weights)
    return dict(zip(columns, best[1])), best[0][0]


def v29_blend(frame, target, weights):
    prediction = np.zeros(len(frame), float)
    for column, weight in weights.items():
        prediction += float(weight) * frame[column].to_numpy(float)
    return prediction


def v29_temporal_smooth(frame, values, alpha):
    """Causal smoothing using only present/past model outputs, never a new cuff."""
    working = frame.copy().reset_index(drop=True)
    working["_v29_value"] = np.asarray(values, float)
    output = np.full(len(working), np.nan, float)
    regime_columns = [
        column for column in [
            "phase", "context_activity_post_exercise",
            "context_posture_seated",
        ] if column in working.columns
    ]
    for _, group in working.groupby("subject_id", sort=False):
        group = group.sort_values("_source_order")
        state = None
        previous_regime = None
        for row_index, row in group.iterrows():
            regime = tuple(str(row[column]) for column in regime_columns)
            current = float(row._v29_value)
            if state is None or regime != previous_regime:
                state = current
            else:
                state = float(alpha) * current + (1.0 - float(alpha)) * state
            output[row_index] = state
            previous_regime = regime
    return output


def v29_choose_temporal_alpha(oof, target, ensemble_prediction):
    best = None
    for alpha in V29_TEMPORAL_ALPHA_GRID:
        smoothed = v29_temporal_smooth(oof, ensemble_prediction, alpha)
        score = (
            oof.assign(_error=np.abs(smoothed - oof[target]))
            .groupby("subject_id")._error.mean().mean()
        )
        key = (float(score), -float(alpha))  # tie favors alpha=1/no smoothing
        if best is None or key < best[0]:
            best = (key, float(alpha))
    return best[1], best[0][0]


def v29_uncertainty_features(frame, target, ensemble_prediction):
    components = frame[v29_component_columns(target)].to_numpy(float)
    features = pd.DataFrame(index=frame.index)
    features["model_disagreement"] = np.std(components, axis=1)
    features["model_range"] = np.ptp(components, axis=1)
    features["abs_predicted_change"] = np.abs(
        np.asarray(ensemble_prediction, float)
        - frame["baseline_" + target].to_numpy(float)
    )
    candidates = [
        "elapsed_seconds", "log_elapsed_days",
        "session_elapsed_seconds", "log_session_elapsed_days",
        "context_optical_quality", "context_pressure_quality",
        "context_duration_seconds", "cal_sd_" + target,
        "cal_range_" + target,
    ]
    for column in candidates:
        if column in frame.columns:
            features[column] = pd.to_numeric(frame[column], errors="coerce")
    return features


def v29_fit_uncertainty(oof, test, target, oof_prediction, test_prediction, seed):
    x_train = v29_uncertainty_features(oof, target, oof_prediction)
    x_test = v29_uncertainty_features(test, target, test_prediction)
    error = np.abs(np.asarray(oof_prediction, float) - oof[target].to_numpy(float))
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.04,
            max_leaf_nodes=15,
            l2_regularization=3.0,
            min_samples_leaf=20,
            loss="absolute_error",
            random_state=seed,
        ),
    )
    model.fit(
        x_train, error,
        histgradientboostingregressor__sample_weight=abl_weights(
            oof, "fst_band", False
        ),
    )
    train_score = np.maximum(model.predict(x_train), 0)
    test_score = np.maximum(model.predict(x_test), 0)
    return train_score, test_score, list(x_train.columns)


prediction_rows = []
selection_rows = []
feature_audits = []

for repeat in range(V29_REPEATS):
    outer_seed = SEED + 29000 + 1000 * repeat
    for outer_fold, outer_train, outer_test in v29_participant_splits(
        v29_frame, V29_OUTER_SPLITS, outer_seed
    ):
        # Inner participant OOF predictions are the only data used to choose
        # ensemble weights, temporal alpha, or uncertainty thresholds.
        inner_parts = []
        for inner_fold, inner_train, inner_valid in v29_participant_splits(
            outer_train,
            V29_INNER_SPLITS,
            outer_seed + 100 + outer_fold,
        ):
            inner_prediction, _ = v29_predict_components(
                inner_train,
                inner_valid,
                outer_seed + 10000 * outer_fold + 100 * inner_fold,
            )
            inner_prediction["inner_fold"] = inner_fold
            inner_parts.append(inner_prediction)
        inner_oof = pd.concat(inner_parts, ignore_index=True)
        assert inner_oof.subject_id.nunique() == outer_train.subject_id.nunique()

        outer_prediction, outer_audit = v29_predict_components(
            outer_train,
            outer_test,
            outer_seed + 50000 + outer_fold,
        )
        outer_audit["repeat"] = repeat + 1
        outer_audit["outer_fold"] = outer_fold
        feature_audits.append(outer_audit)

        for target_index, target in enumerate(["sbp", "dbp"]):
            weights, inner_ensemble_mae = v29_choose_weights(inner_oof, target)
            inner_ensemble = v29_blend(inner_oof, target, weights)
            outer_ensemble = v29_blend(outer_prediction, target, weights)

            alpha, inner_temporal_mae = v29_choose_temporal_alpha(
                inner_oof, target, inner_ensemble
            )
            inner_temporal = v29_temporal_smooth(
                inner_oof, inner_ensemble, alpha
            )
            outer_temporal = v29_temporal_smooth(
                outer_prediction, outer_ensemble, alpha
            )

            train_uncertainty, test_uncertainty, uncertainty_features = (
                v29_fit_uncertainty(
                    inner_oof, outer_prediction, target,
                    inner_temporal, outer_temporal,
                    outer_seed + 70000 + 100 * outer_fold + target_index,
                )
            )

            outer_prediction[f"nested_ensemble_{target}"] = outer_ensemble
            outer_prediction[f"temporal_ensemble_{target}"] = outer_temporal
            outer_prediction[f"uncertainty_{target}"] = test_uncertainty

            selection_rows.append({
                "repeat": repeat + 1,
                "outer_fold": outer_fold,
                "target": target,
                "inner_ensemble_MAE": inner_ensemble_mae,
                "temporal_alpha": alpha,
                "inner_temporal_MAE": inner_temporal_mae,
                "uncertainty_features": "|".join(uncertainty_features),
                **{"weight_" + column: weight for column, weight in weights.items()},
            })

            # Coverage thresholds are learned from inner-OOF uncertainty only.
            for coverage in V29_COVERAGES:
                threshold = (
                    np.inf if coverage == 1.0
                    else float(np.quantile(train_uncertainty, coverage))
                )
                outer_prediction[f"keep_{target}_{int(coverage * 100)}"] = (
                    test_uncertainty <= threshold
                )

        outer_prediction["repeat"] = repeat + 1
        outer_prediction["outer_fold"] = outer_fold
        prediction_rows.append(outer_prediction)
        print(
            f"V29 repeat {repeat + 1}/{V29_REPEATS}, "
            f"outer fold {outer_fold}/{V29_OUTER_SPLITS} complete"
        )

v29_predictions = pd.concat(prediction_rows, ignore_index=True)
v29_selection = pd.DataFrame(selection_rows)
v29_feature_audit = pd.concat(feature_audits, ignore_index=True)

V29_ARMS = {
    "current_raw_HGB": "raw_hgb_{target}",
    "raw_ExtraTrees": "raw_extra_trees_{target}",
    "raw_Ridge": "raw_ridge_{target}",
    "MAP_PP_HGB": "mappp_hgb_{target}",
    "nested_ensemble": "nested_ensemble_{target}",
    "temporal_ensemble": "temporal_ensemble_{target}",
}

error_rows = []
for target in ["sbp", "dbp"]:
    for arm, pattern in V29_ARMS.items():
        prediction_column = pattern.format(target=target)
        working = v29_predictions.assign(
            _error=np.abs(v29_predictions[prediction_column] - v29_predictions[target])
        )
        per_person = (
            working.groupby("subject_id", as_index=False)
            .agg(mae=("_error", "mean"), repeats=("repeat", "nunique"))
        )
        for row in per_person.itertuples(index=False):
            error_rows.append({
                "target": target, "arm": arm,
                "subject_id": row.subject_id, "mae": row.mae,
                "repeats": row.repeats,
            })
v29_participant_errors = pd.DataFrame(error_rows)


def v29_bootstrap(values, seed):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    boot = np.empty(V29_BOOTSTRAPS, float)
    for index in range(V29_BOOTSTRAPS):
        boot[index] = rng.choice(values, len(values), replace=True).mean()
    return tuple(np.quantile(boot, [0.025, 0.975]))


summary_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    for arm_index, arm in enumerate(V29_ARMS):
        group = v29_participant_errors[
            (v29_participant_errors.target == target)
            & (v29_participant_errors.arm == arm)
        ]
        low, high = v29_bootstrap(
            group.mae, SEED + 29500 + 100 * target_index + arm_index
        )
        summary_rows.append({
            "target": target, "arm": arm,
            "participants": group.subject_id.nunique(),
            "participant_MAE": group.mae.mean(),
            "CI_low": low, "CI_high": high,
        })
v29_summary = pd.DataFrame(summary_rows)

contrast_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    pivot = v29_participant_errors[
        v29_participant_errors.target == target
    ].pivot(index="subject_id", columns="arm", values="mae")
    for arm_index, arm in enumerate(
        [name for name in V29_ARMS if name != "current_raw_HGB"]
    ):
        aligned = pivot[["current_raw_HGB", arm]].dropna()
        improvement = aligned.current_raw_HGB - aligned[arm]
        low, high = v29_bootstrap(
            improvement,
            SEED + 29600 + 100 * target_index + arm_index,
        )
        if low > 0:
            conclusion = "supported improvement"
        elif high < 0:
            conclusion = "supported harm"
        else:
            conclusion = "no clear difference"
        contrast_rows.append({
            "target": target, "candidate": arm,
            "participants": len(aligned),
            "current_MAE": aligned.current_raw_HGB.mean(),
            "candidate_MAE": aligned[arm].mean(),
            "improvement_mmhg": improvement.mean(),
            "CI_low": low, "CI_high": high,
            "fraction_helped": (improvement > 0).mean(),
            "conclusion": conclusion,
        })
v29_contrasts = pd.DataFrame(contrast_rows)

# Uncertainty/coverage analysis. Lower coverage is not a universal-model MAE;
# it is the conditional error after asking the user to repeat uncertain readings.
coverage_person_rows = []
coverage_measurement_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    prediction = v29_predictions[f"temporal_ensemble_{target}"]
    errors = np.abs(prediction - v29_predictions[target])
    for coverage in V29_COVERAGES:
        keep = v29_predictions[f"keep_{target}_{int(coverage * 100)}"].astype(bool)
        selected = v29_predictions.loc[keep, ["subject_id", "repeat"]].copy()
        selected["error"] = errors[keep].to_numpy(float)
        per_person = selected.groupby("subject_id").error.mean()
        low, high = v29_bootstrap(
            per_person, SEED + 29700 + 100 * target_index + int(coverage * 10)
        )
        coverage_measurement_rows.append({
            "target": target,
            "requested_coverage": coverage,
            "actual_measurement_coverage": keep.mean(),
            "measurements_retained": int(keep.sum()),
            "participants_with_prediction": per_person.index.nunique(),
            "measurement_MAE": errors[keep].mean(),
            "participant_MAE": per_person.mean(),
            "participant_MAE_CI_low": low,
            "participant_MAE_CI_high": high,
        })
        for subject_id, mae in per_person.items():
            coverage_person_rows.append({
                "target": target, "requested_coverage": coverage,
                "subject_id": subject_id, "mae": mae,
            })
v29_coverage = pd.DataFrame(coverage_measurement_rows)
v29_coverage_participant = pd.DataFrame(coverage_person_rows)

print("\n=== V29 repeated participant-CV MAE ===")
display(v29_summary.round(3))
print("\n=== Paired comparison with the current raw-HGB model ===")
print("Positive improvement favors the candidate.")
display(v29_contrasts.round(3))
print("\n=== Uncertainty-based repeat-measurement policy ===")
display(v29_coverage.round(3))
print("\n=== Inner-CV selections ===")
display(
    v29_selection.groupby("target").agg(
        mean_temporal_alpha=("temporal_alpha", "mean"),
        temporal_alpha_SD=("temporal_alpha", "std"),
        mean_inner_ensemble_MAE=("inner_ensemble_MAE", "mean"),
        mean_inner_temporal_MAE=("inner_temporal_MAE", "mean"),
    ).round(3)
)

supported = v29_contrasts[
    v29_contrasts.conclusion == "supported improvement"
]
print("\nReading V29")
print("-----------")
if supported.empty:
    print("No V29 candidate established an overall improvement over current raw HGB.")
else:
    for row in supported.itertuples(index=False):
        print(
            f"{row.target.upper()}: {row.candidate} improved MAE by "
            f"{row.improvement_mmhg:.3f} mmHg "
            f"[{row.CI_low:.3f}, {row.CI_high:.3f}]."
        )
print(
    "Selective-prediction results must always be reported with retained coverage; "
    "they do not replace the 100% coverage primary endpoint."
)

# Publication-ready paired-effect and coverage figures.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)
for axis, target in zip(axes, ["sbp", "dbp"]):
    shown = v29_contrasts[v29_contrasts.target == target].reset_index(drop=True)
    y = np.arange(len(shown))
    x = shown.improvement_mmhg.to_numpy(float)
    xerr = np.vstack([
        x - shown.CI_low.to_numpy(float),
        shown.CI_high.to_numpy(float) - x,
    ])
    axis.errorbar(x, y, xerr=xerr, fmt="o", capsize=4, color="#0173B2")
    axis.axvline(0, color="#333333", linestyle="--", linewidth=1)
    axis.set_yticks(y)
    axis.set_yticklabels(shown.candidate)
    axis.set_title(target.upper())
    axis.set_xlabel("MAE improvement vs current model (mmHg)")
    axis.grid(axis="x", alpha=0.25)
fig.suptitle("V29 candidate improvements: paired participant-bootstrap 95% CI")
fig.tight_layout()
fig.savefig(V29_RESULTS / "v29_candidate_improvements.png", dpi=300)
plt.show()

fig, axis = plt.subplots(figsize=(6.5, 4.5))
for target, group in v29_coverage.groupby("target"):
    group = group.sort_values("actual_measurement_coverage")
    axis.plot(
        100 * group.actual_measurement_coverage,
        group.participant_MAE,
        marker="o", label=target.upper(),
    )
axis.set_xlabel("Measurements retained (%)")
axis.set_ylabel("Participant-level MAE (mmHg)")
axis.set_title("Uncertainty-based accuracy–coverage trade-off")
axis.legend()
axis.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(V29_RESULTS / "v29_accuracy_coverage.png", dpi=300)
plt.show()

v29_predictions.to_csv(V29_RESULTS / "v29_outer_predictions.csv", index=False)
v29_selection.to_csv(V29_RESULTS / "v29_inner_selections.csv", index=False)
v29_feature_audit.to_csv(V29_RESULTS / "v29_feature_audit.csv", index=False)
v29_participant_errors.to_csv(
    V29_RESULTS / "v29_participant_errors.csv", index=False
)
v29_summary.to_csv(V29_RESULTS / "v29_model_summary.csv", index=False)
v29_contrasts.to_csv(V29_RESULTS / "v29_paired_contrasts.csv", index=False)
v29_coverage.to_csv(V29_RESULTS / "v29_accuracy_coverage.csv", index=False)
v29_coverage_participant.to_csv(
    V29_RESULTS / "v29_coverage_participant_errors.csv", index=False
)
print("Saved V29 outputs to:", V29_RESULTS)


---

# V30: hard-case specialization, scored at 100% coverage

V29's uncertainty model was good at *identifying* hard rows — that's what the
coverage table showed. But that table only demonstrated it by **dropping**
those rows, which cannot generalize to the 100% coverage endpoint: excluding
the hardest 10-30% mechanically lowers MAE on what's left, the same way a
class average rises if you drop the bottom decile.

This cell asks a different question: can the same uncertainty signal make the
model itself better on the hard rows, instead of excluding them? Every arm
here is scored on **all** rows, paired against the frozen V29
`nested_ensemble`, using V29's exact outer/inner participant-CV structure and
feature pool.

Two independent mechanisms:

- **`uncertainty_aware_hgb`** — gives the delta model an honest out-of-fold
  hardness score (model disagreement, elapsed time, signal quality, cal_sd)
  as a feature, plus its product with the top-ranked PPG features, so the
  model can weight those features differently on rows predicted to be hard.
  The hardness score for *training* rows is generated by a small additional
  participant split so the model never sees a row's own error folded back in
  as a feature for that same row — that would leak through the back door
  even though it looks like plain feature engineering.
- **`lability_weighted_hgb`** — same features as raw HGB, but training rows
  from high-`cal_sd` (labile) participants are upweighted, so squared-error
  loss stops effectively under-fitting the tail it currently gets wrong most.
  Prediction-time behavior is unchanged; only what the model was asked to get
  right during fitting changes.

`equal_weight_combo` is an unweighted mean of `nested_ensemble`,
`uncertainty_aware_hgb`, and `lability_weighted_hgb` — cheap to compute since
no new fitting is required, and it tells you whether stacking helps even
before tuning blend weights.

**Runtime**: this repeats V29's full inner/outer CV plus two new HGB fits per
target per outer fold plus one extra small participant split per fold/target
for the leakage-safe hardness score. Budget noticeably more than V29's
runtime — a rough multiple of 1.3-1.5x.

**Reading this**: if `uncertainty_aware_hgb` or `lability_weighted_hgb` clears
zero against `nested_ensemble`, that's a real, deployable, 100%-coverage
improvement, and unlike everything in V28 it's aimed specifically at the
hard-participant tail rather than more anchor-side information. If nothing
clears zero, that is still informative: it means the uncertainty signal is
good for triage (V29's coverage table) but not for making the model itself
better on the rows it identifies as hard — a real distinction, and a clean
thing to report as a boundary of what this feature can do.


## V30: hard-case specialization arms

In [ ]:
# ============================================================================
# V30: HARD-CASE SPECIALIZATION, SCORED AT 100% COVERAGE
# ----------------------------------------------------------------------------
# Reuses V29's frame, feature pool, model families, gate, and uncertainty
# model wholesale. Only three new functions are introduced:
#   v30_lability_sample_weight  -- upweight labile participants at fit time
#   v30_fit_predict_delta       -- v29_fit_predict_delta with explicit weights
#   v30_oof_uncertainty         -- leakage-safe hardness score for TRAIN rows
# Every arm is scored on 100% of rows. No coverage gating anywhere in this cell.
# ============================================================================
V30_RESULTS = OUT_DIR / "avct_v30_hard_case_specialization"
V30_RESULTS.mkdir(parents=True, exist_ok=True)

V30_OUTER_SPLITS = V26_OUTER_SPLITS
V30_REPEATS = V26_REPEATS
V30_INNER_SPLITS = V29_INNER_SPLITS
V30_BOOTSTRAPS = 5000
V30_LABILITY_WEIGHT_CAP = 3.0     # max upweighting for the most labile decile
V30_INTERACTION_TOP_N = 8         # top-ranked PPG features to cross with uncertainty


def v30_lability_sample_weight(frame, target):
    """abl_weights, scaled up for high cal_sd_<target> participants.

    Rows without a usable cal_sd (dose < required readings, or zero variance
    in this fold) fall back to plain abl_weights -- unchanged behaviour, no
    NaNs propagated, no crash on a degenerate fold.
    """
    base = abl_weights(frame, "fst_band", False)
    column = "cal_sd_" + target
    if column not in frame.columns:
        return base
    lability = pd.to_numeric(frame[column], errors="coerce")
    finite = lability[np.isfinite(lability)]
    if len(finite) < 30 or finite.std() < 1e-9:
        return base
    # Rank-normalize to [0, 1] so the scale factor is distribution-free.
    rank = lability.rank(pct=True)
    scale = 1.0 + (V30_LABILITY_WEIGHT_CAP - 1.0) * rank.fillna(0.5).to_numpy(float)
    weighted = base * scale
    return np.asarray(weighted / weighted.mean(), float)


def v30_fit_predict_delta(train, test, target_column, baseline_column,
                           features, sample_weight, seed):
    """v29_fit_predict_delta, but takes an explicit sample_weight array
    (aligned to `train` before NaN filtering) instead of recomputing
    abl_weights internally, so the plain and lability-weighted arms share
    every other line of fitting/gating logic."""
    keep = train[[target_column, baseline_column]].notna().all(axis=1).to_numpy()
    fit = train.loc[keep].copy()
    weights = np.asarray(sample_weight, float)[keep]
    assert len(weights) == len(fit), "sample_weight must be aligned to `train`"
    model = v29_model("hgb", seed)
    final_step = model.steps[-1][0]
    model.fit(fit[features], fit[target_column],
              **{final_step + "__sample_weight": weights})
    absolute_target = target_column[len("delta_"):]
    fit_delta = model.predict(fit[features])
    gate = abl_fit_gate(fit, absolute_target, baseline_column, fit_delta, "fst_band", False)
    test_delta = model.predict(test[features])
    prediction, _ = abl_apply_gate(test, baseline_column, test_delta, gate, "fst_band")
    return np.asarray(prediction, float)


def v30_oof_uncertainty(train, target, ensemble_prediction_column, n_splits, seed):
    """Genuinely out-of-fold hardness score for TRAIN rows.

    v29_fit_uncertainty fits and scores on the same rows when it's only used
    to set a coverage quantile -- harmless there. Using its in-sample
    prediction as a MODEL FEATURE would let the uncertainty model leak each
    row's own error back to the delta model on the exact rows it's trained
    on. This runs one more small participant split purely to avoid that.
    """
    frame = train.reset_index(drop=True)
    scores = np.full(len(frame), np.nan, float)
    for fold, sub_train, sub_valid in v29_participant_splits(frame, n_splits, seed):
        _, valid_score, _ = v29_fit_uncertainty(
            sub_train, sub_valid, target,
            sub_train[ensemble_prediction_column],
            sub_valid[ensemble_prediction_column],
            seed + fold,
        )
        scores[sub_valid.index.to_numpy()] = valid_score
    return scores


v30_prediction_rows = []
for repeat in range(V30_REPEATS):
    outer_seed = SEED + 30000 + 1000 * repeat
    for outer_fold, outer_train, outer_test in v29_participant_splits(
        v29_frame, V30_OUTER_SPLITS, outer_seed
    ):
        inner_parts = []
        for inner_fold, inner_train, inner_valid in v29_participant_splits(
            outer_train, V30_INNER_SPLITS, outer_seed + 100 + outer_fold
        ):
            inner_prediction, _ = v29_predict_components(
                inner_train, inner_valid,
                outer_seed + 10000 * outer_fold + 100 * inner_fold,
            )
            inner_prediction["inner_fold"] = inner_fold
            inner_parts.append(inner_prediction)
        inner_oof = pd.concat(inner_parts, ignore_index=True)
        assert inner_oof.subject_id.nunique() == outer_train.subject_id.nunique()

        outer_prediction, _ = v29_predict_components(
            outer_train, outer_test, outer_seed + 50000 + outer_fold
        )

        for target_index, target in enumerate(["sbp", "dbp"]):
            # Reproduce V29's nested_ensemble exactly, as the paired baseline
            # and as the reference prediction the uncertainty model conditions on.
            weights, _ = v29_choose_weights(inner_oof, target)
            inner_ensemble = v29_blend(inner_oof, target, weights)
            outer_ensemble = v29_blend(outer_prediction, target, weights)
            inner_oof["_v30_ensemble"] = inner_ensemble
            outer_prediction["_v30_ensemble"] = outer_ensemble

            # Leakage-safe hardness score: OOF for train rows, clean for test rows.
            oof_hardness = v30_oof_uncertainty(
                inner_oof, target, "_v30_ensemble", V30_INNER_SPLITS,
                outer_seed + 20000 + 100 * outer_fold + target_index,
            )
            _, test_hardness, _ = v29_fit_uncertainty(
                inner_oof, outer_prediction, target,
                inner_ensemble, outer_ensemble,
                outer_seed + 70000 + 100 * outer_fold + target_index,
            )
            inner_oof["_v30_uncertainty"] = oof_hardness
            outer_prediction["_v30_uncertainty"] = test_hardness

            # Uncertainty-aware arm: base features + hardness + interactions.
            base_features = v29_rank_features(
                inner_oof, "delta_" + target, "baseline_" + target,
                "uci_prior_delta_" + target,
            )
            ppg_terms = [f for f in base_features if f in DELTA_RAW + RELATIVE_RAW][
                :V30_INTERACTION_TOP_N
            ]
            for term in ppg_terms:
                inner_oof[term + "_x_unc"] = inner_oof[term] * inner_oof["_v30_uncertainty"]
                outer_prediction[term + "_x_unc"] = (
                    outer_prediction[term] * outer_prediction["_v30_uncertainty"]
                )
            expanded_features = base_features + ["_v30_uncertainty"] + [
                term + "_x_unc" for term in ppg_terms
            ]
            std_weights = abl_weights(inner_oof, "fst_band", False)
            uncertainty_aware_pred = v30_fit_predict_delta(
                inner_oof, outer_prediction, "delta_" + target, "baseline_" + target,
                expanded_features, std_weights,
                outer_seed + 80000 + 100 * outer_fold + target_index,
            )

            # Lability-weighted arm: same features as raw HGB, reweighted fit.
            lability_weights = v30_lability_sample_weight(inner_oof, target)
            lability_pred = v30_fit_predict_delta(
                inner_oof, outer_prediction, "delta_" + target, "baseline_" + target,
                base_features, lability_weights,
                outer_seed + 90000 + 100 * outer_fold + target_index,
            )

            # v29_blend already returns absolute BP (baseline + gated delta),
            # matching how V29 itself stores nested_ensemble_<target> -- do
            # NOT add baseline again here, that would double-count it.
            nested_pred = np.asarray(outer_ensemble, float)
            outer_prediction[f"nested_ensemble_{target}"] = nested_pred
            outer_prediction[f"uncertainty_aware_{target}"] = uncertainty_aware_pred
            outer_prediction[f"lability_weighted_{target}"] = lability_pred
            outer_prediction[f"equal_weight_combo_{target}"] = (
                nested_pred + uncertainty_aware_pred + lability_pred
            ) / 3.0

        outer_prediction["repeat"] = repeat + 1
        outer_prediction["outer_fold"] = outer_fold
        v30_prediction_rows.append(outer_prediction)
        print(f"V30 repeat {repeat + 1}/{V30_REPEATS}, outer fold {outer_fold}/{V30_OUTER_SPLITS} complete")

v30_predictions = pd.concat(v30_prediction_rows, ignore_index=True)

V30_ARMS = ["nested_ensemble", "uncertainty_aware", "lability_weighted", "equal_weight_combo"]

v30_error_rows = []
for target in ["sbp", "dbp"]:
    for arm in V30_ARMS:
        column = f"{arm}_{target}"
        working = v30_predictions.assign(_error=np.abs(v30_predictions[column] - v30_predictions[target]))
        per_person = working.groupby("subject_id", as_index=False).agg(mae=("_error", "mean"))
        for row in per_person.itertuples(index=False):
            v30_error_rows.append({"target": target, "arm": arm, "subject_id": row.subject_id, "mae": row.mae})
v30_participant_errors = pd.DataFrame(v30_error_rows)

v30_summary_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    for arm_index, arm in enumerate(V30_ARMS):
        group = v30_participant_errors[(v30_participant_errors.target == target) & (v30_participant_errors.arm == arm)]
        low, high = v29_bootstrap(group.mae, SEED + 30500 + 100 * target_index + arm_index)
        v30_summary_rows.append({
            "target": target, "arm": arm, "participants": group.subject_id.nunique(),
            "participant_MAE": group.mae.mean(), "CI_low": low, "CI_high": high,
        })
v30_summary = pd.DataFrame(v30_summary_rows)

v30_contrast_rows = []
for target_index, target in enumerate(["sbp", "dbp"]):
    pivot = v30_participant_errors[v30_participant_errors.target == target].pivot(
        index="subject_id", columns="arm", values="mae"
    )
    for arm_index, arm in enumerate([a for a in V30_ARMS if a != "nested_ensemble"]):
        aligned = pivot[["nested_ensemble", arm]].dropna()
        improvement = aligned.nested_ensemble - aligned[arm]
        low, high = v29_bootstrap(improvement, SEED + 30600 + 100 * target_index + arm_index)
        if low > 0:
            conclusion = "supported improvement"
        elif high < 0:
            conclusion = "supported harm"
        else:
            conclusion = "no clear difference"
        v30_contrast_rows.append({
            "target": target, "candidate": arm, "participants": len(aligned),
            "nested_ensemble_MAE": aligned.nested_ensemble.mean(), "candidate_MAE": aligned[arm].mean(),
            "improvement_mmhg": improvement.mean(), "CI_low": low, "CI_high": high,
            "fraction_helped": (improvement > 0).mean(), "conclusion": conclusion,
        })
v30_contrasts = pd.DataFrame(v30_contrast_rows)

print("\n=== V30 repeated participant-CV MAE (100% coverage, all rows) ===")
display(v30_summary.round(3))
print("\n=== Paired comparison with V29's nested_ensemble ===")
print("Positive improvement favors the candidate. All arms scored on every row.")
display(v30_contrasts.round(3))

supported = v30_contrasts[v30_contrasts.conclusion == "supported improvement"]
print("\nReading V30")
print("-----------")
if supported.empty:
    print("No V30 candidate beat nested_ensemble at 100% coverage.")
    print("The uncertainty signal may still be useful for triage (V29's coverage table)")
    print("without being usable to make the model itself better on the rows it flags.")
else:
    for row in supported.itertuples(index=False):
        print(f"{row.target.upper()}: {row.candidate} improved MAE by {row.improvement_mmhg:.3f} mmHg "
              f"[{row.CI_low:.3f}, {row.CI_high:.3f}] at 100% coverage.")

v30_predictions.to_csv(V30_RESULTS / "v30_outer_predictions.csv", index=False)
v30_participant_errors.to_csv(V30_RESULTS / "v30_participant_errors.csv", index=False)
v30_summary.to_csv(V30_RESULTS / "v30_model_summary.csv", index=False)
v30_contrasts.to_csv(V30_RESULTS / "v30_paired_contrasts.csv", index=False)
print("Saved V30 outputs to:", V30_RESULTS)
